In [1]:
# ==========================================
# STAGE 1 — CELL 1
# ENVIRONMENT SETUP — VSCODE / DGX SPARK
# ==========================================

import os
import sys
import gc
import time
import random
import warnings

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")


# ------------------------------------------
# Reproducibility
# ------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ------------------------------------------
# Device
# ------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# ------------------------------------------
# Dataset directory
# ------------------------------------------

DATA_DIR = "/home/llyods-aids5/AML"


# ------------------------------------------
# Dataset files
# ------------------------------------------

CSV_FILE = os.path.join(
    DATA_DIR,
    "HI-Small_FANOUT_10M_transactions.csv"
)

ACCOUNTS_FILE = os.path.join(
    DATA_DIR,
    "HI-Small_FANOUT_accounts(1).csv"
)

PATTERNS_FILE = os.path.join(
    DATA_DIR,
    "HI-Small_FANOUT_Patterns.txt"
)


# ------------------------------------------
# Basic environment information
# ------------------------------------------

print("==========================================")
print("STAGE 1 — VSCODE / DGX SPARK ENVIRONMENT")
print("==========================================")

print("Python :", sys.version.split()[0])
print("NumPy  :", np.__version__)
print("Pandas :", pd.__version__)
print("PyTorch:", torch.__version__)

print("\nDevice:", DEVICE)


# ------------------------------------------
# GPU information
# ------------------------------------------

if torch.cuda.is_available():

    print(
        "GPU   :",
        torch.cuda.get_device_name(0)
    )

    print(
        "CUDA  :",
        torch.version.cuda
    )

    gpu_memory = (
        torch.cuda.get_device_properties(0)
        .total_memory
        / 1024**3
    )

    print(
        f"GPU Memory: {gpu_memory:.2f} GB"
    )

else:

    print("⚠️ CUDA is NOT available")


# ------------------------------------------
# Dataset paths
# ------------------------------------------

print("\nDataset directory:")
print(DATA_DIR)

print("\nTransactions:")
print(CSV_FILE)

print("\nAccounts:")
print(ACCOUNTS_FILE)

print("\nPatterns:")
print(PATTERNS_FILE)


# ------------------------------------------
# Check files
# ------------------------------------------

for name, path in [
    ("Transactions", CSV_FILE),
    ("Accounts", ACCOUNTS_FILE),
    ("Patterns", PATTERNS_FILE)
]:

    if os.path.exists(path):

        size = (
            os.path.getsize(path)
            / 1024**3
        )

        print(
            f"\n✅ {name} file found"
        )

        print(
            f"   Path : {path}"
        )

        print(
            f"   Size : {size:.2f} GB"
        )

    else:

        raise FileNotFoundError(
            f"\n❌ {name} file not found:\n"
            f"{path}"
        )


print("\n==========================================")
print("✅ CELL 1 COMPLETE")
print("==========================================")

STAGE 1 — VSCODE / DGX SPARK ENVIRONMENT
Python : 3.12.3
NumPy  : 1.26.4
Pandas : 2.1.4
PyTorch: 2.10.0+cu128

Device: cuda
GPU   : NVIDIA GB10
CUDA  : 12.8
GPU Memory: 121.69 GB

Dataset directory:
/home/llyods-aids5/AML

Transactions:
/home/llyods-aids5/AML/HI-Small_FANOUT_10M_transactions.csv

Accounts:
/home/llyods-aids5/AML/HI-Small_FANOUT_accounts(1).csv

Patterns:
/home/llyods-aids5/AML/HI-Small_FANOUT_Patterns.txt

✅ Transactions file found
   Path : /home/llyods-aids5/AML/HI-Small_FANOUT_10M_transactions.csv
   Size : 1.09 GB

✅ Accounts file found
   Path : /home/llyods-aids5/AML/HI-Small_FANOUT_accounts(1).csv
   Size : 0.08 GB

✅ Patterns file found
   Path : /home/llyods-aids5/AML/HI-Small_FANOUT_Patterns.txt
   Size : 0.52 GB

✅ CELL 1 COMPLETE


In [3]:
# ==========================================
# STAGE 1 — CELL 2
# DATASET PATHS — VSCODE / DGX SPARK
# ==========================================

import os

DATA_DIR = "/home/llyods-aids5/AML"

CSV_FILE = os.path.join(
    DATA_DIR,
    "HI-Small_FANOUT_10M_transactions.csv"
)

ACCOUNTS_FILE = os.path.join(
    DATA_DIR,
    "HI-Small_FANOUT_accounts(1).csv"
)

PATTERNS_FILE = os.path.join(
    DATA_DIR,
    "HI-Small_FANOUT_Patterns.txt"
)

print("==========================================")
print("STAGE 1 — DATASET PATHS")
print("==========================================")

print("\nTransactions:")
print(CSV_FILE)

print("\nAccounts:")
print(ACCOUNTS_FILE)

print("\nPatterns:")
print(PATTERNS_FILE)

print("\n------------------------------------------")

for name, path in [
    ("Transactions", CSV_FILE),
    ("Accounts", ACCOUNTS_FILE),
    ("Patterns", PATTERNS_FILE)
]:

    if os.path.exists(path):

        size = os.path.getsize(path)

        print(
            f"{name:15s}: ✅ FOUND"
        )

        if size >= 1024**3:
            print(
                f"{'':15s}  Size: "
                f"{size / 1024**3:.2f} GB"
            )

        elif size >= 1024**2:
            print(
                f"{'':15s}  Size: "
                f"{size / 1024**2:.2f} MB"
            )

        else:
            print(
                f"{'':15s}  Size: "
                f"{size / 1024:.2f} KB"
            )

    else:

        print(
            f"{name:15s}: ❌ NOT FOUND"
        )

        raise FileNotFoundError(
            f"{name} file not found:\n{path}"
        )

print("\n✅ Cell 2 complete")

STAGE 1 — DATASET PATHS

Transactions:
/home/llyods-aids5/AML/HI-Small_FANOUT_10M_transactions.csv

Accounts:
/home/llyods-aids5/AML/HI-Small_FANOUT_accounts(1).csv

Patterns:
/home/llyods-aids5/AML/HI-Small_FANOUT_Patterns.txt

------------------------------------------
Transactions   : ✅ FOUND
                 Size: 1.09 GB
Accounts       : ✅ FOUND
                 Size: 77.44 MB
Patterns       : ✅ FOUND
                 Size: 530.71 MB

✅ Cell 2 complete


In [4]:
# ==========================================
# STAGE 1 — CELL 3
# LOAD 10M FANOUT TRANSACTIONS
# ==========================================

import pandas as pd
import numpy as np
import gc
import time

print("==========================================")
print("STAGE 1 — LOADING TRANSACTIONS")
print("==========================================")

t0 = time.time()

TRANSACTION_COLS = [
    "transaction_id",
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
    "Is Laundering"
]

df = pd.read_csv(
    CSV_FILE,
    usecols=TRANSACTION_COLS,
    low_memory=False
)

# Normalize label
df["Is Laundering"] = (
    pd.to_numeric(
        df["Is Laundering"],
        errors="coerce"
    )
    .fillna(0)
    .astype(np.int8)
)

# Normalize timestamp
df["Timestamp"] = pd.to_datetime(
    df["Timestamp"],
    errors="coerce"
)

# Stable row index for later train/val/test mapping
df["row_id"] = np.arange(
    len(df),
    dtype=np.int64
)

print("\n✅ Transactions loaded")

print(
    f"Rows loaded : {len(df):,}"
)

print(
    f"Columns     : {len(df.columns)}"
)

print(
    f"Load time   : {time.time() - t0:.2f} sec"
)

print("\nMemory usage:")
print(
    f"{df.memory_usage(deep=True).sum() / 1024**3:.2f} GB"
)

print("\nLabel counts:")
print(
    df["Is Laundering"].value_counts()
    .sort_index()
)

STAGE 1 — LOADING TRANSACTIONS

✅ Transactions loaded
Rows loaded : 10,000,000
Columns     : 13
Load time   : 16.16 sec

Memory usage:
3.72 GB

Label counts:
Is Laundering
0    5000000
1    5000000
Name: count, dtype: int64


In [5]:
# ==========================================
# STAGE 1 — CELL 4
# REUSABLE AML FEATURE ENGINE
# ==========================================

import numpy as np
import pandas as pd
import time

from sklearn.preprocessing import StandardScaler


class AMLFeatureEngine:
    """
    Reusable feature engine for AML graph transactions.

    Input:
        Raw transaction dataframe containing the original
        transaction columns.

    Output:
        - df_features : transaction-level engineered features
        - X_node      : node/account feature matrix
        - E_features  : edge/transaction feature matrix
        - node_to_id  : account -> graph node ID mapping
    """

    def __init__(self):
        self.node_scaler = StandardScaler()
        self.edge_scaler = StandardScaler()

        self.payment_format_map = {}
        self.payment_currency_map = {}
        self.receiving_currency_map = {}

        self.node_to_id = {}

        # USD conversion table
        self.FX_TO_USD = {
            "US Dollar": 1.0,
            "Euro": 1.00,
            "UK Pound": 1.10,
            "Yen": 0.0069,
            "Yuan": 0.141,
            "Rupee": 0.0123,
            "Ruble": 0.0166,
            "Swiss Franc": 1.02,
            "Canadian Dollar": 0.735,
            "Australian Dollar": 0.65,
            "Mexican Peso": 0.050,
            "Brazil Real": 0.19,
            "Saudi Riyal": 0.267,
            "Shekel": 0.287,
            "Bitcoin": 19000.0,
        }

    # --------------------------------------
    # Encode categorical values
    # --------------------------------------
    def _encode(self, series, mapping):

        values = series.astype(str).fillna("UNK")

        encoded = np.empty(
            len(values),
            dtype=np.float32
        )

        for i, value in enumerate(values):

            if value not in mapping:
                mapping[value] = len(mapping)

            encoded[i] = mapping[value]

        return encoded

    # --------------------------------------
    # MAIN FEATURE FUNCTION
    # --------------------------------------
    def transform(self, df):

        t0 = time.time()

        df = df.copy()

        # ==================================
        # 1. BASIC STANDARDIZATION
        # ==================================

        df["Timestamp"] = pd.to_datetime(
            df["Timestamp"],
            errors="coerce"
        )

        df["Amount Paid"] = pd.to_numeric(
            df["Amount Paid"],
            errors="coerce"
        ).fillna(0.0)

        df["Amount Received"] = pd.to_numeric(
            df["Amount Received"],
            errors="coerce"
        ).fillna(0.0)

        # ==================================
        # 2. CURRENCY NORMALIZATION
        # ==================================

        paid_fx = (
            df["Payment Currency"]
            .astype(str)
            .str.strip()
            .map(self.FX_TO_USD)
            .fillna(1.0)
            .to_numpy(dtype=np.float32)
        )

        recv_fx = (
            df["Receiving Currency"]
            .astype(str)
            .str.strip()
            .map(self.FX_TO_USD)
            .fillna(1.0)
            .to_numpy(dtype=np.float32)
        )

        df["amt_paid_usd"] = (
            df["Amount Paid"].to_numpy(
                dtype=np.float32
            ) * paid_fx
        )

        df["amt_recv_usd"] = (
            df["Amount Received"].to_numpy(
                dtype=np.float32
            ) * recv_fx
        )

        # ==================================
        # 3. CHRONOLOGICAL ORDER
        # ==================================

        df = (
            df.sort_values(
                ["Timestamp", "transaction_id"]
                if "transaction_id" in df.columns
                else ["Timestamp"]
            )
            .reset_index(drop=True)
        )

        # ==================================
        # 4. ACCOUNT / GRAPH IDENTIFIERS
        # ==================================

        df["src_key"] = (
            df["From Bank"].astype(str).str.strip()
            + "_"
            + df["Account"].astype(str).str.strip()
        )

        df["dst_key"] = (
            df["To Bank"].astype(str).str.strip()
            + "_"
            + df["Account.1"].astype(str).str.strip()
        )

        # ==================================
        # 5. TRANSACTION FEATURES
        # ==================================

        edge_features = pd.DataFrame(index=df.index)

        # Amounts
        edge_features["amount_paid"] = (
            df["amt_paid_usd"]
        )

        edge_features["amount_received"] = (
            df["amt_recv_usd"]
        )

        edge_features["log_amount_paid"] = (
            np.log1p(
                df["amt_paid_usd"].clip(lower=0)
            )
        )

        edge_features["log_amount_received"] = (
            np.log1p(
                df["amt_recv_usd"].clip(lower=0)
            )
        )

        edge_features["amount_difference"] = (
            df["amt_recv_usd"]
            - df["amt_paid_usd"]
        ).abs()

        edge_features["amount_ratio"] = (
            df["amt_recv_usd"]
            /
            (df["amt_paid_usd"] + 1e-5)
        )

        # Time
        hour = (
            df["Timestamp"]
            .dt.hour
            .fillna(0)
            .astype(np.float32)
        )

        edge_features["hour_sin"] = (
            np.sin(2 * np.pi * hour / 24.0)
        )

        edge_features["hour_cos"] = (
            np.cos(2 * np.pi * hour / 24.0)
        )

        edge_features["day_of_week"] = (
            df["Timestamp"]
            .dt.dayofweek
            .fillna(0)
        )

        edge_features["weekend_flag"] = (
            edge_features["day_of_week"] >= 5
        ).astype(np.float32)

        # Graph relationship
        edge_features["self_loop"] = (
            df["src_key"] == df["dst_key"]
        ).astype(np.float32)

        edge_features["same_bank"] = (
            df["From Bank"].astype(str)
            ==
            df["To Bank"].astype(str)
        ).astype(np.float32)

        # AML threshold behavior
        threshold = 10000.0

        edge_features["near_threshold"] = (
            (df["amt_paid_usd"] >= 9000.0)
            &
            (df["amt_paid_usd"] < 10000.0)
        ).astype(np.float32)

        edge_features["threshold_proximity"] = (
            df["amt_paid_usd"].clip(0, threshold)
            / threshold
        )

        # ==================================
        # 6. ACCOUNT-PAIR FEATURES
        # ==================================

        df["pair_key"] = (
            df["src_key"] +
            "->" +
            df["dst_key"]
        )

        ts_seconds = (
            df["Timestamp"]
            .astype("int64")
            // 10**9
        )

        previous_pair_time = (
            ts_seconds
            .groupby(df["pair_key"])
            .shift(1)
        )

        edge_features["pair_recency_hours"] = (
            (
                ts_seconds -
                previous_pair_time
            )
            / 3600.0
        ).fillna(9999.0)

        edge_features["first_pair_transaction"] = (
            previous_pair_time.isna()
        ).astype(np.float32)

        # ==================================
        # 7. SENDER BEHAVIOR
        # ==================================

        sender_mean = (
            df.groupby("src_key")["amt_paid_usd"]
            .transform("mean")
        )

        sender_std = (
            df.groupby("src_key")["amt_paid_usd"]
            .transform("std")
            .fillna(1.0)
            .replace(0.0, 1.0)
        )

        edge_features["sender_amount_zscore"] = (
            (
                df["amt_paid_usd"]
                - sender_mean
            )
            / sender_std
        )

        # ==================================
        # 8. CATEGORICAL FEATURES
        # ==================================

        edge_features["payment_format"] = (
            self._encode(
                df["Payment Format"],
                self.payment_format_map
            )
        )

        edge_features["payment_currency"] = (
            self._encode(
                df["Payment Currency"],
                self.payment_currency_map
            )
        )

        edge_features["receiving_currency"] = (
            self._encode(
                df["Receiving Currency"],
                self.receiving_currency_map
            )
        )

        # ==================================
        # 9. GRAPH NODE MAPPING
        # ==================================

        all_nodes = pd.unique(
            pd.concat(
                [
                    df["src_key"],
                    df["dst_key"]
                ],
                ignore_index=True
            )
        )

        # Stable mapping
        self.node_to_id = {
            node: i
            for i, node in enumerate(all_nodes)
        }

        df["src_id"] = (
            df["src_key"]
            .map(self.node_to_id)
            .astype(np.int64)
        )

        df["dst_id"] = (
            df["dst_key"]
            .map(self.node_to_id)
            .astype(np.int64)
        )

        n_nodes = len(all_nodes)

        # ==================================
        # 10. NODE / ACCOUNT FEATURES
        # ==================================

        out_count = (
            df.groupby("src_id")
            .size()
            .rename("out_count")
        )

        in_count = (
            df.groupby("dst_id")
            .size()
            .rename("in_count")
        )

        out_total = (
            df.groupby("src_id")["amt_paid_usd"]
            .sum()
            .rename("out_total")
        )

        in_total = (
            df.groupby("dst_id")["amt_recv_usd"]
            .sum()
            .rename("in_total")
        )

        out_mean = (
            df.groupby("src_id")["amt_paid_usd"]
            .mean()
            .rename("out_mean")
        )

        in_mean = (
            df.groupby("dst_id")["amt_recv_usd"]
            .mean()
            .rename("in_mean")
        )

        out_max = (
            df.groupby("src_id")["amt_paid_usd"]
            .max()
            .rename("out_max")
        )

        unique_receivers = (
            df.groupby("src_id")["dst_id"]
            .nunique()
            .rename("unique_receivers")
        )

        unique_senders = (
            df.groupby("dst_id")["src_id"]
            .nunique()
            .rename("unique_senders")
        )

        node_df = pd.DataFrame(
            index=np.arange(n_nodes)
        )

        node_df = (
            node_df
            .join(out_count)
            .join(in_count)
            .join(out_total)
            .join(in_total)
            .join(out_mean)
            .join(in_mean)
            .join(out_max)
            .join(unique_receivers)
            .join(unique_senders)
            .fillna(0.0)
        )

        # Flow features
        node_df["net_flow"] = (
            node_df["in_total"]
            -
            node_df["out_total"]
        )

        node_df["total_degree"] = (
            node_df["in_count"]
            +
            node_df["out_count"]
        )

        # Fan-out behavior
        node_df["fanout_ratio"] = (
            node_df["unique_receivers"]
            /
            (node_df["out_count"] + 1.0)
        )

        # In/out behavior
        node_df["pass_through_ratio"] = (
            np.minimum(
                node_df["in_count"],
                node_df["out_count"]
            )
            /
            (
                np.maximum(
                    node_df["in_count"],
                    node_df["out_count"]
                )
                + 1.0
            )
        )

        # ==================================
        # 11. FINAL FEATURE MATRICES
        # ==================================

        edge_feature_names = list(
            edge_features.columns
        )

        node_feature_names = list(
            node_df.columns
        )

        # Replace invalid numerical values
        edge_features = (
            edge_features
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0.0)
        )

        node_df = (
            node_df
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0.0)
        )

        # Scale features
        E_features = (
            self.edge_scaler
            .fit_transform(edge_features)
            .astype(np.float32)
        )

        X_node = (
            self.node_scaler
            .fit_transform(node_df)
            .astype(np.float32)
        )

        print(
            f"✅ Features generated in "
            f"{time.time() - t0:.2f} sec"
        )

        print(
            f"Transactions : {len(df):,}"
        )

        print(
            f"Nodes        : {n_nodes:,}"
        )

        print(
            f"Edge features: {E_features.shape}"
        )

        print(
            f"Node features: {X_node.shape}"
        )

        return (
            df,
            X_node,
            E_features,
            edge_feature_names,
            node_feature_names
        )


# ==========================================
# CREATE FEATURE ENGINE INSTANCE
# ==========================================

feature_engine = AMLFeatureEngine()

print("==========================================")
print("✅ AML FEATURE ENGINE READY")
print("==========================================")

✅ AML FEATURE ENGINE READY


In [6]:
# ==========================================
# STAGE 1 — CELL 5
# FULL 10M FEATURE GENERATION
# ==========================================

print("==========================================")
print("STAGE 1 — FULL FEATURE GENERATION")
print("==========================================")

t0 = time.time()

(
    df_features,
    X_node,
    E_features,
    EDGE_FEATURE_NAMES,
    NODE_FEATURE_NAMES
) = feature_engine.transform(df)

print("\n==========================================")
print("✅ FULL FEATURE GENERATION COMPLETE")
print("==========================================")

print(
    f"Transactions : {len(df_features):,}"
)

print(
    f"Nodes        : {X_node.shape[0]:,}"
)

print(
    f"Edge features: {E_features.shape}"
)

print(
    f"Node features: {X_node.shape}"
)

print(
    f"Total time   : {time.time() - t0:.2f} sec"
)

print("\nEdge feature names:")
print(EDGE_FEATURE_NAMES)

print("\nNode feature names:")
print(NODE_FEATURE_NAMES)

STAGE 1 — FULL FEATURE GENERATION
✅ Features generated in 75.84 sec
Transactions : 10,000,000
Nodes        : 1,213,224
Edge features: (10000000, 20)
Node features: (1213224, 13)

✅ FULL FEATURE GENERATION COMPLETE
Transactions : 10,000,000
Nodes        : 1,213,224
Edge features: (10000000, 20)
Node features: (1213224, 13)
Total time   : 75.85 sec

Edge feature names:
['amount_paid', 'amount_received', 'log_amount_paid', 'log_amount_received', 'amount_difference', 'amount_ratio', 'hour_sin', 'hour_cos', 'day_of_week', 'weekend_flag', 'self_loop', 'same_bank', 'near_threshold', 'threshold_proximity', 'pair_recency_hours', 'first_pair_transaction', 'sender_amount_zscore', 'payment_format', 'payment_currency', 'receiving_currency']

Node feature names:
['out_count', 'in_count', 'out_total', 'in_total', 'out_mean', 'in_mean', 'out_max', 'unique_receivers', 'unique_senders', 'net_flow', 'total_degree', 'fanout_ratio', 'pass_through_ratio']


In [7]:
# ==========================================
# STAGE 1 — CELL 6
# INSTALL PY-G + BUILD 10M GRAPH
# ==========================================

import sys
import subprocess

print("Installing PyTorch Geometric...")

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "torch-geometric"
])

print("✅ PyTorch Geometric installed")

# ------------------------------------------
# Import PyG
# ------------------------------------------

from torch_geometric.data import Data

print(
    "PyG version:",
    __import__("torch_geometric").__version__
)

# ------------------------------------------
# Build original transaction graph
# ------------------------------------------

print("\nBuilding 10M transaction graph...")

t0 = time.time()

src_np = df_features["src_id"].to_numpy(
    dtype=np.int64,
    copy=False
)

dst_np = df_features["dst_id"].to_numpy(
    dtype=np.int64,
    copy=False
)

edge_index = torch.from_numpy(
    np.vstack([
        src_np,
        dst_np
    ])
)

x = torch.from_numpy(
    X_node
)

edge_attr = torch.from_numpy(
    E_features
)

y = torch.from_numpy(
    df_features["Is Laundering"].to_numpy(
        dtype=np.float32,
        copy=False
    )
)

data = Data(
    x=x,
    edge_index=edge_index,
    edge_attr=edge_attr,
    y=y
)

# ------------------------------------------
# Temporal split
# ------------------------------------------

num_edges = data.num_edges

train_end = int(
    num_edges * 0.70
)

val_end = int(
    num_edges * 0.85
)

train_idx = torch.arange(
    0,
    train_end,
    dtype=torch.long
)

val_idx = torch.arange(
    train_end,
    val_end,
    dtype=torch.long
)

test_idx = torch.arange(
    val_end,
    num_edges,
    dtype=torch.long
)

# ------------------------------------------
# Bidirectional message graph
# ------------------------------------------
# Original transaction direction is preserved
# in `data`.
#
# Message passing additionally gets the reverse
# direction so connected accounts can exchange
# information.
# ------------------------------------------

reverse_edge_index = torch.stack([
    edge_index[1],
    edge_index[0]
], dim=0)

message_edge_index = torch.cat(
    [
        edge_index,
        reverse_edge_index
    ],
    dim=1
)

message_edge_attr = torch.cat(
    [
        edge_attr,
        edge_attr
    ],
    dim=0
)

message_graph = Data(
    x=x,
    edge_index=message_edge_index,
    edge_attr=message_edge_attr
)

# ------------------------------------------
# Results
# ------------------------------------------

print("\n==========================================")
print("✅ 10M GRAPH READY")
print("==========================================")

print(
    "Original nodes        :",
    f"{data.num_nodes:,}"
)

print(
    "Original transactions :",
    f"{data.num_edges:,}"
)

print(
    "Message-passing edges :",
    f"{message_graph.num_edges:,}"
)

print(
    "Node features         :",
    tuple(data.x.shape)
)

print(
    "Edge features         :",
    tuple(data.edge_attr.shape)
)

print("\nTemporal split:")
print(
    "Train:",
    f"{len(train_idx):,}"
)

print(
    "Validation:",
    f"{len(val_idx):,}"
)

print(
    "Test:",
    f"{len(test_idx):,}"
)

print(
    f"\nTime: {time.time() - t0:.2f} sec"
)

print("\n✅ CELL 6 COMPLETE")

Installing PyTorch Geometric...
✅ PyTorch Geometric installed
PyG version: 2.8.0.post1

Building 10M transaction graph...

✅ 10M GRAPH READY
Original nodes        : 1,213,224
Original transactions : 10,000,000
Message-passing edges : 20,000,000
Node features         : (1213224, 13)
Edge features         : (10000000, 20)

Temporal split:
Train: 7,000,000
Validation: 1,500,000
Test: 1,500,000

Time: 0.10 sec

✅ CELL 6 COMPLETE


In [8]:
# ==========================================
# STAGE 1 — CELL 7
# FINAL GATv2 AML MODEL
# ==========================================

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import GATv2Conv


class GATAMLModel(nn.Module):
    """
    Edge-level AML classifier.

    Input:
        Node features
        Graph connectivity
        Edge/transaction features

    Output:
        One laundering logit per target transaction.
    """

    def __init__(
        self,
        node_in_dim,
        edge_in_dim,
        hidden_dim=64,
        heads=4,
        dropout=0.20
    ):
        super().__init__()

        self.node_in_dim = node_in_dim
        self.edge_in_dim = edge_in_dim
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.dropout_rate = dropout

        # --------------------------------------
        # Node projection
        # --------------------------------------

        self.node_proj = nn.Linear(
            node_in_dim,
            hidden_dim
        )

        # --------------------------------------
        # GAT Layer 1
        # --------------------------------------

        self.gat1 = GATv2Conv(
            in_channels=hidden_dim,
            out_channels=hidden_dim // heads,
            heads=heads,
            concat=True,
            edge_dim=edge_in_dim,
            dropout=dropout
        )

        # --------------------------------------
        # GAT Layer 2
        # --------------------------------------

        self.gat2 = GATv2Conv(
            in_channels=hidden_dim,
            out_channels=hidden_dim // heads,
            heads=heads,
            concat=True,
            edge_dim=edge_in_dim,
            dropout=dropout
        )

        # --------------------------------------
        # Normalization
        # --------------------------------------

        self.norm1 = nn.LayerNorm(
            hidden_dim
        )

        self.norm2 = nn.LayerNorm(
            hidden_dim
        )

        self.dropout = nn.Dropout(
            dropout
        )

        # --------------------------------------
        # Edge classifier
        #
        # source embedding
        # +
        # destination embedding
        # +
        # transaction features
        # --------------------------------------

        classifier_input = (
            hidden_dim
            + hidden_dim
            + edge_in_dim
        )

        self.classifier = nn.Sequential(

            nn.Linear(
                classifier_input,
                hidden_dim
            ),

            nn.LayerNorm(
                hidden_dim
            ),

            nn.LeakyReLU(
                0.2
            ),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                hidden_dim,
                32
            ),

            nn.LeakyReLU(
                0.2
            ),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                32,
                1
            )
        )

    # ======================================
    # GRAPH ENCODER
    # ======================================

    def encode(
        self,
        x,
        edge_index,
        edge_attr
    ):

        # Initial node projection
        h = F.leaky_relu(
            self.node_proj(x),
            negative_slope=0.2
        )

        # GAT layer 1
        h1 = self.gat1(
            h,
            edge_index,
            edge_attr=edge_attr
        )

        h = self.norm1(
            h + h1
        )

        h = self.dropout(h)

        # GAT layer 2
        h2 = self.gat2(
            h,
            edge_index,
            edge_attr=edge_attr
        )

        h = self.norm2(
            h + h2
        )

        return h

    # ======================================
    # EDGE DECODER
    # ======================================

    def decode(
        self,
        node_embeddings,
        target_edge_index,
        target_edge_attr
    ):

        src = target_edge_index[0]
        dst = target_edge_index[1]

        source_embedding = (
            node_embeddings[src]
        )

        destination_embedding = (
            node_embeddings[dst]
        )

        # ----------------------------------
        # Combine:
        #
        # Source account representation
        # Destination account representation
        # Transaction representation
        # ----------------------------------

        edge_representation = torch.cat(
            [
                source_embedding,
                destination_embedding,
                target_edge_attr
            ],
            dim=-1
        )

        logits = self.classifier(
            edge_representation
        ).squeeze(-1)

        return logits

    # ======================================
    # COMPLETE FORWARD
    # ======================================

    def forward(
        self,
        x,
        edge_index,
        edge_attr,
        target_edge_index,
        target_edge_attr
    ):

        node_embeddings = self.encode(
            x,
            edge_index,
            edge_attr
        )

        logits = self.decode(
            node_embeddings,
            target_edge_index,
            target_edge_attr
        )

        return logits


# ==========================================
# CREATE MODEL
# ==========================================

model = GATAMLModel(
    node_in_dim=X_node.shape[1],
    edge_in_dim=E_features.shape[1],
    hidden_dim=64,
    heads=4,
    dropout=0.20
).to(DEVICE)


# ==========================================
# MODEL INFORMATION
# ==========================================

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("==========================================")
print("✅ FINAL GAT MODEL CREATED")
print("==========================================")

print(
    "Node input features :",
    X_node.shape[1]
)

print(
    "Edge input features :",
    E_features.shape[1]
)

print(
    "Hidden dimension     :",
    model.hidden_dim
)

print(
    "Attention heads      :",
    model.heads
)

print(
    "GAT layers           :",
    2
)

print(
    "Total parameters     :",
    f"{total_params:,}"
)

print(
    "Trainable parameters :",
    f"{trainable_params:,}"
)

print(
    "Device               :",
    next(model.parameters()).device
)

print("\n✅ CELL 7 COMPLETE")

✅ FINAL GAT MODEL CREATED
Node input features : 13
Edge input features : 20
Hidden dimension     : 64
Attention heads      : 4
GAT layers           : 2
Total parameters     : 32,385
Trainable parameters : 32,385
Device               : cuda:0

✅ CELL 7 COMPLETE


In [9]:
# ==========================================
# STAGE 1 — CELL 8
# FAST CPU NEIGHBORHOOD INDEX
# ==========================================

print("==========================================")
print("STAGE 1 — BUILDING NEIGHBOR INDEX")
print("==========================================")

t0 = time.time()

# ------------------------------------------
# Free duplicated PyG message graph if it exists
# ------------------------------------------

if "message_graph" in globals():
    del message_graph
    gc.collect()

# ------------------------------------------
# Original graph arrays
# ------------------------------------------

src_np = df_features["src_id"].to_numpy(
    dtype=np.int32,
    copy=False
)

dst_np = df_features["dst_id"].to_numpy(
    dtype=np.int32,
    copy=False
)

num_nodes = X_node.shape[0]
num_edges = len(df_features)

# ------------------------------------------
# Build undirected adjacency representation
#
# For every original transaction:
#
# A -> B
#
# store:
# A -> B
# B -> A
#
# The original transaction ID is retained,
# so we can reuse the original edge features.
# ------------------------------------------

node_index = np.concatenate(
    [
        src_np,
        dst_np
    ]
)

neighbor_index = np.concatenate(
    [
        dst_np,
        src_np
    ]
)

edge_id_index = np.concatenate(
    [
        np.arange(
            num_edges,
            dtype=np.int32
        ),
        np.arange(
            num_edges,
            dtype=np.int32
        )
    ]
)

# ------------------------------------------
# Sort adjacency by source node
# ------------------------------------------

print("Sorting adjacency index...")

order = np.argsort(
    node_index,
    kind="stable"
)

node_sorted = node_index[order]
neighbor_sorted = neighbor_index[order]
edge_id_sorted = edge_id_index[order]

# ------------------------------------------
# CSR-style pointer array
# ------------------------------------------

counts = np.bincount(
    node_sorted,
    minlength=num_nodes
)

adj_ptr = np.zeros(
    num_nodes + 1,
    dtype=np.int64
)

adj_ptr[1:] = np.cumsum(
    counts,
    dtype=np.int64
)

# ------------------------------------------
# Free temporary arrays
# ------------------------------------------

del node_index
del neighbor_index
del edge_id_index
del order
del node_sorted
del counts

gc.collect()

# ------------------------------------------
# Convert to compact dtypes
# ------------------------------------------

ADJ_NEIGHBORS = neighbor_sorted
ADJ_EDGE_IDS = edge_id_sorted

# ------------------------------------------
# Fast neighborhood sampler
# ------------------------------------------

def sample_neighbors(
    node_ids,
    num_neighbors=10,
    rng=None
):
    """
    Sample up to num_neighbors incident neighbors
    for every requested node.

    Returns:
        sampled_neighbors
        sampled_edge_ids
    """

    if rng is None:
        rng = np.random.default_rng(42)

    sampled_neighbors = []
    sampled_edge_ids = []

    for node in np.asarray(
        node_ids,
        dtype=np.int64
    ):

        start = adj_ptr[node]
        end = adj_ptr[node + 1]

        count = end - start

        if count == 0:
            continue

        if count <= num_neighbors:

            positions = np.arange(
                start,
                end
            )

        else:

            positions = rng.choice(
                np.arange(start, end),
                size=num_neighbors,
                replace=False
            )

        sampled_neighbors.append(
            ADJ_NEIGHBORS[positions]
        )

        sampled_edge_ids.append(
            ADJ_EDGE_IDS[positions]
        )

    if not sampled_neighbors:

        return (
            np.empty(0, dtype=np.int32),
            np.empty(0, dtype=np.int32)
        )

    return (
        np.concatenate(sampled_neighbors),
        np.concatenate(sampled_edge_ids)
    )


# ------------------------------------------
# Two-hop target-edge sampler
# ------------------------------------------

def sample_2hop_subgraph(
    target_edge_ids,
    num_neighbors_1=10,
    num_neighbors_2=10,
    seed=42
):
    """
    Builds a compact 2-hop subgraph around a batch
    of target transactions.

    Target edges remain identifiable through
    target_edge_ids.
    """

    rng = np.random.default_rng(seed)

    target_edge_ids = np.asarray(
        target_edge_ids,
        dtype=np.int64
    )

    target_src = src_np[
        target_edge_ids
    ]

    target_dst = dst_np[
        target_edge_ids
    ]

    # --------------------------------------
    # 1-hop sampling around target endpoints
    # --------------------------------------

    seed_nodes = np.unique(
        np.concatenate([
            target_src,
            target_dst
        ])
    )

    hop1_neighbors, hop1_edges = (
        sample_neighbors(
            seed_nodes,
            num_neighbors=num_neighbors_1,
            rng=rng
        )
    )

    nodes_h1 = np.unique(
        np.concatenate([
            seed_nodes,
            hop1_neighbors
        ])
    )

    # --------------------------------------
    # 2-hop sampling
    # --------------------------------------

    hop2_neighbors, hop2_edges = (
        sample_neighbors(
            nodes_h1,
            num_neighbors=num_neighbors_2,
            rng=rng
        )
    )

    all_nodes = np.unique(
        np.concatenate([
            nodes_h1,
            hop2_neighbors
        ])
    )

    # --------------------------------------
    # Message edges
    # --------------------------------------

    sampled_edge_ids = np.unique(
        np.concatenate([
            hop1_edges,
            hop2_edges,
            target_edge_ids.astype(np.int32)
        ])
    )

    # --------------------------------------
    # Local node mapping
    # --------------------------------------

    local_id = {
        int(node): i
        for i, node in enumerate(all_nodes)
    }

    local_src = []
    local_dst = []
    local_eids = []

    # Add both directions for message passing
    for eid in sampled_edge_ids:

        s = int(src_np[eid])
        d = int(dst_np[eid])

        if s in local_id and d in local_id:

            # Original direction
            local_src.append(
                local_id[s]
            )
            local_dst.append(
                local_id[d]
            )
            local_eids.append(
                int(eid)
            )

            # Reverse direction
            local_src.append(
                local_id[d]
            )
            local_dst.append(
                local_id[s]
            )
            local_eids.append(
                int(eid)
            )

    local_edge_index = np.asarray(
        [
            local_src,
            local_dst
        ],
        dtype=np.int64
    )

    local_edge_ids = np.asarray(
        local_eids,
        dtype=np.int64
    )

    # Target edge mapping
    target_local_src = np.asarray(
        [
            local_id[int(s)]
            for s in target_src
        ],
        dtype=np.int64
    )

    target_local_dst = np.asarray(
        [
            local_id[int(d)]
            for d in target_dst
        ],
        dtype=np.int64
    )

    target_local_edge_index = np.asarray(
        [
            target_local_src,
            target_local_dst
        ],
        dtype=np.int64
    )

    return {
        "nodes": all_nodes,
        "edge_index": local_edge_index,
        "edge_ids": local_edge_ids,
        "target_edge_index": target_local_edge_index,
        "target_edge_ids": target_edge_ids
    }


print("\n==========================================")
print("✅ NEIGHBOR INDEX READY")
print("==========================================")

print(
    "Nodes          :",
    f"{num_nodes:,}"
)

print(
    "Transactions   :",
    f"{num_edges:,}"
)

print(
    "Adjacency refs :",
    f"{len(ADJ_NEIGHBORS):,}"
)

print(
    "Index time     :",
    f"{time.time() - t0:.2f} sec"
)

print("\n✅ 2-hop sampler function ready")

STAGE 1 — BUILDING NEIGHBOR INDEX
Sorting adjacency index...

✅ NEIGHBOR INDEX READY
Nodes          : 1,213,224
Transactions   : 10,000,000
Adjacency refs : 20,000,000
Index time     : 1.64 sec

✅ 2-hop sampler function ready


In [10]:
# ==========================================
# STAGE 1 — CELL 9
# TEST ONE LOCAL 2-HOP GRAPH BATCH
# ==========================================

print("==========================================")
print("STAGE 1 — TESTING LOCAL GRAPH SAMPLER")
print("==========================================")

BATCH_SIZE = 1024

# First 1024 training transactions
target_edge_ids = train_idx[
    :BATCH_SIZE
].numpy()

t0 = time.time()

sampled = sample_2hop_subgraph(
    target_edge_ids=target_edge_ids,
    num_neighbors_1=10,
    num_neighbors_2=10,
    seed=42
)

print(
    f"Sampling time: "
    f"{time.time() - t0:.2f} sec"
)

print("\n==========================================")
print("LOCAL GRAPH")
print("==========================================")

print(
    "Target transactions :",
    len(sampled["target_edge_ids"])
)

print(
    "Local nodes         :",
    len(sampled["nodes"])
)

print(
    "Message edges       :",
    sampled["edge_index"].shape[1]
)

print(
    "Target edge index   :",
    sampled["target_edge_index"].shape
)

# ------------------------------------------
# Verify all target endpoints exist
# ------------------------------------------

assert (
    sampled["target_edge_index"].shape[1]
    ==
    BATCH_SIZE
)

assert (
    sampled["edge_index"].shape[1] > 0
)

print("\n✅ Target edges mapped correctly")
print("✅ Local graph created successfully")

# ------------------------------------------
# Prepare tensors
# ------------------------------------------

local_nodes = sampled["nodes"]

local_edge_index = torch.from_numpy(
    sampled["edge_index"]
).long()

local_edge_ids = sampled["edge_ids"]

local_target_edge_index = torch.from_numpy(
    sampled["target_edge_index"]
).long()

target_edge_ids = sampled["target_edge_ids"]

local_x = torch.from_numpy(
    X_node[local_nodes]
).float()

local_edge_attr = torch.from_numpy(
    E_features[local_edge_ids]
).float()

target_edge_attr = torch.from_numpy(
    E_features[target_edge_ids]
).float()

target_y = torch.from_numpy(
    df_features["Is Laundering"]
    .to_numpy(
        dtype=np.float32,
        copy=False
    )[target_edge_ids]
).float()

print("\n==========================================")
print("TENSOR SHAPES")
print("==========================================")

print(
    "Node features       :",
    tuple(local_x.shape)
)

print(
    "Message edge index  :",
    tuple(local_edge_index.shape)
)

print(
    "Message edge attrs  :",
    tuple(local_edge_attr.shape)
)

print(
    "Target edge index   :",
    tuple(local_target_edge_index.shape)
)

print(
    "Target edge attrs   :",
    tuple(target_edge_attr.shape)
)

print(
    "Target labels       :",
    tuple(target_y.shape)
)

print("\n✅ CELL 9 COMPLETE")

STAGE 1 — TESTING LOCAL GRAPH SAMPLER
Sampling time: 0.05 sec

LOCAL GRAPH
Target transactions : 1024
Local nodes         : 11404
Message edges       : 70070
Target edge index   : (2, 1024)

✅ Target edges mapped correctly
✅ Local graph created successfully

TENSOR SHAPES
Node features       : (11404, 13)
Message edge index  : (2, 70070)
Message edge attrs  : (70070, 20)
Target edge index   : (2, 1024)
Target edge attrs   : (1024, 20)
Target labels       : (1024,)

✅ CELL 9 COMPLETE


In [11]:
# ==========================================
# STAGE 1 — CELL 10
# GAT LOCAL BATCH FORWARD + BACKWARD TEST
# ==========================================

print("==========================================")
print("STAGE 1 — GAT BATCH TEST")
print("==========================================")

import time
import torch

# ------------------------------------------
# Move local batch to GPU
# ------------------------------------------

local_x_gpu = local_x.to(DEVICE)

local_edge_index_gpu = (
    local_edge_index.to(DEVICE)
)

local_edge_attr_gpu = (
    local_edge_attr.to(DEVICE)
)

local_target_edge_index_gpu = (
    local_target_edge_index.to(DEVICE)
)

target_edge_attr_gpu = (
    target_edge_attr.to(DEVICE)
)

target_y_gpu = (
    target_y.to(DEVICE)
)

# ------------------------------------------
# Optimizer + loss
# ------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.003,
    weight_decay=1e-4
)

criterion = torch.nn.BCEWithLogitsLoss()

# ------------------------------------------
# Forward
# ------------------------------------------

model.train()

optimizer.zero_grad(
    set_to_none=True
)

torch.cuda.empty_cache()

t0 = time.time()

logits = model(
    local_x_gpu,
    local_edge_index_gpu,
    local_edge_attr_gpu,
    local_target_edge_index_gpu,
    target_edge_attr_gpu
)

forward_time = time.time() - t0

print(
    f"Forward time: {forward_time:.4f} sec"
)

print(
    "Logits shape:",
    tuple(logits.shape)
)

# ------------------------------------------
# Loss
# ------------------------------------------

loss = criterion(
    logits,
    target_y_gpu
)

print(
    f"Loss: {loss.item():.6f}"
)

# ------------------------------------------
# Backward
# ------------------------------------------

t0 = time.time()

loss.backward()

backward_time = time.time() - t0

# ------------------------------------------
# Gradient clipping
# ------------------------------------------

grad_norm = torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    max_norm=1.0
)

# ------------------------------------------
# Optimizer update
# ------------------------------------------

optimizer.step()

total_time = (
    forward_time +
    backward_time
)

print(
    f"Backward time: {backward_time:.4f} sec"
)

print(
    f"Gradient norm: {float(grad_norm):.6f}"
)

print(
    f"Total step time: {total_time:.4f} sec"
)

print("\n==========================================")
print("✅ GAT FORWARD + BACKWARD SUCCESSFUL")
print("==========================================")

# ------------------------------------------
# GPU memory
# ------------------------------------------

allocated = (
    torch.cuda.memory_allocated()
    / 1024**3
)

reserved = (
    torch.cuda.memory_reserved()
    / 1024**3
)

print(
    f"GPU allocated: {allocated:.3f} GB"
)

print(
    f"GPU reserved : {reserved:.3f} GB"
)

STAGE 1 — GAT BATCH TEST
Forward time: 0.5784 sec
Logits shape: (1024,)
Loss: 0.632175
Backward time: 0.2993 sec
Gradient norm: 1.415900
Total step time: 0.8777 sec

✅ GAT FORWARD + BACKWARD SUCCESSFUL
GPU allocated: 0.024 GB
GPU reserved : 0.262 GB


In [12]:
# ==========================================
# STAGE 1 — CELL 11
# TRAIN-ONLY NEIGHBOR INDEX
# ==========================================

print("==========================================")
print("STAGE 1 — TRAIN-ONLY GRAPH INDEX")
print("==========================================")

t0 = time.time()

# ------------------------------------------
# Training transaction IDs
# ------------------------------------------

TRAIN_EDGE_IDS = train_idx.numpy()

print(
    "Training transactions:",
    f"{len(TRAIN_EDGE_IDS):,}"
)

# ------------------------------------------
# Training-only source/destination arrays
# ------------------------------------------

TRAIN_SRC = src_np[TRAIN_EDGE_IDS].astype(
    np.int32,
    copy=False
)

TRAIN_DST = dst_np[TRAIN_EDGE_IDS].astype(
    np.int32,
    copy=False
)

NUM_TRAIN_EDGES = len(TRAIN_EDGE_IDS)

# ------------------------------------------
# Build undirected adjacency
#
# For a training edge:
#
# A -> B
#
# store:
# A -> B
# B -> A
#
# ------------------------------------------

train_node_index = np.concatenate(
    [
        TRAIN_SRC,
        TRAIN_DST
    ]
)

train_neighbor_index = np.concatenate(
    [
        TRAIN_DST,
        TRAIN_SRC
    ]
)

# Local position inside TRAIN_EDGE_IDS
train_edge_position = np.concatenate(
    [
        np.arange(
            NUM_TRAIN_EDGES,
            dtype=np.int32
        ),
        np.arange(
            NUM_TRAIN_EDGES,
            dtype=np.int32
        )
    ]
)

# ------------------------------------------
# Sort adjacency by node
# ------------------------------------------

print("Sorting training adjacency...")

order = np.argsort(
    train_node_index,
    kind="stable"
)

TRAIN_ADJ_NEIGHBORS = (
    train_neighbor_index[order]
)

TRAIN_ADJ_EDGE_POS = (
    train_edge_position[order]
)

TRAIN_ADJ_PTR = np.zeros(
    X_node.shape[0] + 1,
    dtype=np.int64
)

counts = np.bincount(
    train_node_index,
    minlength=X_node.shape[0]
)

TRAIN_ADJ_PTR[1:] = np.cumsum(
    counts,
    dtype=np.int64
)

# ------------------------------------------
# Clean temporary arrays
# ------------------------------------------

del train_node_index
del train_neighbor_index
del train_edge_position
del order
del counts

gc.collect()

# ------------------------------------------
# Training-only neighbor sampler
# ------------------------------------------

def sample_train_neighbors(
    node_ids,
    num_neighbors=10,
    rng=None
):
    """
    Sample neighbors ONLY from training transactions.
    """

    if rng is None:
        rng = np.random.default_rng(42)

    sampled_neighbors = []
    sampled_edge_positions = []

    for node in np.asarray(
        node_ids,
        dtype=np.int64
    ):

        start = TRAIN_ADJ_PTR[node]
        end = TRAIN_ADJ_PTR[node + 1]

        count = end - start

        if count == 0:
            continue

        if count <= num_neighbors:

            positions = np.arange(
                start,
                end
            )

        else:

            positions = rng.choice(
                np.arange(start, end),
                size=num_neighbors,
                replace=False
            )

        sampled_neighbors.append(
            TRAIN_ADJ_NEIGHBORS[positions]
        )

        sampled_edge_positions.append(
            TRAIN_ADJ_EDGE_POS[positions]
        )

    if not sampled_neighbors:

        return (
            np.empty(0, dtype=np.int32),
            np.empty(0, dtype=np.int32)
        )

    return (
        np.concatenate(sampled_neighbors),
        np.concatenate(sampled_edge_positions)
    )


# ------------------------------------------
# TRAIN-ONLY 2-HOP SUBGRAPH
# ------------------------------------------

def sample_train_subgraph(
    target_edge_ids,
    num_neighbors_1=10,
    num_neighbors_2=10,
    seed=42
):
    """
    Creates a 2-hop local graph around target
    transactions using ONLY training edges
    for message passing.

    target_edge_ids are ORIGINAL transaction IDs.
    """

    rng = np.random.default_rng(seed)

    target_edge_ids = np.asarray(
        target_edge_ids,
        dtype=np.int64
    )

    target_src = src_np[
        target_edge_ids
    ]

    target_dst = dst_np[
        target_edge_ids
    ]

    # --------------------------------------
    # Target endpoints
    # --------------------------------------

    seed_nodes = np.unique(
        np.concatenate([
            target_src,
            target_dst
        ])
    )

    # --------------------------------------
    # 1-hop
    # --------------------------------------

    hop1_nodes, hop1_positions = (
        sample_train_neighbors(
            seed_nodes,
            num_neighbors=num_neighbors_1,
            rng=rng
        )
    )

    nodes_h1 = np.unique(
        np.concatenate([
            seed_nodes,
            hop1_nodes
        ])
    )

    # --------------------------------------
    # 2-hop
    # --------------------------------------

    hop2_nodes, hop2_positions = (
        sample_train_neighbors(
            nodes_h1,
            num_neighbors=num_neighbors_2,
            rng=rng
        )
    )

    all_nodes = np.unique(
        np.concatenate([
            nodes_h1,
            hop2_nodes
        ])
    )

    # --------------------------------------
    # Training transaction IDs involved
    # --------------------------------------

    train_positions = np.unique(
        np.concatenate([
            hop1_positions,
            hop2_positions
        ])
    )

    # Map training positions back to
    # original transaction IDs
    sampled_message_edge_ids = (
        TRAIN_EDGE_IDS[train_positions]
    )

    # --------------------------------------
    # Include target ONLY if it is a
    # training transaction
    # --------------------------------------

    target_set = set(
        target_edge_ids.tolist()
    )

    train_set = set(
        TRAIN_EDGE_IDS.tolist()
    )

    target_train_ids = [
        eid
        for eid in target_edge_ids
        if eid in train_set
    ]

    if target_train_ids:
        sampled_message_edge_ids = np.unique(
            np.concatenate([
                sampled_message_edge_ids,
                np.asarray(
                    target_train_ids,
                    dtype=np.int64
                )
            ])
        )

    # --------------------------------------
    # Local node mapping
    # --------------------------------------

    local_id = {
        int(node): i
        for i, node in enumerate(all_nodes)
    }

    local_src = []
    local_dst = []
    local_edge_ids = []

    # --------------------------------------
    # Add sampled training edges in both
    # directions for message passing
    # --------------------------------------

    for eid in sampled_message_edge_ids:

        s = int(src_np[eid])
        d = int(dst_np[eid])

        if (
            s in local_id
            and d in local_id
        ):

            # Original direction
            local_src.append(
                local_id[s]
            )

            local_dst.append(
                local_id[d]
            )

            local_edge_ids.append(
                int(eid)
            )

            # Reverse direction
            local_src.append(
                local_id[d]
            )

            local_dst.append(
                local_id[s]
            )

            local_edge_ids.append(
                int(eid)
            )

    local_edge_index = np.asarray(
        [
            local_src,
            local_dst
        ],
        dtype=np.int64
    )

    local_edge_ids = np.asarray(
        local_edge_ids,
        dtype=np.int64
    )

    # --------------------------------------
    # Target edges
    # --------------------------------------

    target_local_src = np.asarray(
        [
            local_id[int(s)]
            for s in target_src
        ],
        dtype=np.int64
    )

    target_local_dst = np.asarray(
        [
            local_id[int(d)]
            for d in target_dst
        ],
        dtype=np.int64
    )

    target_local_edge_index = np.asarray(
        [
            target_local_src,
            target_local_dst
        ],
        dtype=np.int64
    )

    return {
        "nodes": all_nodes,
        "edge_index": local_edge_index,
        "edge_ids": local_edge_ids,
        "target_edge_index": target_local_edge_index,
        "target_edge_ids": target_edge_ids
    }


print("\n==========================================")
print("✅ TRAIN-ONLY NEIGHBOR INDEX READY")
print("==========================================")

print(
    "Training edges indexed:",
    f"{NUM_TRAIN_EDGES:,}"
)

print(
    "Adjacency references:",
    f"{len(TRAIN_ADJ_NEIGHBORS):,}"
)

print(
    "Index creation time:",
    f"{time.time() - t0:.2f} sec"
)

print(
    "\n✅ Validation/Test will use this "
    "training graph for message passing."
)

STAGE 1 — TRAIN-ONLY GRAPH INDEX
Training transactions: 7,000,000
Sorting training adjacency...

✅ TRAIN-ONLY NEIGHBOR INDEX READY
Training edges indexed: 7,000,000
Adjacency references: 14,000,000
Index creation time: 1.05 sec

✅ Validation/Test will use this training graph for message passing.


In [13]:
# ==========================================
# STAGE 1 — CELL 12
# MINI-BATCH GRAPH BUILDER
# ==========================================

print("==========================================")
print("STAGE 1 — MINI-BATCH BUILDER")
print("==========================================")

BATCH_SIZE = 2048

NEIGHBORS_1 = 8
NEIGHBORS_2 = 8


def build_training_batch(
    target_edge_ids,
    seed=42
):
    """
    Builds a local 2-hop graph for a batch of
    target training transactions.

    Returns GPU-ready CPU tensors plus labels.
    """

    # --------------------------------------
    # Sample local graph
    # --------------------------------------

    sampled = sample_train_subgraph(
        target_edge_ids=target_edge_ids,
        num_neighbors_1=NEIGHBORS_1,
        num_neighbors_2=NEIGHBORS_2,
        seed=seed
    )

    local_nodes = sampled["nodes"]

    message_edge_ids = (
        sampled["edge_ids"]
    )

    target_edge_ids = (
        sampled["target_edge_ids"]
    )

    # --------------------------------------
    # Node features
    # --------------------------------------

    local_x = torch.from_numpy(
        X_node[local_nodes]
    ).float()

    # --------------------------------------
    # Message-passing edges
    # --------------------------------------

    local_edge_index = torch.from_numpy(
        sampled["edge_index"]
    ).long()

    local_edge_attr = torch.from_numpy(
        E_features[message_edge_ids]
    ).float()

    # --------------------------------------
    # Target transaction edges
    # --------------------------------------

    target_edge_index = torch.from_numpy(
        sampled["target_edge_index"]
    ).long()

    target_edge_attr = torch.from_numpy(
        E_features[target_edge_ids]
    ).float()

    # --------------------------------------
    # Target labels
    # --------------------------------------

    target_y = torch.from_numpy(
        df_features[
            "Is Laundering"
        ]
        .to_numpy(
            dtype=np.float32,
            copy=False
        )[target_edge_ids]
    ).float()

    return {
        "x": local_x,
        "edge_index": local_edge_index,
        "edge_attr": local_edge_attr,
        "target_edge_index": target_edge_index,
        "target_edge_attr": target_edge_attr,
        "y": target_y,
        "target_edge_ids": target_edge_ids,
        "node_ids": local_nodes
    }


# ==========================================
# TEST ONE REAL TRAINING BATCH
# ==========================================

target_ids = train_idx[
    :BATCH_SIZE
].numpy()

t0 = time.time()

batch = build_training_batch(
    target_edge_ids=target_ids,
    seed=42
)

print("\n==========================================")
print("✅ MINI-BATCH BUILDER READY")
print("==========================================")

print(
    "Target transactions :",
    len(batch["target_edge_ids"])
)

print(
    "Local nodes         :",
    batch["x"].shape[0]
)

print(
    "Message edges       :",
    batch["edge_index"].shape[1]
)

print(
    "Node features       :",
    tuple(batch["x"].shape)
)

print(
    "Message edge attrs  :",
    tuple(batch["edge_attr"].shape)
)

print(
    "Target edge attrs   :",
    tuple(batch["target_edge_attr"].shape)
)

print(
    "Labels              :",
    tuple(batch["y"].shape)
)

print(
    f"\nBatch construction time: "
    f"{time.time() - t0:.2f} sec"
)

print("\n✅ CELL 12 COMPLETE")

STAGE 1 — MINI-BATCH BUILDER

✅ MINI-BATCH BUILDER READY
Target transactions : 2048
Local nodes         : 19513
Message edges       : 111510
Node features       : (19513, 13)
Message edge attrs  : (111510, 20)
Target edge attrs   : (2048, 20)
Labels              : (2048,)

Batch construction time: 0.60 sec

✅ CELL 12 COMPLETE


In [17]:
# ==========================================
# STAGE 1 — CELL 12.5
# FAST VECTORIZED TRAINING SAMPLER
# ==========================================

import numpy as np
import torch
import time

print("==========================================")
print("STAGE 1 — FAST SAMPLER")
print("==========================================")


# ==========================================
# CONFIGURATION
# ==========================================

NEIGHBORS_1 = 8
NEIGHBORS_2 = 8


# ==========================================
# 1. FAST NEIGHBOR SAMPLER
# ==========================================

def sample_train_neighbors_fast(
    node_ids,
    num_neighbors=8,
    rng=None
):
    """
    Fast vectorized neighbor sampling.

    Uses the TRAINING-ONLY adjacency index:

        TRAIN_ADJ_PTR
        TRAIN_ADJ_NEIGHBORS
        TRAIN_ADJ_EDGE_POS
    """

    if rng is None:
        rng = np.random.default_rng(42)

    node_ids = np.asarray(
        node_ids,
        dtype=np.int64
    )

    if len(node_ids) == 0:
        return (
            np.empty(0, dtype=np.int32),
            np.empty(0, dtype=np.int32)
        )

    # --------------------------------------
    # Adjacency ranges for requested nodes
    # --------------------------------------

    starts = TRAIN_ADJ_PTR[
        node_ids
    ]

    ends = TRAIN_ADJ_PTR[
        node_ids + 1
    ]

    counts = (
        ends - starts
    )

    # Keep only nodes having neighbors
    valid = counts > 0

    starts = starts[valid]
    counts = counts[valid]

    if len(starts) == 0:
        return (
            np.empty(0, dtype=np.int32),
            np.empty(0, dtype=np.int32)
        )

    k = int(num_neighbors)

    # --------------------------------------
    # Random start inside each adjacency list
    # --------------------------------------

    max_offsets = np.maximum(
        counts - k + 1,
        1
    )

    random_offsets = (
        rng.random(len(starts))
        * max_offsets
    ).astype(np.int64)

    # --------------------------------------
    # Generate k candidate positions/node
    # --------------------------------------

    offsets = np.arange(
        k,
        dtype=np.int64
    )

    positions = (
        starts[:, None]
        +
        random_offsets[:, None]
        +
        offsets[None, :]
    )

    # --------------------------------------
    # Remove positions outside node range
    # --------------------------------------

    mask = (
        positions
        <
        (starts + counts)[:, None]
    )

    positions = positions[
        mask
    ]

    # --------------------------------------
    # Retrieve sampled neighbors
    # --------------------------------------

    sampled_neighbors = (
        TRAIN_ADJ_NEIGHBORS[
            positions
        ]
    )

    sampled_edge_positions = (
        TRAIN_ADJ_EDGE_POS[
            positions
        ]
    )

    return (
        sampled_neighbors.astype(
            np.int32,
            copy=False
        ),

        sampled_edge_positions.astype(
            np.int32,
            copy=False
        )
    )


# ==========================================
# 2. FAST 2-HOP TRAINING SUBGRAPH
# ==========================================

def sample_train_subgraph_fast(
    target_edge_ids,
    num_neighbors_1=8,
    num_neighbors_2=8,
    seed=42
):
    """
    Build a 2-hop local graph around target
    transactions.

    IMPORTANT:
    Message-passing edges come ONLY from
    TRAIN_EDGE_IDS.

    target_edge_ids are original transaction IDs.
    """

    rng = np.random.default_rng(
        seed
    )

    target_edge_ids = np.asarray(
        target_edge_ids,
        dtype=np.int64
    )

    # --------------------------------------
    # Target endpoints
    # --------------------------------------

    target_src = src_np[
        target_edge_ids
    ]

    target_dst = dst_np[
        target_edge_ids
    ]

    # --------------------------------------
    # Initial seed nodes
    # --------------------------------------

    seed_nodes = np.unique(
        np.concatenate([
            target_src,
            target_dst
        ])
    )

    # --------------------------------------
    # 1-HOP
    # --------------------------------------

    hop1_nodes, hop1_positions = (
        sample_train_neighbors_fast(
            seed_nodes,
            num_neighbors=num_neighbors_1,
            rng=rng
        )
    )

    nodes_h1 = np.unique(
        np.concatenate([
            seed_nodes,
            hop1_nodes
        ])
    )

    # --------------------------------------
    # 2-HOP
    # --------------------------------------

    hop2_nodes, hop2_positions = (
        sample_train_neighbors_fast(
            nodes_h1,
            num_neighbors=num_neighbors_2,
            rng=rng
        )
    )

    all_nodes = np.unique(
        np.concatenate([
            nodes_h1,
            hop2_nodes
        ])
    )

    # --------------------------------------
    # Training adjacency positions
    # --------------------------------------

    sampled_train_positions = np.unique(
        np.concatenate([
            hop1_positions,
            hop2_positions
        ])
    )

    # --------------------------------------
    # Convert train positions to ORIGINAL
    # transaction IDs
    # --------------------------------------

    sampled_message_edge_ids = (
        TRAIN_EDGE_IDS[
            sampled_train_positions
        ].astype(
            np.int64,
            copy=False
        )
    )

    # --------------------------------------
    # Include target transactions
    #
    # Targets are themselves training edges
    # during training.
    # --------------------------------------

    sampled_message_edge_ids = np.unique(
        np.concatenate([
            sampled_message_edge_ids,
            target_edge_ids
        ])
    )

    # --------------------------------------
    # Vectorized GLOBAL → LOCAL node mapping
    #
    # np.unique() gives sorted all_nodes.
    # --------------------------------------

    message_src_global = (
        src_np[
            sampled_message_edge_ids
        ]
    )

    message_dst_global = (
        dst_np[
            sampled_message_edge_ids
        ]
    )

    message_src_local = np.searchsorted(
        all_nodes,
        message_src_global
    )

    message_dst_local = np.searchsorted(
        all_nodes,
        message_dst_global
    )

    # --------------------------------------
    # Bidirectional message passing
    # --------------------------------------

    local_edge_index = np.vstack([
        np.concatenate([
            message_src_local,
            message_dst_local
        ]),

        np.concatenate([
            message_dst_local,
            message_src_local
        ])
    ]).astype(
        np.int64,
        copy=False
    )

    # Same original edge ID for both directions
    local_edge_ids = np.concatenate([
        sampled_message_edge_ids,
        sampled_message_edge_ids
    ]).astype(
        np.int64,
        copy=False
    )

    # --------------------------------------
    # Target edges → LOCAL node IDs
    # --------------------------------------

    target_local_src = np.searchsorted(
        all_nodes,
        target_src
    )

    target_local_dst = np.searchsorted(
        all_nodes,
        target_dst
    )

    target_local_edge_index = np.vstack([
        target_local_src,
        target_local_dst
    ]).astype(
        np.int64,
        copy=False
    )

    return {
        "nodes": all_nodes,

        "edge_index": local_edge_index,

        "edge_ids": local_edge_ids,

        "target_edge_index":
            target_local_edge_index,

        "target_edge_ids":
            target_edge_ids
    }


# ==========================================
# 3. FAST TRAINING BATCH BUILDER
# ==========================================

def build_training_batch_fast(
    target_edge_ids,
    seed=42
):
    """
    Build tensors for one training batch.
    """

    sampled = sample_train_subgraph_fast(
        target_edge_ids=target_edge_ids,
        num_neighbors_1=NEIGHBORS_1,
        num_neighbors_2=NEIGHBORS_2,
        seed=seed
    )

    local_nodes = (
        sampled["nodes"]
    )

    message_edge_ids = (
        sampled["edge_ids"]
    )

    target_edge_ids = (
        sampled["target_edge_ids"]
    )

    # --------------------------------------
    # Node features
    # --------------------------------------

    local_x = torch.from_numpy(
        X_node[
            local_nodes
        ]
    ).float()

    # --------------------------------------
    # Message graph
    # --------------------------------------

    local_edge_index = torch.from_numpy(
        sampled["edge_index"]
    ).long()

    local_edge_attr = torch.from_numpy(
        E_features[
            message_edge_ids
        ]
    ).float()

    # --------------------------------------
    # Target transaction edges
    # --------------------------------------

    target_edge_index = torch.from_numpy(
        sampled["target_edge_index"]
    ).long()

    target_edge_attr = torch.from_numpy(
        E_features[
            target_edge_ids
        ]
    ).float()

    # --------------------------------------
    # Target labels
    # --------------------------------------

    target_y = torch.from_numpy(
        df_features[
            "Is Laundering"
        ]
        .to_numpy(
            dtype=np.float32,
            copy=False
        )[
            target_edge_ids
        ]
    ).float()

    return {

        "x":
            local_x,

        "edge_index":
            local_edge_index,

        "edge_attr":
            local_edge_attr,

        "target_edge_index":
            target_edge_index,

        "target_edge_attr":
            target_edge_attr,

        "y":
            target_y,

        "target_edge_ids":
            target_edge_ids,

        "node_ids":
            local_nodes
    }


# ==========================================
# 4. TEST EXACTLY LIKE KAGGLE
# ==========================================

TEST_BATCH_SIZE = 8192

test_target_ids = (
    train_idx[
        :TEST_BATCH_SIZE
    ].numpy()
)

t0 = time.time()

fast_batch = build_training_batch_fast(
    target_edge_ids=test_target_ids,
    seed=42
)

elapsed = (
    time.time() - t0
)

print("\n==========================================")
print("✅ FAST SAMPLER READY")
print("==========================================")

print(
    "Target transactions :",
    len(
        fast_batch["target_edge_ids"]
    )
)

print(
    "Local nodes         :",
    fast_batch["x"].shape[0]
)

print(
    "Message edges       :",
    fast_batch[
        "edge_index"
    ].shape[1]
)

print(
    "Node features       :",
    tuple(
        fast_batch["x"].shape
    )
)

print(
    "Edge features       :",
    tuple(
        fast_batch["edge_attr"].shape
    )
)

print(
    "Target edge attrs   :",
    tuple(
        fast_batch[
            "target_edge_attr"
        ].shape
    )
)

print(
    "Labels              :",
    tuple(
        fast_batch["y"].shape
    )
)

print(
    "Batch build time    :",
    f"{elapsed:.4f} sec"
)

print("\n✅ CELL 12.5 COMPLETE")

STAGE 1 — FAST SAMPLER

✅ FAST SAMPLER READY
Target transactions : 8192
Local nodes         : 55976
Message edges       : 383050
Node features       : (55976, 13)
Edge features       : (383050, 20)
Target edge attrs   : (8192, 20)
Labels              : (8192,)
Batch build time    : 0.1008 sec

✅ CELL 12.5 COMPLETE


In [18]:
# ==========================================
# STAGE 1 — CELL 13
# FINAL 7M GAT TRAINING — DGX SPARK
# ==========================================

import os
import gc
import time
import numpy as np
import torch
import torch.nn as nn

# ==========================================
# CONFIGURATION
# ==========================================

EPOCHS = 3

TRAIN_BATCH_SIZE = 8192

NEIGHBORS_1 = 8
NEIGHBORS_2 = 8

LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4

CHECKPOINT_PATH = os.path.join(
    DATA_DIR,
    "gat_aml_stage1.pt"
)

# ==========================================
# FRESH MODEL
# ==========================================

model = GATAMLModel(
    node_in_dim=X_node.shape[1],
    edge_in_dim=E_features.shape[1],
    hidden_dim=64,
    heads=4,
    dropout=0.20
).to(DEVICE)

# ==========================================
# OPTIMIZER + LOSS
# ==========================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Dataset is exactly 50/50
criterion = nn.BCEWithLogitsLoss()

# ==========================================
# MIXED PRECISION
# ==========================================

USE_AMP = (
    DEVICE.type == "cuda"
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)

# ==========================================
# TRAINING IDS
# ==========================================

TRAIN_IDS = train_idx.numpy()

num_train = len(TRAIN_IDS)

num_batches = int(
    np.ceil(
        num_train /
        TRAIN_BATCH_SIZE
    )
)

rng = np.random.default_rng(
    SEED
)

# ==========================================
# GPU INFO
# ==========================================

print("==========================================")
print("STAGE 1 — FINAL GAT TRAINING")
print("==========================================")

print(
    "Device                :",
    DEVICE
)

if torch.cuda.is_available():
    print(
        "GPU                   :",
        torch.cuda.get_device_name(0)
    )

print(
    "Training transactions :",
    f"{num_train:,}"
)

print(
    "Batch size            :",
    f"{TRAIN_BATCH_SIZE:,}"
)

print(
    "Batches / epoch       :",
    f"{num_batches:,}"
)

print(
    "Neighbors             :",
    f"{NEIGHBORS_1} + {NEIGHBORS_2}"
)

print(
    "Epochs                :",
    EPOCHS
)

print(
    "Learning rate         :",
    LEARNING_RATE
)

print(
    "Mixed precision       :",
    USE_AMP
)

print(
    "Checkpoint             :",
    CHECKPOINT_PATH
)

print(
    "Model parameters      :",
    f"{sum(p.numel() for p in model.parameters()):,}"
)

# ==========================================
# TRAINING
# ==========================================

training_start = time.time()

for epoch in range(
    1,
    EPOCHS + 1
):

    epoch_start = time.time()

    model.train()

    # --------------------------------------
    # Shuffle only training transactions
    # --------------------------------------

    shuffled_ids = TRAIN_IDS.copy()

    rng.shuffle(
        shuffled_ids
    )

    epoch_loss = 0.0

    # --------------------------------------
    # MINI-BATCH TRAINING
    # --------------------------------------

    for batch_num, start in enumerate(
        range(
            0,
            num_train,
            TRAIN_BATCH_SIZE
        ),
        start=1
    ):

        end = min(
            start + TRAIN_BATCH_SIZE,
            num_train
        )

        target_ids = shuffled_ids[
            start:end
        ]

        # ----------------------------------
        # Build FAST local graph
        # ----------------------------------

        batch = build_training_batch_fast(
            target_edge_ids=target_ids,
            seed=(
                SEED
                + epoch * 100000
                + batch_num
            )
        )

        # ----------------------------------
        # Move batch to GPU
        # ----------------------------------

        x_gpu = batch["x"].to(
            DEVICE,
            non_blocking=True
        )

        edge_index_gpu = batch[
            "edge_index"
        ].to(
            DEVICE,
            non_blocking=True
        )

        edge_attr_gpu = batch[
            "edge_attr"
        ].to(
            DEVICE,
            non_blocking=True
        )

        target_edge_index_gpu = batch[
            "target_edge_index"
        ].to(
            DEVICE,
            non_blocking=True
        )

        target_edge_attr_gpu = batch[
            "target_edge_attr"
        ].to(
            DEVICE,
            non_blocking=True
        )

        y_gpu = batch[
            "y"
        ].to(
            DEVICE,
            non_blocking=True
        )

        # ----------------------------------
        # Forward
        # ----------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=USE_AMP
        ):

            logits = model(
                x_gpu,
                edge_index_gpu,
                edge_attr_gpu,
                target_edge_index_gpu,
                target_edge_attr_gpu
            )

            loss = criterion(
                logits,
                y_gpu
            )

        # ----------------------------------
        # Backward
        # ----------------------------------

        scaler.scale(
            loss
        ).backward()

        scaler.unscale_(
            optimizer
        )

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler.step(
            optimizer
        )

        scaler.update()

        # ----------------------------------
        # Record loss
        # ----------------------------------

        epoch_loss += loss.item()

        # ----------------------------------
        # Cleanup
        # ----------------------------------

        del (
            batch,
            x_gpu,
            edge_index_gpu,
            edge_attr_gpu,
            target_edge_index_gpu,
            target_edge_attr_gpu,
            y_gpu,
            logits,
            loss
        )

        # ----------------------------------
        # Progress
        # ----------------------------------

        if (
            batch_num % 100 == 0
            or batch_num == num_batches
        ):

            progress = (
                end /
                num_train
            ) * 100.0

            avg_loss = (
                epoch_loss /
                batch_num
            )

            elapsed = (
                time.time()
                -
                epoch_start
            )

            print(
                f"Epoch {epoch}/{EPOCHS} | "
                f"Batch {batch_num:,}/{num_batches:,} | "
                f"Progress {progress:6.2f}% | "
                f"Loss {avg_loss:.6f} | "
                f"Time {elapsed/60:.2f} min"
            )

    # ======================================
    # EPOCH COMPLETE
    # ======================================

    avg_loss = (
        epoch_loss /
        num_batches
    )

    epoch_time = (
        time.time()
        -
        epoch_start
    )

    print("\n------------------------------------------")
    print(
        f"Epoch {epoch}/{EPOCHS} COMPLETE"
    )

    print(
        f"Average Loss : {avg_loss:.6f}"
    )

    print(
        f"Epoch Time   : {epoch_time/60:.2f} min"
    )

    # ======================================
    # SAVE CHECKPOINT
    # ======================================

    checkpoint = {

        # Model
        "model_state":
            model.state_dict(),

        "model_config": {

            "node_in_dim":
                int(X_node.shape[1]),

            "edge_in_dim":
                int(E_features.shape[1]),

            "hidden_dim":
                64,

            "heads":
                4,

            "dropout":
                0.20
        },

        # Feature metadata
        "edge_feature_names":
            EDGE_FEATURE_NAMES,

        "node_feature_names":
            NODE_FEATURE_NAMES,

        # Feature engine
        "feature_engine":
            feature_engine,

        # Dataset information
        "num_nodes":
            int(X_node.shape[0]),

        "num_transactions":
            int(len(df_features)),

        "train_transactions":
            int(num_train),

        # Training configuration
        "epoch":
            epoch,

        "batch_size":
            TRAIN_BATCH_SIZE,

        "neighbors":
            [
                NEIGHBORS_1,
                NEIGHBORS_2
            ],

        "learning_rate":
            LEARNING_RATE,

        "weight_decay":
            WEIGHT_DECAY,

        "seed":
            SEED
    }

    torch.save(
        checkpoint,
        CHECKPOINT_PATH
    )

    checkpoint_size = (
        os.path.getsize(
            CHECKPOINT_PATH
        )
        / 1024**2
    )

    print(
        "✅ Checkpoint saved:"
    )

    print(
        CHECKPOINT_PATH
    )

    print(
        f"Checkpoint size: "
        f"{checkpoint_size:.2f} MB"
    )

    # --------------------------------------
    # Cleanup
    # --------------------------------------

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ==========================================
# FINAL RESULT
# ==========================================

total_time = (
    time.time()
    -
    training_start
)

print("\n==========================================")
print("✅ STAGE 1 TRAINING COMPLETE")
print("==========================================")

print(
    f"Total training time: "
    f"{total_time/60:.2f} min"
)

print(
    "Final model:",
    CHECKPOINT_PATH
)

print(
    "Model exists:",
    os.path.exists(
        CHECKPOINT_PATH
    )
)

print("\n✅ CELL 13 COMPLETE")

STAGE 1 — FINAL GAT TRAINING
Device                : cuda
GPU                   : NVIDIA GB10
Training transactions : 7,000,000
Batch size            : 8,192
Batches / epoch       : 855
Neighbors             : 8 + 8
Epochs                : 3
Learning rate         : 0.001
Mixed precision       : True
Checkpoint             : /home/llyods-aids5/AML/gat_aml_stage1.pt
Model parameters      : 32,385
Epoch 1/3 | Batch 100/855 | Progress  11.70% | Loss 0.116876 | Time 0.89 min
Epoch 1/3 | Batch 200/855 | Progress  23.41% | Loss 0.059807 | Time 1.77 min
Epoch 1/3 | Batch 300/855 | Progress  35.11% | Loss 0.040200 | Time 2.65 min
Epoch 1/3 | Batch 400/855 | Progress  46.81% | Loss 0.030279 | Time 3.54 min
Epoch 1/3 | Batch 500/855 | Progress  58.51% | Loss 0.024288 | Time 4.43 min
Epoch 1/3 | Batch 600/855 | Progress  70.22% | Loss 0.020280 | Time 5.31 min
Epoch 1/3 | Batch 700/855 | Progress  81.92% | Loss 0.017406 | Time 6.19 min
Epoch 1/3 | Batch 800/855 | Progress  93.62% | Loss 0.015247 | 

In [20]:
# ============================================================
# CELL 14 — CORRECTED VALIDATION + TEST EVALUATION
# ============================================================

import numpy as np
import torch
import json

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from tqdm.auto import tqdm


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

EVAL_BATCH_SIZE = 8192
EVAL_THRESHOLD = 0.50

VAL_IDS = val_idx.cpu().numpy()
TEST_IDS = test_idx.cpu().numpy()

print("=" * 70)
print("GAT AML MODEL EVALUATION")
print("=" * 70)

print(f"Validation transactions : {len(VAL_IDS):,}")
print(f"Test transactions       : {len(TEST_IDS):,}")
print(f"Evaluation batch size   : {EVAL_BATCH_SIZE:,}")
print(f"Threshold               : {EVAL_THRESHOLD}")
print(f"Device                  : {DEVICE}")
print()


# ============================================================
# VERIFY LABEL DISTRIBUTION BEFORE RUNNING MODEL
# ============================================================

all_labels_np = df_features["Is Laundering"].to_numpy()

val_labels_check = all_labels_np[VAL_IDS]
test_labels_check = all_labels_np[TEST_IDS]

print("LABEL DISTRIBUTION CHECK")
print("-" * 70)

print(
    "Validation:",
    {
        int(k): int(v)
        for k, v in zip(
            *np.unique(val_labels_check, return_counts=True)
        )
    }
)

print(
    "Test:",
    {
        int(k): int(v)
        for k, v in zip(
            *np.unique(test_labels_check, return_counts=True)
        )
    }
)

print()


# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate_split(edge_ids, split_name):

    model.eval()

    all_probs = []

    num_batches = (
        len(edge_ids) + EVAL_BATCH_SIZE - 1
    ) // EVAL_BATCH_SIZE

    print("=" * 70)
    print(f"EVALUATING {split_name}")
    print("=" * 70)

    with torch.inference_mode():

        for batch_no, start in enumerate(
            tqdm(
                range(0, len(edge_ids), EVAL_BATCH_SIZE),
                total=num_batches,
                desc=split_name
            ),
            start=1
        ):

            batch_ids = edge_ids[
                start:start + EVAL_BATCH_SIZE
            ]

            # ------------------------------------------------
            # Build graph around target transactions
            # ------------------------------------------------

            batch = build_training_batch_fast(
                batch_ids,
                seed=SEED + batch_no
            )

            x = batch["x"].to(
                DEVICE,
                non_blocking=True
            )

            edge_index = batch["edge_index"].to(
                DEVICE,
                non_blocking=True
            )

            edge_attr = batch["edge_attr"].to(
                DEVICE,
                non_blocking=True
            )

            target_edge_index = batch[
                "target_edge_index"
            ].to(
                DEVICE,
                non_blocking=True
            )

            target_edge_attr = batch[
                "target_edge_attr"
            ].to(
                DEVICE,
                non_blocking=True
            )

            # ------------------------------------------------
            # Model prediction
            # ------------------------------------------------

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(DEVICE.type == "cuda")
            ):

                logits = model(
                    x,
                    edge_index,
                    edge_attr,
                    target_edge_index,
                    target_edge_attr
                )

            probs = torch.sigmoid(logits)

            all_probs.append(
                probs.detach()
                .float()
                .cpu()
                .numpy()
                .reshape(-1)
            )

    # --------------------------------------------------------
    # Combine predictions
    # --------------------------------------------------------

    y_prob = np.concatenate(all_probs)

    # --------------------------------------------------------
    # IMPORTANT:
    # Labels are taken DIRECTLY from the target transaction IDs
    # --------------------------------------------------------

    y_true = all_labels_np[edge_ids].astype(np.int64)

    # Safety check
    if len(y_true) != len(y_prob):
        raise RuntimeError(
            f"Prediction/label length mismatch: "
            f"{len(y_prob)} predictions vs "
            f"{len(y_true)} labels"
        )

    y_pred = (
        y_prob >= EVAL_THRESHOLD
    ).astype(np.int64)

    # --------------------------------------------------------
    # Class distribution
    # --------------------------------------------------------

    unique_classes, class_counts = np.unique(
        y_true,
        return_counts=True
    )

    print()
    print("Target label distribution:")
    for cls, count in zip(
        unique_classes,
        class_counts
    ):
        name = (
            "Legitimate"
            if cls == 0
            else "Laundering"
        )
        print(
            f"  {cls} ({name}): {count:,}"
        )

    # --------------------------------------------------------
    # Standard classification metrics
    # --------------------------------------------------------

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    # ROC / PR require both classes
    if len(unique_classes) == 2:

        roc_auc = roc_auc_score(
            y_true,
            y_prob
        )

        pr_auc = average_precision_score(
            y_true,
            y_prob
        )

    else:

        roc_auc = float("nan")
        pr_auc = float("nan")

        print()
        print(
            "WARNING: Only one class is present; "
            "ROC-AUC and PR-AUC are undefined."
        )

    # --------------------------------------------------------
    # Confusion matrix
    # --------------------------------------------------------

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    # --------------------------------------------------------
    # RESULTS
    # --------------------------------------------------------

    print()
    print("=" * 70)
    print(f"{split_name} RESULTS")
    print("=" * 70)

    print(f"Precision : {precision:.6f}")
    print(f"Recall    : {recall:.6f}")
    print(f"F1 Score  : {f1:.6f}")
    print(f"ROC-AUC   : {roc_auc:.6f}")
    print(f"PR-AUC    : {pr_auc:.6f}")

    print()
    print("Confusion Matrix")
    print("-" * 30)
    print("                 Predicted")
    print("                 Legit   AML")
    print(f"Actual Legit     {tn:7,} {fp:7,}")
    print(f"Actual AML       {fn:7,} {tp:7,}")

    # --------------------------------------------------------
    # Class-wise report
    # --------------------------------------------------------

    print()
    print("CLASS-WISE METRICS")
    print("-" * 70)

    print(
        classification_report(
            y_true,
            y_pred,
            labels=[0, 1],
            target_names=[
                "Legitimate",
                "Laundering"
            ],
            digits=6,
            zero_division=0
        )
    )

    # --------------------------------------------------------
    # AML alert statistics
    # --------------------------------------------------------

    total = len(y_true)
    actual_laundering = tp + fn
    predicted_alerts = tp + fp

    print("AML ALERT STATISTICS")
    print("-" * 70)

    print(f"Total transactions : {total:,}")
    print(
        f"Actual laundering  : {actual_laundering:,}"
    )
    print(
        f"Predicted alerts   : {predicted_alerts:,}"
    )
    print(f"True positives     : {tp:,}")
    print(f"False positives    : {fp:,}")
    print(f"False negatives    : {fn:,}")
    print(f"True negatives     : {tn:,}")

    print(
        f"Alert rate         : "
        f"{predicted_alerts / total:.4%}"
    )

    # --------------------------------------------------------
    # Return results
    # --------------------------------------------------------

    return {
        "y_true": y_true,
        "y_prob": y_prob,
        "y_pred": y_pred,
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "confusion_matrix": cm
    }


# ============================================================
# VALIDATION
# ============================================================

val_results = evaluate_split(
    VAL_IDS,
    "VALIDATION"
)


# ============================================================
# TEST
# ============================================================

test_results = evaluate_split(
    TEST_IDS,
    "TEST"
)


# ============================================================
# SAVE SUMMARY
# ============================================================

results_summary = {
    "threshold": EVAL_THRESHOLD,

    "validation": {
        "precision": val_results["precision"],
        "recall": val_results["recall"],
        "f1": val_results["f1"],
        "roc_auc": val_results["roc_auc"],
        "pr_auc": val_results["pr_auc"],
        "confusion_matrix":
            val_results["confusion_matrix"].tolist()
    },

    "test": {
        "precision": test_results["precision"],
        "recall": test_results["recall"],
        "f1": test_results["f1"],
        "roc_auc": test_results["roc_auc"],
        "pr_auc": test_results["pr_auc"],
        "confusion_matrix":
            test_results["confusion_matrix"].tolist()
    }
}

EVAL_FILE = os.path.join(
    DATA_DIR,
    "gat_aml_evaluation.json"
)

with open(EVAL_FILE, "w") as f:
    json.dump(
        results_summary,
        f,
        indent=2,
        allow_nan=True
    )

print()
print("=" * 70)
print("EVALUATION COMPLETE")
print("=" * 70)
print(f"Saved results: {EVAL_FILE}")

GAT AML MODEL EVALUATION
Validation transactions : 1,500,000
Test transactions       : 1,500,000
Evaluation batch size   : 8,192
Threshold               : 0.5
Device                  : cuda

LABEL DISTRIBUTION CHECK
----------------------------------------------------------------------
Validation: {1: 1500000}
Test: {1: 1500000}

EVALUATING VALIDATION


VALIDATION:   0%|          | 0/184 [00:00<?, ?it/s]


Target label distribution:
  1 (Laundering): 1,500,000


VALIDATION RESULTS
Precision : 1.000000
Recall    : 1.000000
F1 Score  : 1.000000
ROC-AUC   : nan
PR-AUC    : nan

Confusion Matrix
------------------------------
                 Predicted
                 Legit   AML
Actual Legit           0       0
Actual AML             0 1,500,000

CLASS-WISE METRICS
----------------------------------------------------------------------
              precision    recall  f1-score   support

  Legitimate   0.000000  0.000000  0.000000         0
  Laundering   1.000000  1.000000  1.000000   1500000

   micro avg   1.000000  1.000000  1.000000   1500000
   macro avg   0.500000  0.500000  0.500000   1500000
weighted avg   1.000000  1.000000  1.000000   1500000

AML ALERT STATISTICS
----------------------------------------------------------------------
Total transactions : 1,500,000
Actual laundering  : 1,500,000
Predicted alerts   : 1,500,000
True positives     : 1,500,000
False positives    : 

TEST:   0%|          | 0/184 [00:00<?, ?it/s]


Target label distribution:
  1 (Laundering): 1,500,000


TEST RESULTS
Precision : 1.000000
Recall    : 1.000000
F1 Score  : 1.000000
ROC-AUC   : nan
PR-AUC    : nan

Confusion Matrix
------------------------------
                 Predicted
                 Legit   AML
Actual Legit           0       0
Actual AML             0 1,500,000

CLASS-WISE METRICS
----------------------------------------------------------------------
              precision    recall  f1-score   support

  Legitimate   0.000000  0.000000  0.000000         0
  Laundering   1.000000  1.000000  1.000000   1500000

   micro avg   1.000000  1.000000  1.000000   1500000
   macro avg   0.500000  0.500000  0.500000   1500000
weighted avg   1.000000  1.000000  1.000000   1500000

AML ALERT STATISTICS
----------------------------------------------------------------------
Total transactions : 1,500,000
Actual laundering  : 1,500,000
Predicted alerts   : 1,500,000
True positives     : 1,500,000
False positives    : 0
Fals

In [21]:
# ============================================================
# CELL 15 — DIAGNOSE TRAIN / VAL / TEST SPLIT
# ============================================================

import numpy as np
import pandas as pd

labels = df_features["Is Laundering"].to_numpy()

# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------

def inspect_split(name, edge_ids):

    edge_ids = np.asarray(edge_ids)

    y = labels[edge_ids]

    unique, counts = np.unique(
        y,
        return_counts=True
    )

    print("=" * 75)
    print(name)
    print("=" * 75)

    print(f"Transactions : {len(edge_ids):,}")

    for cls, count in zip(unique, counts):
        label_name = (
            "Legitimate"
            if int(cls) == 0
            else "Laundering"
        )

        print(
            f"{label_name:12s} : "
            f"{count:>12,} "
            f"({count / len(y):.2%})"
        )

    # --------------------------------------------------------
    # Timestamp information
    # --------------------------------------------------------

    ts = pd.to_datetime(
        df_features.iloc[edge_ids]["Timestamp"],
        errors="coerce"
    )

    print()
    print(
        f"Timestamp min : {ts.min()}"
    )
    print(
        f"Timestamp max : {ts.max()}"
    )

    # --------------------------------------------------------
    # Label changes inside split
    # --------------------------------------------------------

    if len(y) > 1:

        changes = np.sum(
            y[1:] != y[:-1]
        )

        print(
            f"Adjacent label changes : {changes:,}"
        )

    print()


# ============================================================
# INSPECT ALL SPLITS
# ============================================================

inspect_split(
    "TRAIN",
    train_idx.cpu().numpy()
)

inspect_split(
    "VALIDATION",
    val_idx.cpu().numpy()
)

inspect_split(
    "TEST",
    test_idx.cpu().numpy()
)


# ============================================================
# GLOBAL DATASET DISTRIBUTION
# ============================================================

print("=" * 75)
print("GLOBAL DATASET DISTRIBUTION")
print("=" * 75)

unique, counts = np.unique(
    labels,
    return_counts=True
)

for cls, count in zip(unique, counts):

    label_name = (
        "Legitimate"
        if int(cls) == 0
        else "Laundering"
    )

    print(
        f"{label_name:12s} : "
        f"{count:>12,} "
        f"({count / len(labels):.2%})"
    )


# ============================================================
# CHECK FIRST / LAST TRANSACTIONS
# ============================================================

print()
print("=" * 75)
print("FIRST / LAST LABELS AFTER FEATURE ENGINE SORT")
print("=" * 75)

print(
    "First 30 labels:",
    labels[:30].tolist()
)

print(
    "Last 30 labels :",
    labels[-30:].tolist()
)


# ============================================================
# LABEL COUNTS BY TIME
# ============================================================

tmp = df_features[
    ["Timestamp", "Is Laundering"]
].copy()

tmp["Timestamp"] = pd.to_datetime(
    tmp["Timestamp"],
    errors="coerce"
)

tmp["date"] = tmp["Timestamp"].dt.date

daily = (
    tmp.groupby(
        ["date", "Is Laundering"]
    )
    .size()
    .unstack(fill_value=0)
)

print()
print("=" * 75)
print("DAILY LABEL DISTRIBUTION — FIRST 15 DAYS")
print("=" * 75)

print(daily.head(15))

print()
print("=" * 75)
print("DAILY LABEL DISTRIBUTION — LAST 15 DAYS")
print("=" * 75)

print(daily.tail(15))

TRAIN
Transactions : 7,000,000
Legitimate   :    5,000,000 (71.43%)
Laundering   :    2,000,000 (28.57%)

Timestamp min : 2022-09-01 00:00:00
Timestamp max : 2022-09-18 11:18:00
Adjacent label changes : 1

VALIDATION
Transactions : 1,500,000
Laundering   :    1,500,000 (100.00%)

Timestamp min : NaT
Timestamp max : NaT
Adjacent label changes : 0

TEST
Transactions : 1,500,000
Laundering   :    1,500,000 (100.00%)

Timestamp min : NaT
Timestamp max : NaT
Adjacent label changes : 0

GLOBAL DATASET DISTRIBUTION
Legitimate   :    5,000,000 (50.00%)
Laundering   :    5,000,000 (50.00%)

FIRST / LAST LABELS AFTER FEATURE ENGINE SORT
First 30 labels: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Last 30 labels : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

DAILY LABEL DISTRIBUTION — FIRST 15 DAYS
Is Laundering        0
date                  
2022-09-01     1098587
2022-09-02      743217
2022-09-03     

In [22]:
# ============================================================
# CELL 16 — INSPECT TIMESTAMP / SORTING PROBLEM
# ============================================================

print("=" * 75)
print("TIMESTAMP / DATA ORDER DIAGNOSTIC")
print("=" * 75)

# ------------------------------------------------------------
# 1. df_features structure
# ------------------------------------------------------------

print("\ndf_features shape:", df_features.shape)

print("\nColumn dtypes:")
print(df_features.dtypes)

# ------------------------------------------------------------
# 2. Last legitimate rows
# ------------------------------------------------------------

labels_np = df_features["Is Laundering"].to_numpy()

legit_ids = np.where(labels_np == 0)[0]
aml_ids = np.where(labels_np == 1)[0]

print("\nNumber of legitimate rows :", len(legit_ids))
print("Number of laundering rows:", len(aml_ids))

print("\n--- LAST 10 LEGITIMATE ROWS ---")

print(
    df_features.iloc[
        legit_ids[-10:]
    ][[
        "Timestamp",
        "Is Laundering"
    ]].to_string(index=True)
)

# ------------------------------------------------------------
# 3. FIRST 10 laundering rows
# ------------------------------------------------------------

print("\n--- FIRST 10 LAUNDERING ROWS ---")

print(
    df_features.iloc[
        aml_ids[:10]
    ][[
        "Timestamp",
        "Is Laundering"
    ]].to_string(index=True)
)

# ------------------------------------------------------------
# 4. LAST 10 laundering rows
# ------------------------------------------------------------

print("\n--- LAST 10 LAUNDERING ROWS ---")

print(
    df_features.iloc[
        aml_ids[-10:]
    ][[
        "Timestamp",
        "Is Laundering"
    ]].to_string(index=True)
)

# ------------------------------------------------------------
# 5. Timestamp validity by class
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("TIMESTAMP VALIDITY BY CLASS")
print("=" * 75)

for cls, name in [(0, "LEGITIMATE"), (1, "LAUNDERING")]:

    cls_ts = df_features.loc[
        df_features["Is Laundering"] == cls,
        "Timestamp"
    ]

    valid = cls_ts.notna().sum()
    missing = cls_ts.isna().sum()

    print(f"\n{name}")
    print(f"Valid timestamps   : {valid:,}")
    print(f"Missing timestamps : {missing:,}")

    if valid > 0:
        print(f"Minimum timestamp  : {cls_ts.min()}")
        print(f"Maximum timestamp  : {cls_ts.max()}")

# ------------------------------------------------------------
# 6. Check whether df exists
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("ORIGINAL DATAFRAME CHECK")
print("=" * 75)

if "df" in globals():

    print("df exists.")
    print("df shape:", df.shape)

    print("\nOriginal df dtypes:")
    print(df.dtypes)

    print("\nOriginal last 10 rows:")
    print(
        df.tail(10).to_string()
    )

else:

    print("Original `df` variable is not available.")

# ------------------------------------------------------------
# 7. Inspect raw CSV directly — only a small sample
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("RAW CSV SAMPLE")
print("=" * 75)

raw_sample = pd.read_csv(
    CSV_FILE,
    nrows=5
)

print("\nFirst 5 raw rows:")
print(
    raw_sample.to_string(index=False)
)

print("\nRaw columns:")
print(raw_sample.columns.tolist())

print("\nRaw Timestamp dtype:")
print(
    raw_sample["Timestamp"].dtype
    if "Timestamp" in raw_sample.columns
    else "Timestamp column not found"
)

# ------------------------------------------------------------
# 8. Read a few rows from the END of the CSV
# ------------------------------------------------------------

print("\n--- LAST 5 RAW CSV ROWS ---")

try:

    raw_tail = pd.read_csv(
        CSV_FILE,
        skipfooter=5,
        engine="python"
    )

    print(
        raw_tail.tail(5).to_string(index=False)
    )

except Exception as e:

    print(
        "Could not read CSV tail:",
        repr(e)
    )

TIMESTAMP / DATA ORDER DIAGNOSTIC

df_features shape: (10000000, 20)

Column dtypes:
transaction_id                object
Timestamp             datetime64[ns]
From Bank                      int64
Account                       object
To Bank                        int64
Account.1                     object
Amount Received              float64
Receiving Currency            object
Amount Paid                  float64
Payment Currency              object
Payment Format                object
Is Laundering                   int8
row_id                         int64
amt_paid_usd                 float32
amt_recv_usd                 float32
src_key                       object
dst_key                       object
pair_key                      object
src_id                         int64
dst_id                         int64
dtype: object

Number of legitimate rows : 5000000
Number of laundering rows: 5000000

--- LAST 10 LEGITIMATE ROWS ---
                  Timestamp  Is Laundering
4999990 2022-

In [23]:
# ============================================================
# CELL 17 — FIX TIMESTAMPS AND VERIFY
# ============================================================

print("=" * 75)
print("FIXING TIMESTAMP PARSING")
print("=" * 75)

# ------------------------------------------------------------
# Parse timestamp robustly from the original dataframe
# Handles formats such as:
#   2022/09/01 00:20
#   2022-09-02 19:08:00
# ------------------------------------------------------------

raw_timestamp = df["Timestamp"].astype(str)

parsed_timestamp = pd.to_datetime(
    raw_timestamp,
    format="mixed",
    errors="coerce"
)

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

print(
    f"Total timestamps      : {len(parsed_timestamp):,}"
)

print(
    f"Valid timestamps      : {parsed_timestamp.notna().sum():,}"
)

print(
    f"Invalid timestamps    : {parsed_timestamp.isna().sum():,}"
)

print()

for cls, name in [(0, "LEGITIMATE"), (1, "LAUNDERING")]:

    mask = df["Is Laundering"].to_numpy() == cls

    ts_cls = parsed_timestamp[mask]

    print(name)

    print(
        f"  Valid   : {ts_cls.notna().sum():,}"
    )

    print(
        f"  Missing : {ts_cls.isna().sum():,}"
    )

    if ts_cls.notna().any():

        print(
            f"  Min     : {ts_cls.min()}"
        )

        print(
            f"  Max     : {ts_cls.max()}"
        )

    print()

# ------------------------------------------------------------
# Compare a few laundering rows
# ------------------------------------------------------------

aml_sample = np.where(
    df["Is Laundering"].to_numpy() == 1
)[0][:10]

print("=" * 75)
print("LAUNDERING TIMESTAMP SAMPLE")
print("=" * 75)

for idx in aml_sample:

    print(
        f"Row {idx:>10,} | "
        f"Raw: {df.iloc[idx]['Timestamp']} | "
        f"Parsed: {parsed_timestamp.iloc[idx]}"
    )

# ------------------------------------------------------------
# Update original dataframe timestamp
# ------------------------------------------------------------

df["Timestamp"] = parsed_timestamp

# ------------------------------------------------------------
# Update feature dataframe timestamp WITHOUT changing row order
# ------------------------------------------------------------

df_features["Timestamp"] = parsed_timestamp

print()
print("=" * 75)
print("FINAL VERIFICATION")
print("=" * 75)

print(
    "df_features Timestamp missing:",
    df_features["Timestamp"].isna().sum()
)

print(
    "df_features Timestamp min:",
    df_features["Timestamp"].min()
)

print(
    "df_features Timestamp max:",
    df_features["Timestamp"].max()
)

FIXING TIMESTAMP PARSING
Total timestamps      : 10,000,000
Valid timestamps      : 5,000,000
Invalid timestamps    : 5,000,000

LEGITIMATE
  Valid   : 5,000,000
  Missing : 0
  Min     : 2022-09-01 00:00:00
  Max     : 2022-09-18 11:18:00

LAUNDERING
  Valid   : 0
  Missing : 5,000,000

LAUNDERING TIMESTAMP SAMPLE
Row  5,000,000 | Raw: NaT | Parsed: NaT
Row  5,000,001 | Raw: NaT | Parsed: NaT
Row  5,000,002 | Raw: NaT | Parsed: NaT
Row  5,000,003 | Raw: NaT | Parsed: NaT
Row  5,000,004 | Raw: NaT | Parsed: NaT
Row  5,000,005 | Raw: NaT | Parsed: NaT
Row  5,000,006 | Raw: NaT | Parsed: NaT
Row  5,000,007 | Raw: NaT | Parsed: NaT
Row  5,000,008 | Raw: NaT | Parsed: NaT
Row  5,000,009 | Raw: NaT | Parsed: NaT

FINAL VERIFICATION
df_features Timestamp missing: 5000000
df_features Timestamp min: 2022-09-01 00:00:00
df_features Timestamp max: 2022-09-18 11:18:00


In [24]:
# ============================================================
# CELL 18 — RELOAD RAW TIMESTAMPS DIRECTLY FROM CSV
# ============================================================

print("=" * 75)
print("RELOADING TIMESTAMPS DIRECTLY FROM RAW CSV")
print("=" * 75)

# ------------------------------------------------------------
# Read ONLY the Timestamp column from the original CSV
# This avoids reloading the full 10M-row dataset.
# ------------------------------------------------------------

print("Reading raw Timestamp column...")

raw_ts = pd.read_csv(
    CSV_FILE,
    usecols=["Timestamp"],
    dtype={"Timestamp": "string"}
)["Timestamp"]

print(f"Raw timestamp rows loaded: {len(raw_ts):,}")

# ------------------------------------------------------------
# Parse mixed timestamp formats
# Examples:
#   2022/09/01 00:20
#   2022-09-02 19:08:00
# ------------------------------------------------------------

fixed_ts = pd.to_datetime(
    raw_ts,
    format="mixed",
    errors="coerce"
)

# ------------------------------------------------------------
# Verify parsing
# ------------------------------------------------------------

valid_count = fixed_ts.notna().sum()
missing_count = fixed_ts.isna().sum()

print()
print("TIMESTAMP PARSING RESULT")
print("-" * 75)
print(f"Valid timestamps   : {valid_count:,}")
print(f"Missing timestamps : {missing_count:,}")

# ------------------------------------------------------------
# Class-wise verification
# ------------------------------------------------------------

labels_np = df["Is Laundering"].to_numpy()

print()
print("CLASS-WISE TIMESTAMP CHECK")
print("-" * 75)

for cls, name in [(0, "LEGITIMATE"), (1, "LAUNDERING")]:

    mask = labels_np == cls

    ts_cls = fixed_ts[mask]

    print(f"\n{name}")
    print(f"Valid   : {ts_cls.notna().sum():,}")
    print(f"Missing : {ts_cls.isna().sum():,}")

    if ts_cls.notna().any():
        print(f"Min     : {ts_cls.min()}")
        print(f"Max     : {ts_cls.max()}")

# ------------------------------------------------------------
# Show actual laundering timestamps
# ------------------------------------------------------------

aml_positions = np.where(
    labels_np == 1
)[0][:10]

print()
print("=" * 75)
print("FIRST 10 LAUNDERING TIMESTAMPS FROM RAW CSV")
print("=" * 75)

for idx in aml_positions:

    print(
        f"Row {idx:>10,} | "
        f"Raw: {raw_ts.iloc[idx]} | "
        f"Parsed: {fixed_ts.iloc[idx]}"
    )

# ------------------------------------------------------------
# Safety check
# ------------------------------------------------------------

if missing_count != 0:

    raise RuntimeError(
        f"Timestamp parsing still failed for "
        f"{missing_count:,} rows."
    )

# ------------------------------------------------------------
# Replace broken timestamps
# ------------------------------------------------------------

df["Timestamp"] = fixed_ts.to_numpy()
df_features["Timestamp"] = fixed_ts.to_numpy()

print()
print("=" * 75)
print("TIMESTAMPS SUCCESSFULLY REPAIRED")
print("=" * 75)

print(
    "df missing timestamps        :",
    df["Timestamp"].isna().sum()
)

print(
    "df_features missing timestamps:",
    df_features["Timestamp"].isna().sum()
)

print(
    "Global minimum:",
    df["Timestamp"].min()
)

print(
    "Global maximum:",
    df["Timestamp"].max()
)

RELOADING TIMESTAMPS DIRECTLY FROM RAW CSV
Reading raw Timestamp column...
Raw timestamp rows loaded: 10,000,000

TIMESTAMP PARSING RESULT
---------------------------------------------------------------------------
Valid timestamps   : 10,000,000
Missing timestamps : 0

CLASS-WISE TIMESTAMP CHECK
---------------------------------------------------------------------------

LEGITIMATE
Valid   : 5,000,000
Missing : 0
Min     : 2022-09-01 00:00:00
Max     : 2022-09-18 11:18:00

LAUNDERING
Valid   : 5,000,000
Missing : 0
Min     : 2022-09-01 00:00:00
Max     : 2022-09-18 11:18:00

FIRST 10 LAUNDERING TIMESTAMPS FROM RAW CSV
Row  5,000,000 | Raw: 2022-09-09 19:45:00 | Parsed: 2022-09-09 19:45:00
Row  5,000,001 | Raw: 2022-09-13 12:44:00 | Parsed: 2022-09-13 12:44:00
Row  5,000,002 | Raw: 2022-09-01 00:20:00 | Parsed: 2022-09-01 00:20:00
Row  5,000,003 | Raw: 2022-09-01 00:47:00 | Parsed: 2022-09-01 00:47:00
Row  5,000,004 | Raw: 2022-09-01 06:20:00 | Parsed: 2022-09-01 06:20:00
Row  5,000,00

In [25]:
# ============================================================
# CELL 19 — REBUILD AML FEATURES WITH CORRECT TIMESTAMPS
# ============================================================

print("=" * 75)
print("REBUILDING AML FEATURES")
print("=" * 75)

# ------------------------------------------------------------
# Safety check
# ------------------------------------------------------------

assert df["Timestamp"].notna().all(), \
    "df still contains missing timestamps."

assert df_features["Timestamp"].notna().all(), \
    "df_features still contains missing timestamps."

print(
    f"Input transactions : {len(df):,}"
)

print(
    f"Timestamp missing   : {df['Timestamp'].isna().sum():,}"
)

# ------------------------------------------------------------
# Re-run the feature engine
# ------------------------------------------------------------

print("\nRunning AMLFeatureEngine.transform(...) ...")
print("This will regenerate the edge + node features.")
print()

feature_engine = AMLFeatureEngine()

(
    df_features,
    X_node,
    E_features,
    edge_feature_names,
    node_feature_names
) = feature_engine.transform(
    df.copy()
)

# ------------------------------------------------------------
# Convert / verify feature arrays
# ------------------------------------------------------------

X_node = np.asarray(
    X_node,
    dtype=np.float32
)

E_features = np.asarray(
    E_features,
    dtype=np.float32
)

# ------------------------------------------------------------
# Verify dimensions
# ------------------------------------------------------------

print()
print("=" * 75)
print("FEATURE REBUILD COMPLETE")
print("=" * 75)

print(
    f"Transactions       : {len(df_features):,}"
)

print(
    f"Node features      : {X_node.shape}"
)

print(
    f"Edge features      : {E_features.shape}"
)

print(
    f"Edge feature count : {len(edge_feature_names)}"
)

print(
    f"Node feature count : {len(node_feature_names)}"
)

# ------------------------------------------------------------
# Timestamp verification
# ------------------------------------------------------------

print()
print("TIMESTAMP VERIFICATION")
print("-" * 75)

print(
    f"Missing timestamps : "
    f"{df_features['Timestamp'].isna().sum():,}"
)

print(
    f"Minimum timestamp  : "
    f"{df_features['Timestamp'].min()}"
)

print(
    f"Maximum timestamp  : "
    f"{df_features['Timestamp'].max()}"
)

# ------------------------------------------------------------
# Class distribution
# ------------------------------------------------------------

print()
print("GLOBAL CLASS DISTRIBUTION")
print("-" * 75)

class_counts = (
    df_features["Is Laundering"]
    .value_counts()
    .sort_index()
)

for cls, count in class_counts.items():

    name = (
        "Legitimate"
        if int(cls) == 0
        else "Laundering"
    )

    print(
        f"{name:12s} : "
        f"{int(count):>12,} "
        f"({count / len(df_features):.2%})"
    )

# ------------------------------------------------------------
# Feature NaN / Inf check
# ------------------------------------------------------------

print()
print("FEATURE VALIDITY CHECK")
print("-" * 75)

edge_nan = np.isnan(E_features).sum()
edge_inf = np.isinf(E_features).sum()

node_nan = np.isnan(X_node).sum()
node_inf = np.isinf(X_node).sum()

print(f"Edge NaNs : {edge_nan:,}")
print(f"Edge Infs : {edge_inf:,}")
print(f"Node NaNs : {node_nan:,}")
print(f"Node Infs : {node_inf:,}")

if edge_nan > 0 or edge_inf > 0:
    print("\nWARNING: Edge features contain invalid values.")

if node_nan > 0 or node_inf > 0:
    print("\nWARNING: Node features contain invalid values.")

# ------------------------------------------------------------
# Save rebuilt feature arrays for reuse
# ------------------------------------------------------------

FEATURE_CACHE = os.path.join(
    DATA_DIR,
    "gat_aml_features_rebuilt.npz"
)

np.savez(
    FEATURE_CACHE,
    X_node=X_node,
    E_features=E_features
)

print()
print(
    f"Saved numerical feature cache: {FEATURE_CACHE}"
)

print()
print("=" * 75)
print("READY FOR CORRECT TRAIN / VALIDATION / TEST SPLIT")
print("=" * 75)

REBUILDING AML FEATURES
Input transactions : 10,000,000
Timestamp missing   : 0

Running AMLFeatureEngine.transform(...) ...
This will regenerate the edge + node features.

✅ Features generated in 78.74 sec
Transactions : 10,000,000
Nodes        : 1,213,224
Edge features: (10000000, 20)
Node features: (1213224, 13)

FEATURE REBUILD COMPLETE
Transactions       : 10,000,000
Node features      : (1213224, 13)
Edge features      : (10000000, 20)
Edge feature count : 20
Node feature count : 13

TIMESTAMP VERIFICATION
---------------------------------------------------------------------------
Missing timestamps : 0
Minimum timestamp  : 2022-09-01 00:00:00
Maximum timestamp  : 2022-09-18 11:18:00

GLOBAL CLASS DISTRIBUTION
---------------------------------------------------------------------------
Legitimate   :    5,000,000 (50.00%)
Laundering   :    5,000,000 (50.00%)

FEATURE VALIDITY CHECK
---------------------------------------------------------------------------
Edge NaNs : 0
Edge Infs 

In [26]:
# ============================================================
# CELL 20 — REBUILD GRAPH + CORRECT CHRONOLOGICAL SPLIT
# ============================================================

import numpy as np
import torch
import time

print("=" * 75)
print("REBUILDING GRAPH AND TRAIN / VAL / TEST SPLIT")
print("=" * 75)

# ------------------------------------------------------------
# 1. Build source / destination arrays
# ------------------------------------------------------------

print("\nPreparing graph edges...")

src_np = df_features["src_id"].to_numpy(
    dtype=np.int64,
    copy=False
)

dst_np = df_features["dst_id"].to_numpy(
    dtype=np.int64,
    copy=False
)

num_transactions = len(df_features)
num_nodes = int(
    max(src_np.max(), dst_np.max()) + 1
)

print(f"Transactions : {num_transactions:,}")
print(f"Nodes        : {num_nodes:,}")


# ------------------------------------------------------------
# 2. Verify node IDs
# ------------------------------------------------------------

assert src_np.min() >= 0
assert dst_np.min() >= 0

assert src_np.max() < num_nodes
assert dst_np.max() < num_nodes


# ------------------------------------------------------------
# 3. Verify chronological ordering
# ------------------------------------------------------------

timestamps = df_features["Timestamp"].to_numpy()

chronological = np.all(
    timestamps[:-1] <= timestamps[1:]
)

print()
print(
    "Chronologically sorted:",
    chronological
)

# The feature engine should have sorted the transactions.
# If not, explicitly sort everything by timestamp here.

if not chronological:

    print(
        "Sorting transactions chronologically..."
    )

    order = np.argsort(
        timestamps,
        kind="stable"
    )

    df_features = df_features.iloc[
        order
    ].reset_index(drop=True)

    src_np = src_np[order]
    dst_np = dst_np[order]
    E_features = E_features[order]

    timestamps = df_features[
        "Timestamp"
    ].to_numpy()

    print(
        "Re-sort complete."
    )

    assert np.all(
        timestamps[:-1] <= timestamps[1:]
    )


# ------------------------------------------------------------
# 4. Chronological 70 / 15 / 15 split
# ------------------------------------------------------------

TRAIN_END = int(
    num_transactions * 0.70
)

VAL_END = int(
    num_transactions * 0.85
)

train_idx = torch.arange(
    0,
    TRAIN_END,
    dtype=torch.long
)

val_idx = torch.arange(
    TRAIN_END,
    VAL_END,
    dtype=torch.long
)

test_idx = torch.arange(
    VAL_END,
    num_transactions,
    dtype=torch.long
)

print()
print("=" * 75)
print("SPLIT SIZES")
print("=" * 75)

print(
    f"Train      : {len(train_idx):,}"
)

print(
    f"Validation : {len(val_idx):,}"
)

print(
    f"Test       : {len(test_idx):,}"
)


# ------------------------------------------------------------
# 5. Class distribution for each split
# ------------------------------------------------------------

labels_np = df_features[
    "Is Laundering"
].to_numpy(
    dtype=np.int8,
    copy=False
)


def print_distribution(name, ids):

    ids_np = ids.numpy()

    y = labels_np[ids_np]

    legitimate = int(
        np.sum(y == 0)
    )

    laundering = int(
        np.sum(y == 1)
    )

    total = len(y)

    print()
    print(name)
    print("-" * 50)

    print(
        f"Total        : {total:,}"
    )

    print(
        f"Legitimate   : "
        f"{legitimate:,} "
        f"({legitimate / total:.2%})"
    )

    print(
        f"Laundering   : "
        f"{laundering:,} "
        f"({laundering / total:.2%})"
    )


print_distribution(
    "TRAIN",
    train_idx
)

print_distribution(
    "VALIDATION",
    val_idx
)

print_distribution(
    "TEST",
    test_idx
)


# ------------------------------------------------------------
# 6. Time range of each split
# ------------------------------------------------------------

print()
print("=" * 75)
print("TIME RANGES")
print("=" * 75)

for name, ids in [
    ("TRAIN", train_idx),
    ("VALIDATION", val_idx),
    ("TEST", test_idx)
]:

    ids_np = ids.numpy()

    ts = df_features.iloc[
        [ids_np[0], ids_np[-1]]
    ]["Timestamp"]

    print(
        f"{name:12s}: "
        f"{ts.iloc[0]}  -->  {ts.iloc[1]}"
    )


# ------------------------------------------------------------
# 7. Build original directed transaction edge index
# ------------------------------------------------------------

print()
print("Building directed transaction edge index...")

edge_index = torch.from_numpy(
    np.vstack([
        src_np,
        dst_np
    ])
).long()

print(
    "edge_index shape:",
    tuple(edge_index.shape)
)


# ------------------------------------------------------------
# 8. Build bidirectional message-passing graph
# ------------------------------------------------------------

print(
    "\nBuilding bidirectional message-passing edges..."
)

message_edge_index = torch.cat(
    [
        edge_index,
        edge_index.flip(0)
    ],
    dim=1
)

print(
    "message_edge_index shape:",
    tuple(message_edge_index.shape)
)

print(
    f"Original transaction edges : "
    f"{edge_index.shape[1]:,}"
)

print(
    f"Message-passing edges      : "
    f"{message_edge_index.shape[1]:,}"
)


# ------------------------------------------------------------
# 9. Train-only CSR adjacency
# ------------------------------------------------------------

print()
print("=" * 75)
print("BUILDING TRAIN-ONLY NEIGHBOR INDEX")
print("=" * 75)

train_ids_np = train_idx.numpy()

train_src = src_np[
    train_ids_np
]

train_dst = dst_np[
    train_ids_np
]

# Bidirectional adjacency for message passing
adj_nodes = np.concatenate(
    [
        train_src,
        train_dst
    ]
)

adj_neighbors = np.concatenate(
    [
        train_dst,
        train_src
    ]
)

adj_edge_pos = np.concatenate(
    [
        train_ids_np,
        train_ids_np
    ]
)

# ------------------------------------------------------------
# Sort by source node
# ------------------------------------------------------------

sort_order = np.argsort(
    adj_nodes,
    kind="stable"
)

adj_nodes = adj_nodes[
    sort_order
]

TRAIN_ADJ_NEIGHBORS = adj_neighbors[
    sort_order
]

TRAIN_ADJ_EDGE_POS = adj_edge_pos[
    sort_order
]

# ------------------------------------------------------------
# Build CSR pointers
# ------------------------------------------------------------

TRAIN_ADJ_PTR = np.zeros(
    num_nodes + 1,
    dtype=np.int64
)

np.add.at(
    TRAIN_ADJ_PTR,
    adj_nodes + 1,
    1
)

TRAIN_ADJ_PTR = np.cumsum(
    TRAIN_ADJ_PTR
)

print(
    f"Train adjacency references : "
    f"{len(TRAIN_ADJ_NEIGHBORS):,}"
)

print(
    f"CSR pointer shape           : "
    f"{TRAIN_ADJ_PTR.shape}"
)

print(
    f"Unique training source nodes: "
    f"{np.count_nonzero(np.diff(TRAIN_ADJ_PTR)):,}"
)


# ------------------------------------------------------------
# 10. Make sure sampler variables are synchronized
# ------------------------------------------------------------

TRAIN_EDGE_IDS = train_ids_np

print()
print("=" * 75)
print("GRAPH SETUP COMPLETE")
print("=" * 75)

print(
    "src_np                 :", src_np.shape
)

print(
    "dst_np                 :", dst_np.shape
)

print(
    "E_features             :", E_features.shape
)

print(
    "X_node                 :", X_node.shape
)

print(
    "TRAIN_EDGE_IDS         :", TRAIN_EDGE_IDS.shape
)

print(
    "TRAIN_ADJ_NEIGHBORS   :",
    TRAIN_ADJ_NEIGHBORS.shape
)

print(
    "TRAIN_ADJ_EDGE_POS    :",
    TRAIN_ADJ_EDGE_POS.shape
)

print(
    "TRAIN_ADJ_PTR         :",
    TRAIN_ADJ_PTR.shape
)

# ------------------------------------------------------------
# 11. Final assertions
# ------------------------------------------------------------

assert len(src_np) == num_transactions
assert len(dst_np) == num_transactions
assert len(E_features) == num_transactions

assert (
    len(train_idx)
    + len(val_idx)
    + len(test_idx)
    == num_transactions
)

assert (
    len(TRAIN_EDGE_IDS)
    == len(train_idx)
)

assert len(TRAIN_ADJ_NEIGHBORS) == (
    2 * len(train_idx)
)

print()
print("✅ All graph/split assertions passed.")

REBUILDING GRAPH AND TRAIN / VAL / TEST SPLIT

Preparing graph edges...
Transactions : 10,000,000
Nodes        : 1,213,224

Chronologically sorted: True

SPLIT SIZES
Train      : 7,000,000
Validation : 1,500,000
Test       : 1,500,000

TRAIN
--------------------------------------------------
Total        : 7,000,000
Legitimate   : 3,904,126 (55.77%)
Laundering   : 3,095,874 (44.23%)

VALIDATION
--------------------------------------------------
Total        : 1,500,000
Legitimate   : 818,893 (54.59%)
Laundering   : 681,107 (45.41%)

TEST
--------------------------------------------------
Total        : 1,500,000
Legitimate   : 276,981 (18.47%)
Laundering   : 1,223,019 (81.53%)

TIME RANGES
TRAIN       : 2022-09-01 00:00:00  -->  2022-09-08 11:11:00
VALIDATION  : 2022-09-08 11:11:00  -->  2022-09-09 21:15:00
TEST        : 2022-09-09 21:15:00  -->  2022-09-18 11:18:00

Building directed transaction edge index...
edge_index shape: (2, 10000000)

Building bidirectional message-passing edge

In [27]:
# ============================================================
# CELL 21 — FRESH GAT AML TRAINING
# ============================================================

import os
import gc
import time
import numpy as np
import torch
import torch.nn as nn
from tqdm.auto import tqdm

print("=" * 75)
print("FRESH GAT AML TRAINING")
print("=" * 75)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

SEED = 42

EPOCHS = 3
TRAIN_BATCH_SIZE = 8192

NEIGHBORS_1 = 8
NEIGHBORS_2 = 8

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

CHECKPOINT_PATH = os.path.join(
    DATA_DIR,
    "gat_aml_stage1.pt"
)

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.set_float32_matmul_precision("high")

print(f"Epochs          : {EPOCHS}")
print(f"Batch size      : {TRAIN_BATCH_SIZE:,}")
print(f"1-hop neighbors : {NEIGHBORS_1}")
print(f"2-hop neighbors : {NEIGHBORS_2}")
print(f"Learning rate   : {LEARNING_RATE}")
print(f"Weight decay    : {WEIGHT_DECAY}")
print(f"Device          : {DEVICE}")
print()


# ============================================================
# FRESH MODEL
# ============================================================

model = GATAMLModel(
    node_in_dim=X_node.shape[1],
    edge_in_dim=E_features.shape[1],
    hidden_dim=64,
    heads=4,
    dropout=0.20
).to(DEVICE)

# ------------------------------------------------------------
# Count parameters
# ------------------------------------------------------------

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    f"Trainable parameters : {trainable_params:,}"
)

# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# ------------------------------------------------------------
# Balanced dataset
# ------------------------------------------------------------

criterion = nn.BCEWithLogitsLoss()

# ------------------------------------------------------------
# Mixed precision
# ------------------------------------------------------------

use_amp = DEVICE.type == "cuda"

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp
)

# ------------------------------------------------------------
# Training IDs
# ------------------------------------------------------------

TRAIN_IDS = train_idx.cpu().numpy()

num_train = len(TRAIN_IDS)

num_batches = (
    num_train + TRAIN_BATCH_SIZE - 1
) // TRAIN_BATCH_SIZE

print(
    f"Training transactions : {num_train:,}"
)

print(
    f"Batches per epoch     : {num_batches:,}"
)

# ------------------------------------------------------------
# Check class balance
# ------------------------------------------------------------

train_labels = labels_np[TRAIN_IDS]

print()
print("Training label distribution:")
print(
    f"  Legitimate : "
    f"{np.sum(train_labels == 0):,}"
)

print(
    f"  Laundering : "
    f"{np.sum(train_labels == 1):,}"
)

print()


# ============================================================
# GPU BATCH PREPARATION
# ============================================================

def prepare_gpu_batch(target_ids, seed):

    batch = build_training_batch_fast(
        target_ids,
        seed=seed
    )

    x = batch["x"].to(
        DEVICE,
        non_blocking=True
    )

    edge_index = batch["edge_index"].to(
        DEVICE,
        non_blocking=True
    )

    edge_attr = batch["edge_attr"].to(
        DEVICE,
        non_blocking=True
    )

    target_edge_index = batch[
        "target_edge_index"
    ].to(
        DEVICE,
        non_blocking=True
    )

    target_edge_attr = batch[
        "target_edge_attr"
    ].to(
        DEVICE,
        non_blocking=True
    )

    y = batch["y"].to(
        DEVICE,
        non_blocking=True
    ).float()

    return (
        x,
        edge_index,
        edge_attr,
        target_edge_index,
        target_edge_attr,
        y
    )


# ============================================================
# TRAINING LOOP
# ============================================================

history = []

global_start = time.time()

for epoch in range(1, EPOCHS + 1):

    epoch_start = time.time()

    model.train()

    # Shuffle training transactions
    rng = np.random.default_rng(
        SEED + epoch
    )

    shuffled_ids = TRAIN_IDS.copy()

    rng.shuffle(
        shuffled_ids
    )

    running_loss = 0.0

    print()
    print("=" * 75)
    print(f"EPOCH {epoch}/{EPOCHS}")
    print("=" * 75)

    progress = tqdm(
        range(
            0,
            num_train,
            TRAIN_BATCH_SIZE
        ),
        total=num_batches,
        desc=f"Epoch {epoch}"
    )

    for batch_no, start in enumerate(
        progress,
        start=1
    ):

        target_ids = shuffled_ids[
            start:start + TRAIN_BATCH_SIZE
        ]

        # ----------------------------------------------------
        # Build local sampled graph
        # ----------------------------------------------------

        (
            x,
            edge_index,
            edge_attr,
            target_edge_index,
            target_edge_attr,
            y
        ) = prepare_gpu_batch(
            target_ids,
            seed=SEED + epoch * 100000 + batch_no
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        # ----------------------------------------------------
        # Forward + loss
        # ----------------------------------------------------

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=use_amp
        ):

            logits = model(
                x,
                edge_index,
                edge_attr,
                target_edge_index,
                target_edge_attr
            )

            loss = criterion(
                logits.view(-1),
                y.view(-1)
            )

        # ----------------------------------------------------
        # Backprop
        # ----------------------------------------------------

        scaler.scale(
            loss
        ).backward()

        scaler.step(
            optimizer
        )

        scaler.update()

        running_loss += (
            loss.item()
            * len(target_ids)
        )

        avg_loss = (
            running_loss
            / (start + len(target_ids))
        )

        progress.set_postfix(
            loss=f"{avg_loss:.6f}"
        )

        # ----------------------------------------------------
        # Cleanup references
        # ----------------------------------------------------

        del (
            x,
            edge_index,
            edge_attr,
            target_edge_index,
            target_edge_attr,
            y,
            logits,
            loss
        )

    epoch_loss = (
        running_loss
        / num_train
    )

    epoch_time = (
        time.time()
        - epoch_start
    )

    history.append(
        {
            "epoch": epoch,
            "loss": epoch_loss,
            "time_sec": epoch_time
        }
    )

    print()
    print(
        f"Epoch {epoch} complete"
    )

    print(
        f"Average loss : {epoch_loss:.8f}"
    )

    print(
        f"Time         : {epoch_time:.2f} sec"
    )

    # --------------------------------------------------------
    # Save checkpoint after every epoch
    # --------------------------------------------------------

    checkpoint = {
        "model_state": model.state_dict(),

        "model_config": {
            "node_in_dim": int(X_node.shape[1]),
            "edge_in_dim": int(E_features.shape[1]),
            "hidden_dim": 64,
            "heads": 4,
            "dropout": 0.20
        },

        "edge_feature_names": edge_feature_names,
        "node_feature_names": node_feature_names,

        "num_nodes": int(num_nodes),
        "num_transactions": int(num_transactions),

        "train_transactions": int(num_train),

        "epoch": epoch,
        "batch_size": TRAIN_BATCH_SIZE,

        "neighbors": {
            "hop1": NEIGHBORS_1,
            "hop2": NEIGHBORS_2
        },

        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,

        "seed": SEED,

        "history": history
    }

    torch.save(
        checkpoint,
        CHECKPOINT_PATH
    )

    print(
        f"Checkpoint saved: {CHECKPOINT_PATH}"
    )

    # --------------------------------------------------------
    # GPU memory
    # --------------------------------------------------------

    if DEVICE.type == "cuda":

        allocated = (
            torch.cuda.memory_allocated(
                DEVICE
            )
            / 1024**3
        )

        reserved = (
            torch.cuda.memory_reserved(
                DEVICE
            )
            / 1024**3
        )

        print(
            f"GPU memory allocated : "
            f"{allocated:.3f} GB"
        )

        print(
            f"GPU memory reserved  : "
            f"{reserved:.3f} GB"
        )

    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


# ============================================================
# TRAINING COMPLETE
# ============================================================

total_time = (
    time.time()
    - global_start
)

print()
print("=" * 75)
print("TRAINING COMPLETE")
print("=" * 75)

for item in history:

    print(
        f"Epoch {item['epoch']}: "
        f"loss={item['loss']:.8f}, "
        f"time={item['time_sec']:.2f}s"
    )

print()
print(
    f"Total training time : "
    f"{total_time:.2f} sec"
)

print(
    f"Final checkpoint    : "
    f"{CHECKPOINT_PATH}"
)

print()
print("✅ Fresh model trained using repaired timestamps/features.")

FRESH GAT AML TRAINING
Epochs          : 3
Batch size      : 8,192
1-hop neighbors : 8
2-hop neighbors : 8
Learning rate   : 0.001
Weight decay    : 0.0001
Device          : cuda

Trainable parameters : 32,385
Training transactions : 7,000,000
Batches per epoch     : 855

Training label distribution:
  Legitimate : 3,904,126
  Laundering : 3,095,874


EPOCH 1/3


Epoch 1:   0%|          | 0/855 [00:00<?, ?it/s]


Epoch 1 complete
Average loss : 0.01066187
Time         : 612.90 sec
Checkpoint saved: /home/llyods-aids5/AML/gat_aml_stage1.pt
GPU memory allocated : 0.036 GB
GPU memory reserved  : 5.275 GB

EPOCH 2/3


Epoch 2:   0%|          | 0/855 [00:00<?, ?it/s]


Epoch 2 complete
Average loss : 0.00004415
Time         : 610.93 sec
Checkpoint saved: /home/llyods-aids5/AML/gat_aml_stage1.pt
GPU memory allocated : 0.036 GB
GPU memory reserved  : 5.049 GB

EPOCH 3/3


Epoch 3:   0%|          | 0/855 [00:00<?, ?it/s]


Epoch 3 complete
Average loss : 0.00001529
Time         : 618.30 sec
Checkpoint saved: /home/llyods-aids5/AML/gat_aml_stage1.pt
GPU memory allocated : 0.036 GB
GPU memory reserved  : 5.041 GB

TRAINING COMPLETE
Epoch 1: loss=0.01066187, time=612.90s
Epoch 2: loss=0.00004415, time=610.93s
Epoch 3: loss=0.00001529, time=618.30s

Total training time : 1842.76 sec
Final checkpoint    : /home/llyods-aids5/AML/gat_aml_stage1.pt

✅ Fresh model trained using repaired timestamps/features.


In [28]:
# ============================================================
# CELL 22 — FINAL VALIDATION + TEST EVALUATION
# ============================================================

import numpy as np
import torch
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)
from tqdm.auto import tqdm

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

EVAL_BATCH_SIZE = 8192
EVAL_THRESHOLD = 0.50

VAL_IDS = val_idx.cpu().numpy()
TEST_IDS = test_idx.cpu().numpy()

labels_np = df_features[
    "Is Laundering"
].to_numpy(
    dtype=np.int8,
    copy=False
)

print("=" * 75)
print("FINAL GAT AML EVALUATION")
print("=" * 75)

print(f"Validation transactions : {len(VAL_IDS):,}")
print(f"Test transactions       : {len(TEST_IDS):,}")
print(f"Threshold               : {EVAL_THRESHOLD}")
print(f"Device                  : {DEVICE}")
print()


# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate_gat(edge_ids, split_name):

    model.eval()

    all_probs = []

    total_start = time.time()

    print("=" * 75)
    print(f"EVALUATING {split_name}")
    print("=" * 75)

    with torch.inference_mode():

        for batch_no, start in enumerate(
            tqdm(
                range(
                    0,
                    len(edge_ids),
                    EVAL_BATCH_SIZE
                ),
                desc=split_name
            ),
            start=1
        ):

            target_ids = edge_ids[
                start:start + EVAL_BATCH_SIZE
            ]

            # ------------------------------------------------
            # Build local graph
            # ------------------------------------------------

            batch = build_training_batch_fast(
                target_ids,
                seed=SEED + batch_no
            )

            x = batch["x"].to(
                DEVICE,
                non_blocking=True
            )

            edge_index = batch["edge_index"].to(
                DEVICE,
                non_blocking=True
            )

            edge_attr = batch["edge_attr"].to(
                DEVICE,
                non_blocking=True
            )

            target_edge_index = batch[
                "target_edge_index"
            ].to(
                DEVICE,
                non_blocking=True
            )

            target_edge_attr = batch[
                "target_edge_attr"
            ].to(
                DEVICE,
                non_blocking=True
            )

            # ------------------------------------------------
            # Prediction
            # ------------------------------------------------

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(DEVICE.type == "cuda")
            ):

                logits = model(
                    x,
                    edge_index,
                    edge_attr,
                    target_edge_index,
                    target_edge_attr
                )

            probs = torch.sigmoid(
                logits
            ).float().cpu().numpy().reshape(-1)

            all_probs.append(
                probs
            )

            del (
                x,
                edge_index,
                edge_attr,
                target_edge_index,
                target_edge_attr,
                logits,
                probs
            )

    # --------------------------------------------------------
    # Collect predictions
    # --------------------------------------------------------

    y_prob = np.concatenate(
        all_probs
    )

    # Labels DIRECTLY from target transaction IDs
    y_true = labels_np[
        edge_ids
    ].astype(
        np.int64
    )

    assert len(y_true) == len(y_prob)

    # --------------------------------------------------------
    # Threshold predictions
    # --------------------------------------------------------

    y_pred = (
        y_prob >= EVAL_THRESHOLD
    ).astype(
        np.int64
    )

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    unique_classes = np.unique(
        y_true
    )

    if len(unique_classes) == 2:

        roc_auc = roc_auc_score(
            y_true,
            y_prob
        )

        pr_auc = average_precision_score(
            y_true,
            y_prob
        )

    else:

        roc_auc = np.nan
        pr_auc = np.nan

    # --------------------------------------------------------
    # Confusion matrix
    # --------------------------------------------------------

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    # --------------------------------------------------------
    # Print results
    # --------------------------------------------------------

    elapsed = (
        time.time()
        - total_start
    )

    print()
    print("=" * 75)
    print(f"{split_name} RESULTS")
    print("=" * 75)

    print(
        f"Precision : {precision:.6f}"
    )

    print(
        f"Recall    : {recall:.6f}"
    )

    print(
        f"F1 Score  : {f1:.6f}"
    )

    print(
        f"ROC-AUC   : {roc_auc:.6f}"
    )

    print(
        f"PR-AUC    : {pr_auc:.6f}"
    )

    print()
    print("CONFUSION MATRIX")
    print("-" * 75)

    print(
        f"True Negatives  : {tn:,}"
    )

    print(
        f"False Positives : {fp:,}"
    )

    print(
        f"False Negatives : {fn:,}"
    )

    print(
        f"True Positives  : {tp:,}"
    )

    print()
    print("CLASS-WISE METRICS")
    print("-" * 75)

    print(
        classification_report(
            y_true,
            y_pred,
            labels=[0, 1],
            target_names=[
                "Legitimate",
                "Laundering"
            ],
            digits=6,
            zero_division=0
        )
    )

    # --------------------------------------------------------
    # AML workload metrics
    # --------------------------------------------------------

    total = len(y_true)
    alerts = tp + fp
    actual_aml = tp + fn

    print("AML ALERT STATISTICS")
    print("-" * 75)

    print(
        f"Total transactions : {total:,}"
    )

    print(
        f"Actual laundering  : {actual_aml:,}"
    )

    print(
        f"Alerts generated   : {alerts:,}"
    )

    print(
        f"Alert rate         : {alerts / total:.4%}"
    )

    if tp > 0:

        print(
            f"Transactions reviewed per TP : "
            f"{alerts / tp:.2f}"
        )

    print(
        f"Evaluation time    : {elapsed:.2f} sec"
    )

    # --------------------------------------------------------
    # Probability distribution
    # --------------------------------------------------------

    print()
    print("PREDICTION DISTRIBUTION")
    print("-" * 75)

    print(
        f"Min probability    : {y_prob.min():.6f}"
    )

    print(
        f"Mean probability   : {y_prob.mean():.6f}"
    )

    print(
        f"Median probability : {np.median(y_prob):.6f}"
    )

    print(
        f"Max probability    : {y_prob.max():.6f}"
    )

    return {
        "y_true": y_true,
        "y_prob": y_prob,
        "y_pred": y_pred,
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "confusion_matrix": cm,
    }


# ============================================================
# VALIDATION
# ============================================================

val_results = evaluate_gat(
    VAL_IDS,
    "VALIDATION"
)


# ============================================================
# TEST
# ============================================================

test_results = evaluate_gat(
    TEST_IDS,
    "TEST"
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 75)
print("FINAL MODEL SUMMARY")
print("=" * 75)

print()
print("VALIDATION")
print(
    f"Precision : {val_results['precision']:.6f}"
)
print(
    f"Recall    : {val_results['recall']:.6f}"
)
print(
    f"F1        : {val_results['f1']:.6f}"
)
print(
    f"ROC-AUC   : {val_results['roc_auc']:.6f}"
)
print(
    f"PR-AUC    : {val_results['pr_auc']:.6f}"
)

print()
print("TEST")
print(
    f"Precision : {test_results['precision']:.6f}"
)
print(
    f"Recall    : {test_results['recall']:.6f}"
)
print(
    f"F1        : {test_results['f1']:.6f}"
)
print(
    f"ROC-AUC   : {test_results['roc_auc']:.6f}"
)
print(
    f"PR-AUC    : {test_results['pr_auc']:.6f}"
)

# ------------------------------------------------------------
# Save compact evaluation result
# ------------------------------------------------------------

import json

evaluation_summary = {
    "threshold": EVAL_THRESHOLD,

    "validation": {
        "precision": val_results["precision"],
        "recall": val_results["recall"],
        "f1": val_results["f1"],
        "roc_auc": val_results["roc_auc"],
        "pr_auc": val_results["pr_auc"],
        "confusion_matrix":
            val_results["confusion_matrix"].tolist()
    },

    "test": {
        "precision": test_results["precision"],
        "recall": test_results["recall"],
        "f1": test_results["f1"],
        "roc_auc": test_results["roc_auc"],
        "pr_auc": test_results["pr_auc"],
        "confusion_matrix":
            test_results["confusion_matrix"].tolist()
    }
}

with open(
    os.path.join(
        DATA_DIR,
        "gat_aml_final_evaluation.json"
    ),
    "w"
) as f:

    json.dump(
        evaluation_summary,
        f,
        indent=2,
        allow_nan=True
    )

print()
print(
    "✅ Evaluation saved to:"
)

print(
    os.path.join(
        DATA_DIR,
        "gat_aml_final_evaluation.json"
    )
)

FINAL GAT AML EVALUATION
Validation transactions : 1,500,000
Test transactions       : 1,500,000
Threshold               : 0.5
Device                  : cuda

EVALUATING VALIDATION


VALIDATION:   0%|          | 0/184 [00:00<?, ?it/s]


VALIDATION RESULTS
Precision : 1.000000
Recall    : 1.000000
F1 Score  : 1.000000
ROC-AUC   : 1.000000
PR-AUC    : 1.000000

CONFUSION MATRIX
---------------------------------------------------------------------------
True Negatives  : 818,893
False Positives : 0
False Negatives : 0
True Positives  : 681,107

CLASS-WISE METRICS
---------------------------------------------------------------------------
              precision    recall  f1-score   support

  Legitimate   1.000000  1.000000  1.000000    818893
  Laundering   1.000000  1.000000  1.000000    681107

    accuracy                       1.000000   1500000
   macro avg   1.000000  1.000000  1.000000   1500000
weighted avg   1.000000  1.000000  1.000000   1500000

AML ALERT STATISTICS
---------------------------------------------------------------------------
Total transactions : 1,500,000
Actual laundering  : 681,107
Alerts generated   : 681,107
Alert rate         : 45.4071%
Transactions reviewed per TP : 1.00
Evaluation tim

TEST:   0%|          | 0/184 [00:00<?, ?it/s]


TEST RESULTS
Precision : 1.000000
Recall    : 1.000000
F1 Score  : 1.000000
ROC-AUC   : 1.000000
PR-AUC    : 1.000000

CONFUSION MATRIX
---------------------------------------------------------------------------
True Negatives  : 276,981
False Positives : 0
False Negatives : 0
True Positives  : 1,223,019

CLASS-WISE METRICS
---------------------------------------------------------------------------
              precision    recall  f1-score   support

  Legitimate   1.000000  1.000000  1.000000    276981
  Laundering   1.000000  1.000000  1.000000   1223019

    accuracy                       1.000000   1500000
   macro avg   1.000000  1.000000  1.000000   1500000
weighted avg   1.000000  1.000000  1.000000   1500000

AML ALERT STATISTICS
---------------------------------------------------------------------------
Total transactions : 1,500,000
Actual laundering  : 1,223,019
Alerts generated   : 1,223,019
Alert rate         : 81.5346%
Transactions reviewed per TP : 1.00
Evaluation tim

In [29]:
# ============================================================
# CELL 23 — INVESTIGATE WHY PERFORMANCE IS PERFECT
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

print("=" * 80)
print("WHY IS THE GAT GETTING 100%?")
print("=" * 80)

# ------------------------------------------------------------
# 1. Use the actual TEST labels and predictions
# ------------------------------------------------------------

y = test_results["y_true"]
p = test_results["y_prob"]

print("\nTEST LABEL DISTRIBUTION")
print("-" * 80)

print(
    f"Legitimate : {np.sum(y == 0):,}"
)

print(
    f"Laundering : {np.sum(y == 1):,}"
)

# ------------------------------------------------------------
# 2. Prediction separation
# ------------------------------------------------------------

print("\nPREDICTION SEPARATION")
print("-" * 80)

for cls, name in [
    (0, "LEGITIMATE"),
    (1, "LAUNDERING")
]:

    values = p[y == cls]

    print(f"\n{name}")

    print(f"Count  : {len(values):,}")
    print(f"Min    : {values.min():.10f}")
    print(f"1%     : {np.percentile(values, 1):.10f}")
    print(f"10%    : {np.percentile(values, 10):.10f}")
    print(f"25%    : {np.percentile(values, 25):.10f}")
    print(f"50%    : {np.percentile(values, 50):.10f}")
    print(f"75%    : {np.percentile(values, 75):.10f}")
    print(f"90%    : {np.percentile(values, 90):.10f}")
    print(f"99%    : {np.percentile(values, 99):.10f}")
    print(f"Max    : {values.max():.10f}")

# ------------------------------------------------------------
# 3. Check every edge feature individually
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("INDIVIDUAL EDGE FEATURE SEPARABILITY")
print("=" * 80)

test_ids = TEST_IDS

edge_test = E_features[test_ids]

feature_results = []

for i, feature_name in enumerate(edge_feature_names):

    values = edge_test[:, i]

    try:
        auc = roc_auc_score(
            y,
            values
        )

        auc = max(
            auc,
            1.0 - auc
        )

    except Exception:
        auc = np.nan

    legit = values[y == 0]
    aml = values[y == 1]

    feature_results.append({
        "feature": feature_name,
        "auc_abs_from_0.5": auc,
        "legit_mean": float(np.mean(legit)),
        "aml_mean": float(np.mean(aml)),
        "legit_std": float(np.std(legit)),
        "aml_std": float(np.std(aml))
    })

feature_df = pd.DataFrame(
    feature_results
).sort_values(
    "auc_abs_from_0.5",
    ascending=False
)

print(
    feature_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# ------------------------------------------------------------
# 4. Check node-feature separation
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SOURCE NODE FEATURE SEPARABILITY")
print("=" * 80)

src_test = src_np[test_ids]

source_features = X_node[src_test]

node_results = []

for i, feature_name in enumerate(node_feature_names):

    values = source_features[:, i]

    try:

        auc = roc_auc_score(
            y,
            values
        )

        auc = max(
            auc,
            1.0 - auc
        )

    except Exception:

        auc = np.nan

    legit = values[y == 0]
    aml = values[y == 1]

    node_results.append({
        "feature": feature_name,
        "auc_abs_from_0.5": auc,
        "legit_mean": float(np.mean(legit)),
        "aml_mean": float(np.mean(aml)),
        "legit_std": float(np.std(legit)),
        "aml_std": float(np.std(aml))
    })

node_df = pd.DataFrame(
    node_results
).sort_values(
    "auc_abs_from_0.5",
    ascending=False
)

print(
    node_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# ------------------------------------------------------------
# 5. Account identity overlap
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("ACCOUNT ID OVERLAP BETWEEN CLASSES")
print("=" * 80)

src_test_labels = labels_np[test_ids]

legit_sources = set(
    src_np[
        test_ids[src_test_labels == 0]
    ]
)

aml_sources = set(
    src_np[
        test_ids[src_test_labels == 1]
    ]
)

source_overlap = (
    legit_sources &
    aml_sources
)

print(
    f"Unique legitimate senders : "
    f"{len(legit_sources):,}"
)

print(
    f"Unique AML senders        : "
    f"{len(aml_sources):,}"
)

print(
    f"Shared senders            : "
    f"{len(source_overlap):,}"
)

if len(legit_sources) > 0:

    print(
        f"Sender overlap rate      : "
        f"{len(source_overlap) / len(legit_sources):.4%}"
    )

# ------------------------------------------------------------
# 6. Destination identity overlap
# ------------------------------------------------------------

legit_destinations = set(
    dst_np[
        test_ids[src_test_labels == 0]
    ]
)

aml_destinations = set(
    dst_np[
        test_ids[src_test_labels == 1]
    ]
)

destination_overlap = (
    legit_destinations &
    aml_destinations
)

print("\nDESTINATION OVERLAP")

print(
    f"Unique legitimate destinations : "
    f"{len(legit_destinations):,}"
)

print(
    f"Unique AML destinations        : "
    f"{len(aml_destinations):,}"
)

print(
    f"Shared destinations             : "
    f"{len(destination_overlap):,}"
)

# ------------------------------------------------------------
# 7. Source/destination ranges
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("NODE ID RANGES BY CLASS")
print("=" * 80)

for cls, name in [
    (0, "LEGITIMATE"),
    (1, "LAUNDERING")
]:

    ids = test_ids[
        src_test_labels == cls
    ]

    s = src_np[ids]
    d = dst_np[ids]

    print(f"\n{name}")

    print(
        f"Source min/max : "
        f"{s.min():,} / {s.max():,}"
    )

    print(
        f"Dest min/max   : "
        f"{d.min():,} / {d.max():,}"
    )

# ------------------------------------------------------------
# 8. Check whether target labels are accidentally aligned
#    with a feature or ID ordering
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DIRECT LABEL / FEATURE CORRELATION CHECK")
print("=" * 80)

for i, feature_name in enumerate(edge_feature_names):

    values = E_features[test_ids, i]

    if np.std(values) == 0:
        corr = 0.0
    else:
        corr = np.corrcoef(
            values,
            y
        )[0, 1]

    if abs(corr) > 0.95:

        print(
            f"⚠️ VERY HIGH correlation: "
            f"{feature_name:30s} "
            f"{corr:.6f}"
        )

print("\nDiagnostic complete.")

WHY IS THE GAT GETTING 100%?

TEST LABEL DISTRIBUTION
--------------------------------------------------------------------------------
Legitimate : 276,981
Laundering : 1,223,019

PREDICTION SEPARATION
--------------------------------------------------------------------------------

LEGITIMATE
Count  : 276,981
Min    : 0.0000005364
1%     : 0.0000005364
10%    : 0.0000005364
25%    : 0.0000005960
50%    : 0.0000005960
75%    : 0.0000005960
90%    : 0.0000006557
99%    : 0.0000007749
Max    : 0.0000351071

LAUNDERING
Count  : 1,223,019
Min    : 1.0000000000
1%     : 1.0000000000
10%    : 1.0000000000
25%    : 1.0000000000
50%    : 1.0000000000
75%    : 1.0000000000
90%    : 1.0000000000
99%    : 1.0000000000
Max    : 1.0000000000

INDIVIDUAL EDGE FEATURE SEPARABILITY
               feature  auc_abs_from_0.5  legit_mean  aml_mean  legit_std  aml_std
    pair_recency_hours          0.965347   -1.083345  0.815492   0.518220 0.015259
first_pair_transaction          0.965325   -1.084592  0.8

In [30]:
# ============================================================
# CELL 24 — SIMPLE FEATURE-ONLY BASELINES
# ============================================================

import numpy as np
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("=" * 80)
print("FEATURE-ONLY BASELINE DIAGNOSTIC")
print("=" * 80)

# ------------------------------------------------------------
# Test data
# ------------------------------------------------------------

test_ids = TEST_IDS

y_test = labels_np[test_ids]

edge_test = E_features[test_ids]
source_test = src_np[test_ids]
node_test = X_node[source_test]

# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------

def evaluate_single_feature(
    values,
    name
):

    auc = roc_auc_score(
        y_test,
        values
    )

    # AUC is direction-independent for this diagnostic
    auc_abs = max(
        auc,
        1.0 - auc
    )

    # Try both directions and select the direction
    # corresponding to larger AUC
    if auc < 0.5:
        scores = -values
    else:
        scores = values

    precision_recall_auc = (
        average_precision_score(
            y_test,
            scores
        )
        if np.unique(y_test).size == 2
        else np.nan
    )

    # Find threshold at median score simply for demonstration
    threshold = np.median(scores)

    pred = (
        scores >= threshold
    ).astype(np.int8)

    precision = precision_score(
        y_test,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        pred,
        zero_division=0
    )

    cm = confusion_matrix(
        y_test,
        pred,
        labels=[0, 1]
    )

    print()
    print("-" * 80)
    print(name)
    print("-" * 80)

    print(
        f"AUC             : {auc_abs:.6f}"
    )

    print(
        f"PR-AUC          : {precision_recall_auc:.6f}"
    )

    print(
        f"Median threshold: {threshold:.6f}"
    )

    print(
        f"Precision       : {precision:.6f}"
    )

    print(
        f"Recall          : {recall:.6f}"
    )

    print(
        f"F1              : {f1:.6f}"
    )

    print(
        "Confusion matrix:"
    )

    print(cm)

    return {
        "auc": auc_abs,
        "pr_auc": precision_recall_auc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


# ============================================================
# 1. TOP EDGE FEATURES
# ============================================================

print("\nTOP EDGE FEATURE BASELINES")

edge_candidates = [
    "pair_recency_hours",
    "first_pair_transaction",
    "amount_difference",
    "hour_cos",
    "sender_amount_zscore",
    "amount_paid",
    "amount_received"
]

for feature_name in edge_candidates:

    if feature_name in edge_feature_names:

        idx = edge_feature_names.index(
            feature_name
        )

        evaluate_single_feature(
            edge_test[:, idx],
            feature_name
        )


# ============================================================
# 2. TOP NODE FEATURES
# ============================================================

print("\n" + "=" * 80)
print("TOP NODE FEATURE BASELINES")
print("=" * 80)

node_candidates = [
    "fanout_ratio",
    "in_count",
    "unique_senders",
    "pass_through_ratio",
    "in_mean",
    "in_total",
    "out_mean",
    "unique_receivers",
    "out_max",
    "net_flow",
    "out_total"
]

for feature_name in node_candidates:

    if feature_name in node_feature_names:

        idx = node_feature_names.index(
            feature_name
        )

        evaluate_single_feature(
            node_test[:, idx],
            feature_name
        )


# ============================================================
# 3. DIRECT ACCOUNT-ID SEPARATION CHECK
# ============================================================

print()
print("=" * 80)
print("ACCOUNT ID / CLASS RELATIONSHIP")
print("=" * 80)

for cls, name in [
    (0, "LEGITIMATE"),
    (1, "LAUNDERING")
]:

    ids = source_test[
        y_test == cls
    ]

    print(
        f"{name:12s} source ID mean : "
        f"{ids.mean():,.2f}"
    )

    print(
        f"{name:12s} source ID min  : "
        f"{ids.min():,}"
    )

    print(
        f"{name:12s} source ID max  : "
        f"{ids.max():,}"
    )

# ------------------------------------------------------------
# Check whether any source IDs are shared
# ------------------------------------------------------------

legit_source_ids = set(
    source_test[
        y_test == 0
    ]
)

aml_source_ids = set(
    source_test[
        y_test == 1
    ]
)

shared = (
    legit_source_ids &
    aml_source_ids
)

print()
print(
    f"Shared source accounts : {len(shared):,}"
)

# ============================================================
# CONCLUSION
# ============================================================

print()
print("=" * 80)
print("DIAGNOSTIC CONCLUSION")
print("=" * 80)

print(
    "If fanout_ratio / account structure alone achieves "
    "near-perfect AUC, the dataset is intrinsically easy."
)

print(
    "That means the GAT's 1.0000 score cannot be interpreted "
    "as evidence of production-level AML detection."
)

print(
    "The next model iteration should use overlapping accounts, "
    "causal temporal features, and mixed legitimate + suspicious "
    "behavior within the same graph."
)

FEATURE-ONLY BASELINE DIAGNOSTIC

TOP EDGE FEATURE BASELINES

--------------------------------------------------------------------------------
pair_recency_hours
--------------------------------------------------------------------------------
AUC             : 0.965347
PR-AUC          : 0.984548
Median threshold: 0.815609
Precision       : 0.984548
Recall          : 0.999943
F1              : 0.992186
Confusion matrix:
[[ 257788   19193]
 [     70 1222949]]

--------------------------------------------------------------------------------
first_pair_transaction
--------------------------------------------------------------------------------
AUC             : 0.965325
PR-AUC          : 0.984539
Median threshold: 0.815616
Precision       : 0.984548
Recall          : 0.999943
F1              : 0.992186
Confusion matrix:
[[ 257788   19193]
 [     70 1222949]]

--------------------------------------------------------------------------------
amount_difference
---------------------------------

In [31]:
# ============================================================
# CELL 25 — FULL DATASET ACCOUNT / LABEL STRUCTURE CHECK
# ============================================================

import numpy as np
import pandas as pd

print("=" * 80)
print("FULL DATASET ACCOUNT / LABEL STRUCTURE")
print("=" * 80)

# ------------------------------------------------------------
# Transaction-level arrays
# ------------------------------------------------------------

src_all = src_np
dst_all = dst_np
y_all = labels_np

# ------------------------------------------------------------
# 1. SOURCE ACCOUNT OVERLAP
# ------------------------------------------------------------

legit_src = set(
    src_all[y_all == 0]
)

aml_src = set(
    src_all[y_all == 1]
)

shared_src = (
    legit_src &
    aml_src
)

print("\nSOURCE ACCOUNTS")
print("-" * 80)

print(
    f"Unique legitimate senders : {len(legit_src):,}"
)

print(
    f"Unique laundering senders : {len(aml_src):,}"
)

print(
    f"Shared senders            : {len(shared_src):,}"
)

# ------------------------------------------------------------
# 2. DESTINATION ACCOUNT OVERLAP
# ------------------------------------------------------------

legit_dst = set(
    dst_all[y_all == 0]
)

aml_dst = set(
    dst_all[y_all == 1]
)

shared_dst = (
    legit_dst &
    aml_dst
)

print("\nDESTINATION ACCOUNTS")
print("-" * 80)

print(
    f"Unique legitimate destinations : "
    f"{len(legit_dst):,}"
)

print(
    f"Unique laundering destinations : "
    f"{len(aml_dst):,}"
)

print(
    f"Shared destinations             : "
    f"{len(shared_dst):,}"
)


# ------------------------------------------------------------
# 3. ALL NODE OVERLAP
# ------------------------------------------------------------

legit_nodes = (
    set(src_all[y_all == 0]) |
    set(dst_all[y_all == 0])
)

aml_nodes = (
    set(src_all[y_all == 1]) |
    set(dst_all[y_all == 1])
)

shared_nodes = (
    legit_nodes &
    aml_nodes
)

print("\nALL ACCOUNT / NODE OVERLAP")
print("-" * 80)

print(
    f"Legitimate nodes : {len(legit_nodes):,}"
)

print(
    f"AML nodes        : {len(aml_nodes):,}"
)

print(
    f"Shared nodes     : {len(shared_nodes):,}"
)


# ------------------------------------------------------------
# 4. ACCOUNT-LEVEL LABEL PURITY
# ------------------------------------------------------------

# For every source node, count legitimate and AML transactions
# and determine whether it is:
#   legitimate-only
#   AML-only
#   mixed

print("\n" + "=" * 80)
print("SOURCE ACCOUNT LABEL PURITY")
print("=" * 80)

src_df = pd.DataFrame({
    "src": src_all,
    "label": y_all
})

src_counts = (
    src_df
    .groupby(["src", "label"])
    .size()
    .unstack(fill_value=0)
)

# Ensure both columns exist
if 0 not in src_counts.columns:
    src_counts[0] = 0

if 1 not in src_counts.columns:
    src_counts[1] = 0

src_counts = src_counts[[0, 1]]

src_counts["total"] = (
    src_counts[0] +
    src_counts[1]
)

src_counts["purity"] = (
    src_counts[[0, 1]].max(axis=1)
    /
    src_counts["total"]
)

legit_only = np.sum(
    (src_counts[0] > 0) &
    (src_counts[1] == 0)
)

aml_only = np.sum(
    (src_counts[1] > 0) &
    (src_counts[0] == 0)
)

mixed = np.sum(
    (src_counts[0] > 0) &
    (src_counts[1] > 0)
)

print(
    f"Legitimate-only senders : {legit_only:,}"
)

print(
    f"AML-only senders        : {aml_only:,}"
)

print(
    f"Mixed-behavior senders  : {mixed:,}"
)


# ------------------------------------------------------------
# 5. FANOUT RATIO BY CLASS
# ------------------------------------------------------------

fanout_idx = node_feature_names.index(
    "fanout_ratio"
)

fanout_source = X_node[
    src_all,
    fanout_idx
]

print("\n" + "=" * 80)
print("FANOUT RATIO DISTRIBUTION")
print("=" * 80)

for cls, name in [
    (0, "LEGITIMATE"),
    (1, "LAUNDERING")
]:

    values = fanout_source[
        y_all == cls
    ]

    print(f"\n{name}")

    print(
        f"Min    : {np.min(values):.6f}"
    )

    print(
        f"Median : {np.median(values):.6f}"
    )

    print(
        f"Mean   : {np.mean(values):.6f}"
    )

    print(
        f"90%    : {np.percentile(values, 90):.6f}"
    )

    print(
        f"99%    : {np.percentile(values, 99):.6f}"
    )

    print(
        f"Max    : {np.max(values):.6f}"
    )


# ------------------------------------------------------------
# 6. FANOUT THRESHOLD SEPARATION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FANOUT THRESHOLD ANALYSIS")
print("=" * 80)

thresholds = [
    1.0,
    1.5,
    2.0,
    2.5,
    3.0,
    4.0,
    5.0
]

for threshold in thresholds:

    pred = (
        fanout_source >= threshold
    )

    tp = np.sum(
        pred & (y_all == 1)
    )

    fp = np.sum(
        pred & (y_all == 0)
    )

    fn = np.sum(
        (~pred) & (y_all == 1)
    )

    precision = (
        tp / (tp + fp)
        if tp + fp > 0
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn > 0
        else 0.0
    )

    print(
        f"Threshold={threshold:>4.1f} | "
        f"Precision={precision:.6f} | "
        f"Recall={recall:.6f}"
    )


# ------------------------------------------------------------
# 7. SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)

if len(shared_nodes) == 0:

    print(
        "⚠️ No account/node is shared between legitimate "
        "and laundering activity anywhere in the dataset."
    )

else:

    print(
        f"Shared nodes exist: {len(shared_nodes):,}"
    )

if mixed == 0:

    print(
        "⚠️ No source account has mixed legitimate + AML behavior."
    )

else:

    print(
        f"Mixed-behavior source accounts: {mixed:,}"
    )

print()
print(
    "This diagnostic determines whether the dataset itself "
    "makes classification almost deterministic."
)

FULL DATASET ACCOUNT / LABEL STRUCTURE

SOURCE ACCOUNTS
--------------------------------------------------------------------------------
Unique legitimate senders : 494,582
Unique laundering senders : 99,574
Shared senders            : 0

DESTINATION ACCOUNTS
--------------------------------------------------------------------------------
Unique legitimate destinations : 418,938
Unique laundering destinations : 600,000
Shared destinations             : 0

ALL ACCOUNT / NODE OVERLAP
--------------------------------------------------------------------------------
Legitimate nodes : 513,650
AML nodes        : 699,574
Shared nodes     : 0

SOURCE ACCOUNT LABEL PURITY
Legitimate-only senders : 494,582
AML-only senders        : 99,574
Mixed-behavior senders  : 0

FANOUT RATIO DISTRIBUTION

LEGITIMATE
Min    : -0.710653
Median : -0.477284
Mean   : -0.246346
90%    : 0.522987
99%    : 1.541229
Max    : 2.235486

LAUNDERING
Min    : 1.386950
Median : 2.413792
Mean   : 2.404068
90%    : 2.432801

In [32]:
# ============================================================
# CELL 26 — GAT FEATURE ABLATION CHECK
# ============================================================
#
# Purpose:
# Determine whether the current perfect performance is mainly
# coming from:
#
#   1. Node features
#   2. Edge features
#   3. Both together
#
# We do NOT retrain the full model here.
# This cell simply measures the strongest single-feature
# predictors already present in the dataset.
# ============================================================

import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

print("=" * 80)
print("AML FEATURE ABLATION / SEPARABILITY CHECK")
print("=" * 80)

# ------------------------------------------------------------
# Test data
# ------------------------------------------------------------

test_ids = TEST_IDS

y = labels_np[test_ids]

edge_test = E_features[test_ids]

source_nodes = src_np[test_ids]

node_test = X_node[
    source_nodes
]

# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------

def auc_pr(values):

    auc = roc_auc_score(
        y,
        values
    )

    # Make orientation independent
    if auc < 0.5:
        values = -values
        auc = 1.0 - auc

    pr = average_precision_score(
        y,
        values
    )

    return auc, pr


# ============================================================
# NODE FEATURES
# ============================================================

print()
print("=" * 80)
print("NODE FEATURES")
print("=" * 80)

node_scores = []

for i, name in enumerate(node_feature_names):

    auc, pr = auc_pr(
        node_test[:, i]
    )

    node_scores.append(
        (
            name,
            auc,
            pr
        )
    )

node_scores.sort(
    key=lambda x: x[1],
    reverse=True
)

for name, auc, pr in node_scores:

    print(
        f"{name:25s} "
        f"AUC={auc:.6f} "
        f"PR-AUC={pr:.6f}"
    )


# ============================================================
# EDGE FEATURES
# ============================================================

print()
print("=" * 80)
print("EDGE FEATURES")
print("=" * 80)

edge_scores = []

for i, name in enumerate(edge_feature_names):

    auc, pr = auc_pr(
        edge_test[:, i]
    )

    edge_scores.append(
        (
            name,
            auc,
            pr
        )
    )

edge_scores.sort(
    key=lambda x: x[1],
    reverse=True
)

for name, auc, pr in edge_scores:

    print(
        f"{name:25s} "
        f"AUC={auc:.6f} "
        f"PR-AUC={pr:.6f}"
    )


# ============================================================
# BEST FEATURES
# ============================================================

best_node = node_scores[0]
best_edge = edge_scores[0]

print()
print("=" * 80)
print("STRONGEST CURRENT SIGNALS")
print("=" * 80)

print(
    f"Best node feature : "
    f"{best_node[0]} | "
    f"AUC={best_node[1]:.6f} | "
    f"PR-AUC={best_node[2]:.6f}"
)

print(
    f"Best edge feature : "
    f"{best_edge[0]} | "
    f"AUC={best_edge[1]:.6f} | "
    f"PR-AUC={best_edge[2]:.6f}"
)


# ============================================================
# ACCOUNT OVERLAP
# ============================================================

legit_nodes = set(
    source_nodes[y == 0]
) | set(
    dst_np[test_ids[y == 0]]
)

aml_nodes = set(
    source_nodes[y == 1]
) | set(
    dst_np[test_ids[y == 1]]
)

shared_nodes = (
    legit_nodes &
    aml_nodes
)

print()
print("=" * 80)
print("ACCOUNT SHARING")
print("=" * 80)

print(
    f"Legitimate nodes : {len(legit_nodes):,}"
)

print(
    f"AML nodes        : {len(aml_nodes):,}"
)

print(
    f"Shared nodes     : {len(shared_nodes):,}"
)


# ============================================================
# FINAL INTERPRETATION
# ============================================================

print()
print("=" * 80)
print("INTERPRETATION")
print("=" * 80)

if len(shared_nodes) == 0:

    print(
        "⚠️ Legitimate and laundering graph components are "
        "completely disconnected in this benchmark."
    )

    print(
        "The GAT can therefore exploit component/account identity "
        "rather than learning subtle transaction-level behavior."
    )

print()
print(
    "For the final AML benchmark we should introduce mixed-"
    "behavior accounts and overlapping graph structure."
)

AML FEATURE ABLATION / SEPARABILITY CHECK

NODE FEATURES
fanout_ratio              AUC=0.999995 PR-AUC=0.999999
in_count                  AUC=0.976917 PR-AUC=0.989653
unique_senders            AUC=0.976917 PR-AUC=0.989653
pass_through_ratio        AUC=0.976917 PR-AUC=0.989653
in_mean                   AUC=0.971256 PR-AUC=0.987148
in_total                  AUC=0.961508 PR-AUC=0.982864
out_mean                  AUC=0.935889 PR-AUC=0.983306
unique_receivers          AUC=0.904045 PR-AUC=0.918176
out_max                   AUC=0.900620 PR-AUC=0.965681
net_flow                  AUC=0.874103 PR-AUC=0.939292
out_total                 AUC=0.864426 PR-AUC=0.937131
out_count                 AUC=0.581363 PR-AUC=0.797737
total_degree              AUC=0.573233 PR-AUC=0.801871

EDGE FEATURES
pair_recency_hours        AUC=0.965347 PR-AUC=0.984548
first_pair_transaction    AUC=0.965325 PR-AUC=0.984539
amount_difference         AUC=0.733551 PR-AUC=0.901891
hour_cos                  AUC=0.591940 PR-AUC=0.

In [33]:
# ============================================================
# CELL 27 — UPDATE EXISTING 10M DATASET IN PLACE
# ============================================================
#
# We DO NOT create a new dataset.
#
# We modify the existing 10M transaction dataset so that
# selected laundering transactions use legitimate account
# identities, creating mixed-behavior accounts.
#
# Labels, amounts, currencies, formats and timestamps remain
# unchanged.
# ============================================================

import os
import time
import numpy as np
import pandas as pd

print("=" * 80)
print("UPDATING EXISTING AML DATASET IN PLACE")
print("=" * 80)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

RANDOM_SEED = 42

# Fraction of AML sender identities to merge into
# legitimate sender identities.
SOURCE_MIX_FRACTION = 0.70

# Fraction of AML destination identities to merge into
# legitimate destination identities.
DEST_MIX_FRACTION = 0.50

rng = np.random.default_rng(
    RANDOM_SEED
)

# ------------------------------------------------------------
# Use the repaired current dataframe
# ------------------------------------------------------------

update_df = df.copy()

# Keep only the original CSV columns
original_columns = [
    "transaction_id",
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
    "Is Laundering"
]

update_df = update_df[
    original_columns
].copy()

print(
    f"Transactions : {len(update_df):,}"
)

# ------------------------------------------------------------
# Verify timestamps before modification
# ------------------------------------------------------------

assert update_df["Timestamp"].notna().all(), \
    "Timestamp contains missing values."

print(
    "Timestamps   : 100% valid"
)

# ------------------------------------------------------------
# Build composite account identities
# ------------------------------------------------------------

src_key = (
    update_df["From Bank"].astype(str)
    + "::"
    + update_df["Account"].astype(str)
)

dst_key = (
    update_df["To Bank"].astype(str)
    + "::"
    + update_df["Account.1"].astype(str)
)

labels = update_df[
    "Is Laundering"
].to_numpy(
    dtype=np.int8
)

legit_mask = labels == 0
aml_mask = labels == 1

# ------------------------------------------------------------
# Existing account populations
# ------------------------------------------------------------

legit_src = np.unique(
    src_key[legit_mask]
)

aml_src = np.unique(
    src_key[aml_mask]
)

legit_dst = np.unique(
    dst_key[legit_mask]
)

aml_dst = np.unique(
    dst_key[aml_mask]
)

print()
print("BEFORE UPDATE")
print("-" * 80)

print(
    f"Legitimate source accounts : "
    f"{len(legit_src):,}"
)

print(
    f"AML source accounts        : "
    f"{len(aml_src):,}"
)

print(
    f"Legitimate destinations    : "
    f"{len(legit_dst):,}"
)

print(
    f"AML destinations           : "
    f"{len(aml_dst):,}"
)

# ------------------------------------------------------------
# Number of accounts to merge
# ------------------------------------------------------------

n_source_merge = min(
    int(
        len(aml_src)
        * SOURCE_MIX_FRACTION
    ),
    len(legit_src)
)

n_destination_merge = min(
    int(
        len(aml_dst)
        * DEST_MIX_FRACTION
    ),
    len(legit_dst)
)

print()
print("ACCOUNT MERGING")
print("-" * 80)

print(
    f"AML source accounts merged : "
    f"{n_source_merge:,}"
)

print(
    f"AML destination accounts merged : "
    f"{n_destination_merge:,}"
)

# ------------------------------------------------------------
# Select accounts
# ------------------------------------------------------------

selected_aml_src = rng.choice(
    aml_src,
    size=n_source_merge,
    replace=False
)

selected_legit_src = rng.choice(
    legit_src,
    size=n_source_merge,
    replace=False
)

selected_aml_dst = rng.choice(
    aml_dst,
    size=n_destination_merge,
    replace=False
)

selected_legit_dst = rng.choice(
    legit_dst,
    size=n_destination_merge,
    replace=False
)

# ------------------------------------------------------------
# Create one-to-one mappings
# ------------------------------------------------------------

source_mapping = dict(
    zip(
        selected_aml_src,
        selected_legit_src
    )
)

destination_mapping = dict(
    zip(
        selected_aml_dst,
        selected_legit_dst
    )
)

# ------------------------------------------------------------
# Re-map laundering source accounts
# ------------------------------------------------------------

print()
print("Updating laundering SOURCE accounts...")

aml_indices = np.where(
    aml_mask
)[0]

aml_src_series = src_key.iloc[
    aml_indices
]

mapped_sources = aml_src_series.map(
    source_mapping
)

source_update_mask = (
    mapped_sources.notna()
)

source_update_indices = aml_indices[
    source_update_mask.to_numpy()
]

mapped_source_values = (
    mapped_sources[
        source_update_mask
    ]
    .astype(str)
)

source_parts = mapped_source_values.str.split(
    "::",
    n=1,
    expand=True
)

update_df.loc[
    source_update_indices,
    "From Bank"
] = pd.to_numeric(
    source_parts[0]
).to_numpy()

update_df.loc[
    source_update_indices,
    "Account"
] = source_parts[1].to_numpy()

print(
    f"Source transactions updated : "
    f"{len(source_update_indices):,}"
)

# ------------------------------------------------------------
# Re-map laundering destination accounts
# ------------------------------------------------------------

print(
    "Updating laundering DESTINATION accounts..."
)

aml_dst_series = dst_key.iloc[
    aml_indices
]

mapped_destinations = aml_dst_series.map(
    destination_mapping
)

destination_update_mask = (
    mapped_destinations.notna()
)

destination_update_indices = aml_indices[
    destination_update_mask.to_numpy()
]

mapped_destination_values = (
    mapped_destinations[
        destination_update_mask
    ]
    .astype(str)
)

destination_parts = mapped_destination_values.str.split(
    "::",
    n=1,
    expand=True
)

update_df.loc[
    destination_update_indices,
    "To Bank"
] = pd.to_numeric(
    destination_parts[0]
).to_numpy()

update_df.loc[
    destination_update_indices,
    "Account.1"
] = destination_parts[1].to_numpy()

print(
    f"Destination transactions updated : "
    f"{len(destination_update_indices):,}"
)

# ------------------------------------------------------------
# Fix integer types
# ------------------------------------------------------------

update_df["From Bank"] = (
    pd.to_numeric(
        update_df["From Bank"]
    )
    .astype(np.int64)
)

update_df["To Bank"] = (
    pd.to_numeric(
        update_df["To Bank"]
    )
    .astype(np.int64)
)

# ------------------------------------------------------------
# Verify labels have NOT changed
# ------------------------------------------------------------

new_labels = update_df[
    "Is Laundering"
].to_numpy(
    dtype=np.int8
)

assert np.array_equal(
    labels,
    new_labels
)

print()
print(
    "✅ Labels unchanged."
)

# ------------------------------------------------------------
# Verify transaction count
# ------------------------------------------------------------

assert len(update_df) == 10_000_000

print(
    "✅ Transaction count remains 10,000,000."
)

# ------------------------------------------------------------
# Recalculate account identity overlap
# ------------------------------------------------------------

new_src_key = (
    update_df["From Bank"].astype(str)
    + "::"
    + update_df["Account"].astype(str)
)

new_dst_key = (
    update_df["To Bank"].astype(str)
    + "::"
    + update_df["Account.1"].astype(str)
)

legit_src_after = set(
    new_src_key[labels == 0].unique()
)

aml_src_after = set(
    new_src_key[labels == 1].unique()
)

legit_dst_after = set(
    new_dst_key[labels == 0].unique()
)

aml_dst_after = set(
    new_dst_key[labels == 1].unique()
)

shared_sources = (
    legit_src_after &
    aml_src_after
)

shared_destinations = (
    legit_dst_after &
    aml_dst_after
)

# ------------------------------------------------------------
# Mixed-behavior source accounts
# ------------------------------------------------------------

behavior_df = pd.DataFrame({
    "src": new_src_key,
    "label": labels
})

behavior_counts = (
    behavior_df
    .groupby(
        ["src", "label"]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

if 0 not in behavior_counts.columns:
    behavior_counts[0] = 0

if 1 not in behavior_counts.columns:
    behavior_counts[1] = 0

mixed_sources = np.sum(
    (behavior_counts[0] > 0)
    &
    (behavior_counts[1] > 0)
)

# ------------------------------------------------------------
# Save BACK to the SAME CSV
# ------------------------------------------------------------

print()
print("=" * 80)
print("OVERWRITING ORIGINAL DATASET")
print("=" * 80)

save_start = time.time()

update_df.to_csv(
    CSV_FILE,
    index=False
)

save_time = (
    time.time()
    - save_start
)

# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

print()
print("=" * 80)
print("DATASET UPDATED SUCCESSFULLY")
print("=" * 80)

print(
    f"File : {CSV_FILE}"
)

print(
    f"Save time : {save_time:.2f} sec"
)

print()
print("AFTER UPDATE")
print("-" * 80)

print(
    f"Shared source accounts      : "
    f"{len(shared_sources):,}"
)

print(
    f"Shared destination accounts : "
    f"{len(shared_destinations):,}"
)

print(
    f"Mixed-behavior senders      : "
    f"{mixed_sources:,}"
)

print()
print("Labels:")
print(
    pd.Series(
        labels
    ).value_counts().sort_index().to_dict()
)

print()
print(
    "✅ Same 10M dataset updated."
)

print(
    "✅ No separate hard-mode dataset created."
)

print(
    "✅ Labels and transaction quantities preserved."
)

UPDATING EXISTING AML DATASET IN PLACE
Transactions : 10,000,000
Timestamps   : 100% valid

BEFORE UPDATE
--------------------------------------------------------------------------------
Legitimate source accounts : 494,582
AML source accounts        : 99,574
Legitimate destinations    : 418,938
AML destinations           : 600,000

ACCOUNT MERGING
--------------------------------------------------------------------------------
AML source accounts merged : 69,701
AML destination accounts merged : 300,000

Updating laundering SOURCE accounts...
Source transactions updated : 3,499,321
Updating laundering DESTINATION accounts...
Destination transactions updated : 2,499,481

✅ Labels unchanged.
✅ Transaction count remains 10,000,000.

OVERWRITING ORIGINAL DATASET

DATASET UPDATED SUCCESSFULLY
File : /home/llyods-aids5/AML/HI-Small_FANOUT_10M_transactions.csv
Save time : 25.31 sec

AFTER UPDATE
--------------------------------------------------------------------------------
Shared source ac

In [38]:
# ============================================================
# CELL 27 — CORRECTED ACCOUNT-LEVEL AML PURITY ANALYSIS
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("CORRECT ACCOUNT-LEVEL AML PURITY ANALYSIS")
print("=" * 80)

# ------------------------------------------------------------
# Build unique account identity = Bank + Account
# ------------------------------------------------------------

src_accounts = pd.DataFrame({
    "account_id": (
        aml_data["From Bank"].astype(str)
        + "::"
        + aml_data["Account"].astype(str)
    ),
    "is_laundering": aml_data["Is Laundering"].astype(np.int8)
})

dst_accounts = pd.DataFrame({
    "account_id": (
        aml_data["To Bank"].astype(str)
        + "::"
        + aml_data["Account.1"].astype(str)
    ),
    "is_laundering": aml_data["Is Laundering"].astype(np.int8)
})

# ------------------------------------------------------------
# Combine every account appearance
# ------------------------------------------------------------

all_accounts = pd.concat(
    [src_accounts, dst_accounts],
    ignore_index=True
)

# ------------------------------------------------------------
# Count total and laundering transactions per account
# ------------------------------------------------------------

account_status = (
    all_accounts
    .groupby("account_id")["is_laundering"]
    .agg(
        total_transactions="count",
        laundering_transactions="sum"
    )
)

# ------------------------------------------------------------
# Create aligned boolean masks
# ------------------------------------------------------------

legitimate_only = (
    account_status["laundering_transactions"] == 0
)

laundering_only = (
    account_status["laundering_transactions"]
    == account_status["total_transactions"]
)

mixed_behavior = (
    (account_status["laundering_transactions"] > 0)
    &
    (
        account_status["laundering_transactions"]
        < account_status["total_transactions"]
    )
)

# ------------------------------------------------------------
# Counts
# ------------------------------------------------------------

num_total = len(account_status)

num_legitimate_only = int(
    legitimate_only.sum()
)

num_laundering_only = int(
    laundering_only.sum()
)

num_mixed = int(
    mixed_behavior.sum()
)

print()
print("=" * 80)
print("ACCOUNT POPULATION")
print("=" * 80)

print(
    f"Total unique accounts       : {num_total:,}"
)

print(
    f"Legitimate-only accounts    : "
    f"{num_legitimate_only:,}"
)

print(
    f"Laundering-only accounts    : "
    f"{num_laundering_only:,}"
)

print(
    f"Mixed-behavior accounts     : "
    f"{num_mixed:,}"
)

print()

print(
    f"Laundering-only %           : "
    f"{num_laundering_only / num_total:.2%}"
)

print(
    f"Mixed-behavior %            : "
    f"{num_mixed / num_total:.2%}"
)

# ------------------------------------------------------------
# Account sets
# ------------------------------------------------------------

legitimate_accounts = set(
    account_status.index[
        ~ (account_status["laundering_transactions"] > 0)
    ]
)

aml_accounts = set(
    account_status.index[
        account_status["laundering_transactions"] > 0
    ]
)

shared_accounts = (
    legitimate_accounts &
    aml_accounts
)

print()
print("=" * 80)
print("ACCOUNT OVERLAP")
print("=" * 80)

print(
    f"Accounts with no AML activity : "
    f"{len(legitimate_accounts):,}"
)

print(
    f"Accounts with AML activity    : "
    f"{len(aml_accounts):,}"
)

print(
    f"Accounts appearing in both    : "
    f"{len(shared_accounts):,}"
)

# ------------------------------------------------------------
# Verify relationship between overlap and mixed behavior
# ------------------------------------------------------------

print()
print(
    "Mixed-behavior accounts should be the accounts appearing "
    "in both legitimate and laundering transactions."
)

# ------------------------------------------------------------
# Show sample mixed accounts
# ------------------------------------------------------------

if num_mixed > 0:

    print()
    print("=" * 80)
    print("SAMPLE MIXED-BEHAVIOR ACCOUNTS")
    print("=" * 80)

    sample_mixed = (
        account_status[
            mixed_behavior
        ]
        .copy()
        .sort_values(
            "total_transactions",
            ascending=False
        )
        .head(10)
    )

    sample_mixed["legitimate_transactions"] = (
        sample_mixed["total_transactions"]
        -
        sample_mixed["laundering_transactions"]
    )

    print(
        sample_mixed[
            [
                "total_transactions",
                "legitimate_transactions",
                "laundering_transactions"
            ]
        ].to_string()
    )

else:

    print()
    print("=" * 80)
    print("NO MIXED-BEHAVIOR ACCOUNTS")
    print("=" * 80)

    print(
        "Every account is either legitimate-only or "
        "laundering-only."
    )

# ------------------------------------------------------------
# Final sanity check
# ------------------------------------------------------------

assert (
    num_legitimate_only
    + num_laundering_only
    + num_mixed
    == num_total
)

print()
print("=" * 80)
print("✅ ACCOUNT ANALYSIS COMPLETE")
print("=" * 80)

CORRECT ACCOUNT-LEVEL AML PURITY ANALYSIS

ACCOUNT POPULATION
Total unique accounts       : 843,523
Legitimate-only accounts    : 184,329
Laundering-only accounts    : 329,873
Mixed-behavior accounts     : 329,321

Laundering-only %           : 39.11%
Mixed-behavior %            : 39.04%

ACCOUNT OVERLAP
Accounts with no AML activity : 184,329
Accounts with AML activity    : 659,194
Accounts appearing in both    : 0

Mixed-behavior accounts should be the accounts appearing in both legitimate and laundering transactions.

SAMPLE MIXED-BEHAVIOR ACCOUNTS
               total_transactions  legitimate_transactions  laundering_transactions
account_id                                                                         
70::100428660              167027                   167019                        8
70::1004286A8              102018                   102009                        9
70::1004286F0               18482                    18474                        8
70::100428780         

In [40]:
# ============================================================
# CELL 28 — VERIFY ACCOUNT ID ↔ GRAPH NODE MAPPING
# ============================================================

import numpy as np
import pandas as pd

print("=" * 80)
print("VERIFYING ACCOUNT ID ↔ GRAPH NODE MAPPING")
print("=" * 80)

# ------------------------------------------------------------
# 1. Reconstruct account keys exactly as feature engine does
# ------------------------------------------------------------

feature_src_key = (
    df_features["From Bank"].astype(str)
    + "::"
    + df_features["Account"].astype(str)
)

feature_dst_key = (
    df_features["To Bank"].astype(str)
    + "::"
    + df_features["Account.1"].astype(str)
)

print("\nFeature-engine account keys created.")

# ------------------------------------------------------------
# 2. Check whether the same account key maps to exactly
#    one source node ID
# ------------------------------------------------------------

src_map_check = pd.DataFrame({
    "account_key": feature_src_key,
    "src_id": src_np
})

src_unique_counts = (
    src_map_check
    .groupby("account_key")["src_id"]
    .nunique()
)

bad_src_mappings = (
    src_unique_counts > 1
).sum()

print()
print("SOURCE NODE MAPPING")
print("-" * 80)

print(
    f"Unique source account keys : "
    f"{src_unique_counts.shape[0]:,}"
)

print(
    f"Keys mapping to >1 src_id   : "
    f"{bad_src_mappings:,}"
)

# ------------------------------------------------------------
# 3. Destination mapping
# ------------------------------------------------------------

dst_map_check = pd.DataFrame({
    "account_key": feature_dst_key,
    "dst_id": dst_np
})

dst_unique_counts = (
    dst_map_check
    .groupby("account_key")["dst_id"]
    .nunique()
)

bad_dst_mappings = (
    dst_unique_counts > 1
).sum()

print()
print("DESTINATION NODE MAPPING")
print("-" * 80)

print(
    f"Unique destination account keys : "
    f"{dst_unique_counts.shape[0]:,}"
)

print(
    f"Keys mapping to >1 dst_id         : "
    f"{bad_dst_mappings:,}"
)

# ------------------------------------------------------------
# 4. Check whether the SAME account key has both labels
#    at the transaction level
# ------------------------------------------------------------

account_label_df = pd.DataFrame({
    "account_key": pd.concat(
        [
            feature_src_key,
            feature_dst_key
        ],
        ignore_index=True
    ),
    "label": np.concatenate(
        [
            labels_np,
            labels_np
        ]
    )
})

account_labels = (
    account_label_df
    .groupby("account_key")["label"]
    .agg(
        min_label="min",
        max_label="max",
        count="count"
    )
)

mixed_keys = (
    (account_labels["min_label"] == 0)
    &
    (account_labels["max_label"] == 1)
)

print()
print("=" * 80)
print("ACCOUNT-LEVEL LABEL MIXING")
print("=" * 80)

print(
    f"Unique account keys : "
    f"{len(account_labels):,}"
)

print(
    f"Mixed account keys  : "
    f"{mixed_keys.sum():,}"
)

# ------------------------------------------------------------
# 5. For mixed accounts, inspect their graph node IDs
# ------------------------------------------------------------

mixed_account_keys = (
    account_labels.index[
        mixed_keys
    ]
)

print()
print("=" * 80)
print("MIXED ACCOUNT → GRAPH NODE CHECK")
print("=" * 80)

# Source mappings
mixed_src_map = (
    src_map_check[
        src_map_check["account_key"].isin(
            mixed_account_keys
        )
    ]
    .groupby("account_key")["src_id"]
    .unique()
)

# Destination mappings
mixed_dst_map = (
    dst_map_check[
        dst_map_check["account_key"].isin(
            mixed_account_keys
        )
    ]
    .groupby("account_key")["dst_id"]
    .unique()
)

# ------------------------------------------------------------
# Show first 10
# ------------------------------------------------------------

sample_keys = mixed_account_keys[:10]

for key in sample_keys:

    src_ids = mixed_src_map.get(
        key,
        np.array([], dtype=np.int64)
    )

    dst_ids = mixed_dst_map.get(
        key,
        np.array([], dtype=np.int64)
    )

    print()
    print(f"Account: {key}")
    print(
        f"  src node IDs : {src_ids[:10]}"
    )
    print(
        f"  dst node IDs : {dst_ids[:10]}"
    )

# ------------------------------------------------------------
# 6. Direct graph-node overlap check using labels
# ------------------------------------------------------------

legit_nodes = set(
    src_np[labels_np == 0]
) | set(
    dst_np[labels_np == 0]
)

aml_nodes = set(
    src_np[labels_np == 1]
) | set(
    dst_np[labels_np == 1]
)

shared_graph_nodes = (
    legit_nodes &
    aml_nodes
)

print()
print("=" * 80)
print("GRAPH NODE OVERLAP")
print("=" * 80)

print(
    f"Legitimate graph nodes : "
    f"{len(legit_nodes):,}"
)

print(
    f"AML graph nodes        : "
    f"{len(aml_nodes):,}"
)

print(
    f"Shared graph nodes     : "
    f"{len(shared_graph_nodes):,}"
)

# ------------------------------------------------------------
# 7. Final consistency interpretation
# ------------------------------------------------------------

print()
print("=" * 80)
print("CONSISTENCY CHECK")
print("=" * 80)

if mixed_keys.sum() > 0 and len(shared_graph_nodes) == 0:

    print(
        "⚠️ Account-level analysis sees mixed accounts, "
        "but graph node IDs show zero overlap."
    )

    print(
        "This means the account-to-node mapping is likely "
        "the source of the inconsistency."
    )

elif mixed_keys.sum() > 0 and len(shared_graph_nodes) > 0:

    print(
        "✅ Account-level and graph-level analyses agree."
    )

    print(
        "The dataset genuinely contains shared/mixed accounts."
    )

else:

    print(
        "⚠️ No mixed account keys were found."
    )

print()
print("Diagnostic complete.")

VERIFYING ACCOUNT ID ↔ GRAPH NODE MAPPING

Feature-engine account keys created.

SOURCE NODE MAPPING
--------------------------------------------------------------------------------
Unique source account keys : 594,156
Keys mapping to >1 src_id   : 0

DESTINATION NODE MAPPING
--------------------------------------------------------------------------------
Unique destination account keys : 1,018,938
Keys mapping to >1 dst_id         : 0

ACCOUNT-LEVEL LABEL MIXING
Unique account keys : 1,213,224
Mixed account keys  : 0

MIXED ACCOUNT → GRAPH NODE CHECK

GRAPH NODE OVERLAP
Legitimate graph nodes : 513,650
AML graph nodes        : 699,574
Shared graph nodes     : 0

CONSISTENCY CHECK
⚠️ No mixed account keys were found.

Diagnostic complete.


In [41]:
# ============================================================
# CELL 29 — RAW ACCOUNT VS GRAPH ACCOUNT IDENTITY
# ============================================================

import numpy as np
import pandas as pd

print("=" * 80)
print("RAW ACCOUNT vs GRAPH ACCOUNT IDENTITY CHECK")
print("=" * 80)

# ------------------------------------------------------------
# 1. RAW CSV: Bank + Account identity
# ------------------------------------------------------------

raw_src = (
    aml_data["From Bank"].astype(str)
    + "::"
    + aml_data["Account"].astype(str)
)

raw_dst = (
    aml_data["To Bank"].astype(str)
    + "::"
    + aml_data["Account.1"].astype(str)
)

raw_labels = aml_data[
    "Is Laundering"
].to_numpy(dtype=np.int8)

raw_all = pd.concat(
    [
        pd.DataFrame({
            "account": raw_src,
            "label": raw_labels
        }),
        pd.DataFrame({
            "account": raw_dst,
            "label": raw_labels
        })
    ],
    ignore_index=True
)

raw_status = (
    raw_all
    .groupby("account")["label"]
    .agg(["min", "max"])
)

raw_mixed = (
    (raw_status["min"] == 0)
    & (raw_status["max"] == 1)
)

print()
print("RAW CSV")
print("-" * 80)

print(
    f"Unique Bank+Account identities : "
    f"{len(raw_status):,}"
)

print(
    f"Mixed Bank+Account identities  : "
    f"{int(raw_mixed.sum()):,}"
)

# ------------------------------------------------------------
# 2. FEATURE ENGINE: Bank + Account identity
# ------------------------------------------------------------

feature_src = (
    df_features["From Bank"].astype(str)
    + "::"
    + df_features["Account"].astype(str)
)

feature_dst = (
    df_features["To Bank"].astype(str)
    + "::"
    + df_features["Account.1"].astype(str)
)

feature_labels = df_features[
    "Is Laundering"
].to_numpy(dtype=np.int8)

feature_all = pd.concat(
    [
        pd.DataFrame({
            "account": feature_src,
            "label": feature_labels
        }),
        pd.DataFrame({
            "account": feature_dst,
            "label": feature_labels
        })
    ],
    ignore_index=True
)

feature_status = (
    feature_all
    .groupby("account")["label"]
    .agg(["min", "max"])
)

feature_mixed = (
    (feature_status["min"] == 0)
    & (feature_status["max"] == 1)
)

print()
print("FEATURE ENGINE")
print("-" * 80)

print(
    f"Unique Bank+Account identities : "
    f"{len(feature_status):,}"
)

print(
    f"Mixed Bank+Account identities  : "
    f"{int(feature_mixed.sum()):,}"
)

# ------------------------------------------------------------
# 3. Compare the account-key populations
# ------------------------------------------------------------

raw_keys = set(
    raw_status.index
)

feature_keys = set(
    feature_status.index
)

print()
print("=" * 80)
print("ACCOUNT KEY POPULATION COMPARISON")
print("=" * 80)

print(
    f"Raw unique keys      : {len(raw_keys):,}"
)

print(
    f"Feature unique keys  : {len(feature_keys):,}"
)

print(
    f"Keys only in raw     : "
    f"{len(raw_keys - feature_keys):,}"
)

print(
    f"Keys only in feature : "
    f"{len(feature_keys - raw_keys):,}"
)

print(
    f"Keys shared          : "
    f"{len(raw_keys & feature_keys):,}"
)

# ------------------------------------------------------------
# 4. Check whether transactions remain row-aligned
# ------------------------------------------------------------

print()
print("=" * 80)
print("TRANSACTION ALIGNMENT CHECK")
print("=" * 80)

print(
    f"Raw rows     : {len(aml_data):,}"
)

print(
    f"Feature rows : {len(df_features):,}"
)

same_transaction_ids = np.array_equal(
    aml_data["transaction_id"].astype(str).to_numpy(),
    df_features["transaction_id"].astype(str).to_numpy()
)

same_labels = np.array_equal(
    raw_labels,
    feature_labels
)

print(
    f"Same transaction order : {same_transaction_ids}"
)

print(
    f"Same label order       : {same_labels}"
)

# ------------------------------------------------------------
# 5. Check actual src_key construction
# ------------------------------------------------------------

print()
print("=" * 80)
print("GRAPH KEY CONSTRUCTION")
print("=" * 80)

src_expected = (
    df_features["From Bank"].astype(str)
    + "::"
    + df_features["Account"].astype(str)
)

dst_expected = (
    df_features["To Bank"].astype(str)
    + "::"
    + df_features["Account.1"].astype(str)
)

src_match = np.mean(
    df_features["src_key"].astype(str).to_numpy()
    ==
    src_expected.to_numpy()
)

dst_match = np.mean(
    df_features["dst_key"].astype(str).to_numpy()
    ==
    dst_expected.to_numpy()
)

print(
    f"src_key == Bank::Account : {src_match:.4%}"
)

print(
    f"dst_key == Bank::Account : {dst_match:.4%}"
)

# ------------------------------------------------------------
# 6. Final interpretation
# ------------------------------------------------------------

print()
print("=" * 80)
print("INTERPRETATION")
print("=" * 80)

if raw_mixed > 0 and feature_mixed == 0:

    print(
        "⚠️ Raw data has mixed accounts, but the feature-engine "
        "data does not."
    )

elif raw_mixed == feature_mixed and feature_mixed > 0:

    print(
        "✅ Raw and feature-engine data agree that mixed accounts exist."
    )

elif raw_mixed == 0:

    print(
        "✅ The raw Bank+Account identity has no mixed accounts."
    )

else:

    print(
        "⚠️ The two representations differ."
    )

RAW ACCOUNT vs GRAPH ACCOUNT IDENTITY CHECK

RAW CSV
--------------------------------------------------------------------------------
Unique Bank+Account identities : 843,523
Mixed Bank+Account identities  : 329,321

FEATURE ENGINE
--------------------------------------------------------------------------------
Unique Bank+Account identities : 1,213,224
Mixed Bank+Account identities  : 0

ACCOUNT KEY POPULATION COMPARISON
Raw unique keys      : 843,523
Feature unique keys  : 1,213,224
Keys only in raw     : 0
Keys only in feature : 369,701
Keys shared          : 843,523

TRANSACTION ALIGNMENT CHECK
Raw rows     : 10,000,000
Feature rows : 10,000,000
Same transaction order : False
Same label order       : False

GRAPH KEY CONSTRUCTION
src_key == Bank::Account : 0.0000%
dst_key == Bank::Account : 0.0000%

INTERPRETATION


ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [42]:

# ============================================================
# CELL 29B — FIX FINAL ACCOUNT IDENTITY INTERPRETATION
# ============================================================

raw_mixed_count = int(
    raw_mixed.sum()
)

feature_mixed_count = int(
    feature_mixed.sum()
)

print("=" * 80)
print("FINAL ACCOUNT IDENTITY COMPARISON")
print("=" * 80)

print(
    f"Raw CSV mixed accounts      : {raw_mixed_count:,}"
)

print(
    f"Feature-engine mixed accounts: {feature_mixed_count:,}"
)

print()

if raw_mixed_count > 0 and feature_mixed_count == 0:

    print(
        "⚠️ Raw CSV has mixed accounts, but the feature-engine "
        "representation has none."
    )

elif (
    raw_mixed_count == feature_mixed_count
    and feature_mixed_count > 0
):

    print(
        "✅ Raw CSV and feature-engine representation agree."
    )

elif raw_mixed_count == 0:

    print(
        "✅ Raw Bank+Account identity contains no mixed accounts."
    )

else:

    print(
        "⚠️ The raw and feature-engine account representations differ."
    )

print()
print("=" * 80)
print("KEY FORMAT MATCH")
print("=" * 80)

print(
    f"src_key == Bank::Account : "
    f"{src_match:.4%}"
)

print(
    f"dst_key == Bank::Account : "
    f"{dst_match:.4%}"
)

print()
print("=" * 80)
print("TRANSACTION ALIGNMENT")
print("=" * 80)

print(
    f"Same transaction order : {same_transaction_ids}"
)

print(
    f"Same label order       : {same_labels}"
)

FINAL ACCOUNT IDENTITY COMPARISON
Raw CSV mixed accounts      : 329,321
Feature-engine mixed accounts: 0

⚠️ Raw CSV has mixed accounts, but the feature-engine representation has none.

KEY FORMAT MATCH
src_key == Bank::Account : 0.0000%
dst_key == Bank::Account : 0.0000%

TRANSACTION ALIGNMENT
Same transaction order : False
Same label order       : False


In [43]:
# ============================================================
# CELL 30 — INSPECT ACTUAL FEATURE-ENGINE ACCOUNT MAPPING
# ============================================================

import inspect
import pandas as pd

print("=" * 80)
print("ACTUAL AMLFeatureEngine ACCOUNT MAPPING")
print("=" * 80)

# ------------------------------------------------------------
# 1. Print the actual transform source around src_key/dst_key
# ------------------------------------------------------------

source = inspect.getsource(
    AMLFeatureEngine.transform
)

lines = source.splitlines()

print("\nFEATURE ENGINE TRANSFORM CODE")
print("-" * 80)

for i, line in enumerate(lines):

    if (
        "src_key" in line
        or
        "dst_key" in line
        or
        "src_id" in line
        or
        "dst_id" in line
        or
        "node" in line.lower()
    ):

        start = max(
            0,
            i - 3
        )

        end = min(
            len(lines),
            i + 4
        )

        print(
            f"\n--- lines {start+1} to {end} ---"
        )

        print(
            "\n".join(
                lines[start:end]
            )
        )

# ------------------------------------------------------------
# 2. Show actual feature-engine keys
# ------------------------------------------------------------

print()
print("=" * 80)
print("ACTUAL FEATURE-ENGINE KEYS")
print("=" * 80)

print(
    df_features[
        [
            "From Bank",
            "Account",
            "src_key",
            "src_id",
            "To Bank",
            "Account.1",
            "dst_key",
            "dst_id",
            "Is Laundering"
        ]
    ]
    .head(20)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 3. Show Python types of src_key / dst_key
# ------------------------------------------------------------

print()
print("=" * 80)
print("KEY TYPES")
print("=" * 80)

print(
    "src_key dtype:",
    df_features["src_key"].dtype
)

print(
    "dst_key dtype:",
    df_features["dst_key"].dtype
)

print(
    "First src_key Python type:",
    type(df_features["src_key"].iloc[0])
)

print(
    "First dst_key Python type:",
    type(df_features["dst_key"].iloc[0])
)

# ------------------------------------------------------------
# 4. Compare actual keys with candidate constructions
# ------------------------------------------------------------

src_actual = (
    df_features["src_key"]
    .astype(str)
)

dst_actual = (
    df_features["dst_key"]
    .astype(str)
)

candidates_src = {
    "Account":
        df_features["Account"].astype(str),

    "Bank::Account":
        (
            df_features["From Bank"].astype(str)
            + "::"
            + df_features["Account"].astype(str)
        ),

    "Bank_Account":
        (
            df_features["From Bank"].astype(str)
            + "_"
            + df_features["Account"].astype(str)
        ),

    "Bank-Account":
        (
            df_features["From Bank"].astype(str)
            + "-"
            + df_features["Account"].astype(str)
        ),

    "BankAccount":
        (
            df_features["From Bank"].astype(str)
            + df_features["Account"].astype(str)
        )
}

print()
print("=" * 80)
print("SOURCE KEY FORMAT MATCHES")
print("=" * 80)

for name, candidate in candidates_src.items():

    match = np.mean(
        src_actual.to_numpy()
        ==
        candidate.to_numpy()
    )

    print(
        f"{name:20s}: {match:.4%}"
    )

candidates_dst = {
    "Account":
        df_features["Account.1"].astype(str),

    "Bank::Account":
        (
            df_features["To Bank"].astype(str)
            + "::"
            + df_features["Account.1"].astype(str)
        ),

    "Bank_Account":
        (
            df_features["To Bank"].astype(str)
            + "_"
            + df_features["Account.1"].astype(str)
        ),

    "Bank-Account":
        (
            df_features["To Bank"].astype(str)
            + "-"
            + df_features["Account.1"].astype(str)
        ),

    "BankAccount":
        (
            df_features["To Bank"].astype(str)
            + df_features["Account.1"].astype(str)
        )
}

print()
print("=" * 80)
print("DESTINATION KEY FORMAT MATCHES")
print("=" * 80)

for name, candidate in candidates_dst.items():

    match = np.mean(
        dst_actual.to_numpy()
        ==
        candidate.to_numpy()
    )

    print(
        f"{name:20s}: {match:.4%}"
    )

print()
print("=" * 80)
print("DIAGNOSTIC COMPLETE")
print("=" * 80)

print(
    "We now use the actual feature-engine code/key format "
    "instead of assuming how account IDs were constructed."
)

ACTUAL AMLFeatureEngine ACCOUNT MAPPING

FEATURE ENGINE TRANSFORM CODE
--------------------------------------------------------------------------------

--- lines 74 to 80 ---
        # 4. ACCOUNT / GRAPH IDENTIFIERS
        # ==================================

        df["src_key"] = (
            df["From Bank"].astype(str).str.strip()
            + "_"
            + df["Account"].astype(str).str.strip()

--- lines 80 to 86 ---
            + df["Account"].astype(str).str.strip()
        )

        df["dst_key"] = (
            df["To Bank"].astype(str).str.strip()
            + "_"
            + df["Account.1"].astype(str).str.strip()

--- lines 152 to 158 ---

        # Graph relationship
        edge_features["self_loop"] = (
            df["src_key"] == df["dst_key"]
        ).astype(np.float32)

        edge_features["same_bank"] = (

--- lines 180 to 186 ---
        # ==================================

        df["pair_key"] = (
            df["src_key"] +
            "->" +
 

In [44]:
# ============================================================
# CELL 31 — ACCOUNT STRING VS TRUE GRAPH ACCOUNT IDENTITY
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("ACCOUNT IDENTITY COMPARISON")
print("=" * 80)

# ------------------------------------------------------------
# TRUE GRAPH identity = Bank + Account
# ------------------------------------------------------------

src_true = (
    aml_data["From Bank"].astype(str)
    + "_"
    + aml_data["Account"].astype(str)
)

dst_true = (
    aml_data["To Bank"].astype(str)
    + "_"
    + aml_data["Account.1"].astype(str)
)

labels_raw = aml_data[
    "Is Laundering"
].to_numpy(dtype=np.int8)

# ------------------------------------------------------------
# ACCOUNT STRING ONLY
# ------------------------------------------------------------

src_account_only = (
    aml_data["Account"].astype(str)
)

dst_account_only = (
    aml_data["Account.1"].astype(str)
)

# ------------------------------------------------------------
# Count mixed identities
# ------------------------------------------------------------

def mixed_count(
    src_keys,
    dst_keys,
    labels
):

    temp = pd.concat(
        [
            pd.DataFrame({
                "account": src_keys,
                "label": labels
            }),
            pd.DataFrame({
                "account": dst_keys,
                "label": labels
            })
        ],
        ignore_index=True
    )

    grouped = (
        temp
        .groupby("account")["label"]
        .agg(["min", "max"])
    )

    mixed = (
        (grouped["min"] == 0)
        &
        (grouped["max"] == 1)
    )

    return (
        len(grouped),
        int(mixed.sum())
    )

# ------------------------------------------------------------
# Account-only result
# ------------------------------------------------------------

account_only_total, account_only_mixed = mixed_count(
    src_account_only,
    dst_account_only,
    labels_raw
)

# ------------------------------------------------------------
# True graph identity result
# ------------------------------------------------------------

true_identity_total, true_identity_mixed = mixed_count(
    src_true,
    dst_true,
    labels_raw
)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print()
print("ACCOUNT STRING ONLY")
print("-" * 80)

print(
    f"Unique identities : "
    f"{account_only_total:,}"
)

print(
    f"Mixed identities  : "
    f"{account_only_mixed:,}"
)

print()
print("TRUE GRAPH IDENTITY (BANK + ACCOUNT)")
print("-" * 80)

print(
    f"Unique identities : "
    f"{true_identity_total:,}"
)

print(
    f"Mixed identities  : "
    f"{true_identity_mixed:,}"
)

# ------------------------------------------------------------
# Find account strings that looked mixed only because they
# belong to multiple banks.
# ------------------------------------------------------------

account_bank = pd.DataFrame({
    "account": aml_data["Account"].astype(str),
    "bank": aml_data["From Bank"],
    "label": labels_raw
})

bank_counts = (
    account_bank
    .groupby("account")["bank"]
    .nunique()
)

multi_bank_accounts = (
    bank_counts > 1
)

print()
print("=" * 80)
print("ACCOUNT STRINGS USED BY MULTIPLE BANKS")
print("=" * 80)

print(
    f"Account strings appearing across >1 bank : "
    f"{int(multi_bank_accounts.sum()):,}"
)

print()
print("=" * 80)
print("CONCLUSION")
print("=" * 80)

if true_identity_mixed == 0:

    print(
        "✅ No true Bank+Account identity currently has "
        "both legitimate and laundering transactions."
    )

    print(
        "Therefore the current graph has no mixed-behavior nodes."
    )

else:

    print(
        f"True mixed graph identities: "
        f"{true_identity_mixed:,}"
    )

print()
print(
    "The earlier Account-only mixed count should NOT be used "
    "as evidence of mixed graph accounts."
)

ACCOUNT IDENTITY COMPARISON

ACCOUNT STRING ONLY
--------------------------------------------------------------------------------
Unique identities : 843,515
Mixed identities  : 329,318

TRUE GRAPH IDENTITY (BANK + ACCOUNT)
--------------------------------------------------------------------------------
Unique identities : 843,523
Mixed identities  : 329,321

ACCOUNT STRINGS USED BY MULTIPLE BANKS
Account strings appearing across >1 bank : 4

CONCLUSION
True mixed graph identities: 329,321

The earlier Account-only mixed count should NOT be used as evidence of mixed graph accounts.


In [45]:
# ============================================================
# CELL 32 — DIRECT RAW CSV ↔ FEATURE GRAPH CONSISTENCY CHECK
# ============================================================

import numpy as np
import pandas as pd

print("=" * 80)
print("RAW CSV ↔ FEATURE GRAPH CONSISTENCY CHECK")
print("=" * 80)

# ------------------------------------------------------------
# 1. Build true Bank+Account keys from current df_features
# ------------------------------------------------------------

f_src_key = (
    df_features["From Bank"].astype(str).str.strip()
    + "_"
    + df_features["Account"].astype(str).str.strip()
)

f_dst_key = (
    df_features["To Bank"].astype(str).str.strip()
    + "_"
    + df_features["Account.1"].astype(str).str.strip()
)

f_labels = df_features[
    "Is Laundering"
].to_numpy(dtype=np.int8)

# ------------------------------------------------------------
# 2. Count mixed accounts DIRECTLY from df_features
# ------------------------------------------------------------

feature_accounts = pd.concat(
    [
        pd.DataFrame({
            "account_key": f_src_key,
            "label": f_labels
        }),
        pd.DataFrame({
            "account_key": f_dst_key,
            "label": f_labels
        })
    ],
    ignore_index=True
)

feature_status = (
    feature_accounts
    .groupby("account_key")["label"]
    .agg(["min", "max", "count"])
)

feature_mixed = (
    (feature_status["min"] == 0)
    &
    (feature_status["max"] == 1)
)

print()
print("DF_FEATURES DIRECT RESULT")
print("-" * 80)

print(
    f"Unique graph account keys : {len(feature_status):,}"
)

print(
    f"Mixed graph account keys  : {int(feature_mixed.sum()):,}"
)

# ------------------------------------------------------------
# 3. Build mixed accounts from RAW CSV
# ------------------------------------------------------------

r_src_key = (
    aml_data["From Bank"].astype(str).str.strip()
    + "_"
    + aml_data["Account"].astype(str).str.strip()
)

r_dst_key = (
    aml_data["To Bank"].astype(str).str.strip()
    + "_"
    + aml_data["Account.1"].astype(str).str.strip()
)

r_labels = aml_data[
    "Is Laundering"
].to_numpy(dtype=np.int8)

raw_accounts = pd.concat(
    [
        pd.DataFrame({
            "account_key": r_src_key,
            "label": r_labels
        }),
        pd.DataFrame({
            "account_key": r_dst_key,
            "label": r_labels
        })
    ],
    ignore_index=True
)

raw_status = (
    raw_accounts
    .groupby("account_key")["label"]
    .agg(["min", "max", "count"])
)

raw_mixed = (
    (raw_status["min"] == 0)
    &
    (raw_status["max"] == 1)
)

print()
print("RAW CSV DIRECT RESULT")
print("-" * 80)

print(
    f"Unique raw account keys    : {len(raw_status):,}"
)

print(
    f"Mixed raw account keys     : {int(raw_mixed.sum()):,}"
)

# ------------------------------------------------------------
# 4. Compare the actual account-key sets
# ------------------------------------------------------------

raw_keys = set(raw_status.index)
feature_keys = set(feature_status.index)

print()
print("=" * 80)
print("ACCOUNT KEY SET COMPARISON")
print("=" * 80)

print(
    f"Shared keys          : "
    f"{len(raw_keys & feature_keys):,}"
)

print(
    f"Raw-only keys        : "
    f"{len(raw_keys - feature_keys):,}"
)

print(
    f"Feature-only keys    : "
    f"{len(feature_keys - raw_keys):,}"
)

# ------------------------------------------------------------
# 5. Compare a few mixed accounts
# ------------------------------------------------------------

raw_mixed_keys = set(
    raw_status.index[
        raw_mixed
    ]
)

feature_mixed_keys = set(
    feature_status.index[
        feature_mixed
    ]
)

print()
print("=" * 80)
print("MIXED-ACCOUNT SET COMPARISON")
print("=" * 80)

print(
    f"Raw mixed accounts      : "
    f"{len(raw_mixed_keys):,}"
)

print(
    f"Feature mixed accounts  : "
    f"{len(feature_mixed_keys):,}"
)

print(
    f"Mixed keys shared       : "
    f"{len(raw_mixed_keys & feature_mixed_keys):,}"
)

print(
    f"Mixed keys raw-only     : "
    f"{len(raw_mixed_keys - feature_mixed_keys):,}"
)

print(
    f"Mixed keys feature-only : "
    f"{len(feature_mixed_keys - raw_mixed_keys):,}"
)

if len(raw_mixed_keys & feature_mixed_keys) > 0:

    print("\nSample shared mixed accounts:")

    for key in list(
        raw_mixed_keys & feature_mixed_keys
    )[:10]:

        print(
            f"  {key}"
        )

# ------------------------------------------------------------
# 6. Verify current graph node IDs against src_key/dst_key
# ------------------------------------------------------------

print()
print("=" * 80)
print("GRAPH NODE-ID CONSISTENCY")
print("=" * 80)

src_key_to_ids = (
    df_features[
        ["src_key", "src_id"]
    ]
    .drop_duplicates()
)

dst_key_to_ids = (
    df_features[
        ["dst_key", "dst_id"]
    ]
    .drop_duplicates()
)

print(
    f"Unique src_key → src_id pairs : "
    f"{len(src_key_to_ids):,}"
)

print(
    f"Unique dst_key → dst_id pairs : "
    f"{len(dst_key_to_ids):,}"
)

# ------------------------------------------------------------
# 7. Compute shared graph nodes DIRECTLY FROM CURRENT KEYS
# ------------------------------------------------------------

legit_src_keys = set(
    f_src_key[f_labels == 0].unique()
)

aml_src_keys = set(
    f_src_key[f_labels == 1].unique()
)

legit_dst_keys = set(
    f_dst_key[f_labels == 0].unique()
)

aml_dst_keys = set(
    f_dst_key[f_labels == 1].unique()
)

legit_keys = (
    legit_src_keys |
    legit_dst_keys
)

aml_keys = (
    aml_src_keys |
    aml_dst_keys
)

shared_account_keys = (
    legit_keys &
    aml_keys
)

print()
print(
    f"Shared account keys from CURRENT df_features : "
    f"{len(shared_account_keys):,}"
)

# ------------------------------------------------------------
# 8. Map those shared keys to node IDs
# ------------------------------------------------------------

node_ids_for_shared = set()

for key in shared_account_keys:

    if key in feature_engine.node_to_id:

        node_ids_for_shared.add(
            feature_engine.node_to_id[key]
        )

print(
    f"Shared graph node IDs                    : "
    f"{len(node_ids_for_shared):,}"
)

# ------------------------------------------------------------
# 9. Final conclusion
# ------------------------------------------------------------

print()
print("=" * 80)
print("FINAL DIAGNOSTIC")
print("=" * 80)

if (
    raw_mixed_keys == feature_mixed_keys
    and len(node_ids_for_shared) > 0
):

    print(
        "✅ Mixed accounts are genuinely present."
    )

    print(
        "The earlier '0 shared graph nodes' result was caused "
        "by stale/inconsistent graph arrays, not the dataset."
    )

elif (
    len(feature_mixed_keys) == 0
    and len(raw_mixed_keys) > 0
):

    print(
        "⚠️ Raw and feature data disagree."
    )

    print(
        "The feature-engine dataframe has changed account/label "
        "information relative to the raw CSV."
    )

else:

    print(
        "⚠️ Further investigation is required."
    )

print()
print("NO DATASET MODIFICATION WAS PERFORMED.")

RAW CSV ↔ FEATURE GRAPH CONSISTENCY CHECK

DF_FEATURES DIRECT RESULT
--------------------------------------------------------------------------------
Unique graph account keys : 1,213,224
Mixed graph account keys  : 0

RAW CSV DIRECT RESULT
--------------------------------------------------------------------------------
Unique raw account keys    : 843,523
Mixed raw account keys     : 329,321

ACCOUNT KEY SET COMPARISON
Shared keys          : 843,523
Raw-only keys        : 0
Feature-only keys    : 369,701

MIXED-ACCOUNT SET COMPARISON
Raw mixed accounts      : 329,321
Feature mixed accounts  : 0
Mixed keys shared       : 0
Mixed keys raw-only     : 329,321
Mixed keys feature-only : 0

GRAPH NODE-ID CONSISTENCY
Unique src_key → src_id pairs : 594,156
Unique dst_key → dst_id pairs : 1,018,938

Shared account keys from CURRENT df_features : 0
Shared graph node IDs                    : 0

FINAL DIAGNOSTIC
⚠️ Raw and feature data disagree.
The feature-engine dataframe has changed account/la

In [46]:
# ============================================================
# CELL 33 — FIND WHAT CHANGES ACCOUNT VALUES
# ============================================================

import inspect
import pandas as pd
import numpy as np

print("=" * 80)
print("INSPECTING PREPROCESSING BEFORE src_key CREATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. Print first 75 lines of transform()
# ------------------------------------------------------------

source = inspect.getsource(
    AMLFeatureEngine.transform
)

lines = source.splitlines()

print()
print("=" * 80)
print("AMLFeatureEngine.transform() — FIRST 75 LINES")
print("=" * 80)

for i, line in enumerate(lines[:75], start=1):

    print(
        f"{i:03d}: {line}"
    )


# ============================================================
# 2. Compare raw CSV vs feature dataframe by transaction_id
# ============================================================

print()
print("=" * 80)
print("RAW CSV vs df_features — SAME TRANSACTION ID")
print("=" * 80)

compare_columns = [
    "transaction_id",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Is Laundering"
]

raw_compare = aml_data[
    compare_columns
].copy()

feature_compare = df_features[
    compare_columns
].copy()

# ------------------------------------------------------------
# Important:
# df_features is chronologically sorted, so compare by ID,
# not by row position.
# ------------------------------------------------------------

merged = raw_compare.merge(
    feature_compare,
    on="transaction_id",
    how="inner",
    suffixes=("_raw", "_feature")
)

print(
    f"Raw transactions     : {len(raw_compare):,}"
)

print(
    f"Feature transactions : {len(feature_compare):,}"
)

print(
    f"Matched transaction IDs : {len(merged):,}"
)

# ------------------------------------------------------------
# 3. Check whether labels changed
# ------------------------------------------------------------

label_changed = (
    merged["Is Laundering_raw"]
    !=
    merged["Is Laundering_feature"]
)

print()
print(
    f"Label values changed : "
    f"{int(label_changed.sum()):,}"
)

# ------------------------------------------------------------
# 4. Check each account/bank field
# ------------------------------------------------------------

checks = {
    "From Bank": (
        merged["From Bank_raw"].astype(str)
        !=
        merged["From Bank_feature"].astype(str)
    ),

    "Account": (
        merged["Account_raw"].astype(str)
        !=
        merged["Account_feature"].astype(str)
    ),

    "To Bank": (
        merged["To Bank_raw"].astype(str)
        !=
        merged["To Bank_feature"].astype(str)
    ),

    "Account.1": (
        merged["Account.1_raw"].astype(str)
        !=
        merged["Account.1_feature"].astype(str)
    )
}

print()
print("=" * 80)
print("FIELD-LEVEL CHANGES")
print("=" * 80)

for field, changed in checks.items():

    print(
        f"{field:12s} changed : "
        f"{int(changed.sum()):,}"
    )


# ============================================================
# 5. Show actual examples
# ============================================================

any_changed = (
    checks["From Bank"]
    |
    checks["Account"]
    |
    checks["To Bank"]
    |
    checks["Account.1"]
)

changed_examples = merged[
    any_changed
].head(20)

print()
print("=" * 80)
print("FIRST CHANGED TRANSACTIONS")
print("=" * 80)

if len(changed_examples) == 0:

    print(
        "No account/bank fields changed for matched transactions."
    )

else:

    display_columns = [
        "transaction_id",

        "From Bank_raw",
        "From Bank_feature",

        "Account_raw",
        "Account_feature",

        "To Bank_raw",
        "To Bank_feature",

        "Account.1_raw",
        "Account.1_feature",

        "Is Laundering_raw",
        "Is Laundering_feature"
    ]

    print(
        changed_examples[
            display_columns
        ].to_string(index=False)
    )


# ============================================================
# 6. Check whether transaction IDs are duplicated
# ============================================================

print()
print("=" * 80)
print("TRANSACTION ID UNIQUENESS")
print("=" * 80)

raw_duplicate_ids = (
    aml_data["transaction_id"]
    .duplicated()
    .sum()
)

feature_duplicate_ids = (
    df_features["transaction_id"]
    .duplicated()
    .sum()
)

print(
    f"Raw duplicate transaction IDs     : "
    f"{raw_duplicate_ids:,}"
)

print(
    f"Feature duplicate transaction IDs : "
    f"{feature_duplicate_ids:,}"
)


# ============================================================
# 7. Compare unique account-key counts AFTER ALIGNMENT
# ============================================================

raw_src_key = (
    merged["From Bank_raw"].astype(str).str.strip()
    + "_"
    + merged["Account_raw"].astype(str).str.strip()
)

feature_src_key = (
    merged["From Bank_feature"].astype(str).str.strip()
    + "_"
    + merged["Account_feature"].astype(str).str.strip()
)

raw_dst_key = (
    merged["To Bank_raw"].astype(str).str.strip()
    + "_"
    + merged["Account.1_raw"].astype(str).str.strip()
)

feature_dst_key = (
    merged["To Bank_feature"].astype(str).str.strip()
    + "_"
    + merged["Account.1_feature"].astype(str).str.strip()
)

print()
print("=" * 80)
print("UNIQUE KEY COUNTS ON THE SAME TRANSACTIONS")
print("=" * 80)

print(
    f"Raw source keys     : "
    f"{raw_src_key.nunique():,}"
)

print(
    f"Feature source keys : "
    f"{feature_src_key.nunique():,}"
)

print(
    f"Raw destination keys     : "
    f"{raw_dst_key.nunique():,}"
)

print(
    f"Feature destination keys : "
    f"{feature_dst_key.nunique():,}"
)


# ============================================================
# 8. Final message
# ============================================================

print()
print("=" * 80)
print("DIAGNOSTIC COMPLETE")
print("=" * 80)

print(
    "If account fields changed above, the modification happens "
    "inside transform() before src_key/dst_key creation."
)

print(
    "If they did NOT change, then df_features was generated "
    "from a different dataframe state than aml_data."
)

INSPECTING PREPROCESSING BEFORE src_key CREATION

AMLFeatureEngine.transform() — FIRST 75 LINES
001:     def transform(self, df):
002: 
003:         t0 = time.time()
004: 
005:         df = df.copy()
006: 
007:         # ==================================
008:         # 1. BASIC STANDARDIZATION
009:         # ==================================
010: 
011:         df["Timestamp"] = pd.to_datetime(
012:             df["Timestamp"],
013:             errors="coerce"
014:         )
015: 
016:         df["Amount Paid"] = pd.to_numeric(
017:             df["Amount Paid"],
018:             errors="coerce"
019:         ).fillna(0.0)
020: 
021:         df["Amount Received"] = pd.to_numeric(
022:             df["Amount Received"],
023:             errors="coerce"
024:         ).fillna(0.0)
025: 
026:         # ==================================
027:         # 2. CURRENCY NORMALIZATION
028:         # ==================================
029: 
030:         paid_fx = (
031:             df["Payment Curr

In [47]:
# ============================================================
# CELL 34 — RESET WORKING DATA FROM ORIGINAL RAW DATA
# ============================================================
#
# IMPORTANT:
# aml_data = original CSV data
# df       = working dataframe that became inconsistent
#
# We reset df from aml_data and rebuild everything.
# No CSV modification is performed.
# ============================================================

import gc
import time
import numpy as np
import pandas as pd

print("=" * 80)
print("RESETTING WORKING DATA FROM ORIGINAL RAW CSV")
print("=" * 80)

# ------------------------------------------------------------
# 1. Reset df from the untouched raw dataframe
# ------------------------------------------------------------

df = aml_data.copy()

print(
    f"Raw rows copied : {len(df):,}"
)

# ------------------------------------------------------------
# 2. Parse timestamps correctly
# ------------------------------------------------------------

df["Timestamp"] = pd.to_datetime(
    df["Timestamp"],
    format="mixed",
    errors="coerce"
)

print(
    f"Missing timestamps after parsing : "
    f"{df['Timestamp'].isna().sum():,}"
)

assert df["Timestamp"].notna().all()

# ------------------------------------------------------------
# 3. Verify original labels
# ------------------------------------------------------------

label_counts = (
    df["Is Laundering"]
    .value_counts()
    .sort_index()
)

print()
print("RAW LABEL DISTRIBUTION")
print("-" * 80)

print(
    f"Legitimate : "
    f"{int(label_counts.get(0, 0)):,}"
)

print(
    f"Laundering : "
    f"{int(label_counts.get(1, 0)):,}"
)

assert int(label_counts.get(0, 0)) == 5_000_000
assert int(label_counts.get(1, 0)) == 5_000_000

# ------------------------------------------------------------
# 4. Verify true Bank+Account mixed behavior BEFORE transform
# ------------------------------------------------------------

raw_src = (
    df["From Bank"].astype(str).str.strip()
    + "_"
    + df["Account"].astype(str).str.strip()
)

raw_dst = (
    df["To Bank"].astype(str).str.strip()
    + "_"
    + df["Account.1"].astype(str).str.strip()
)

raw_labels = df[
    "Is Laundering"
].to_numpy(dtype=np.int8)

account_check = pd.concat(
    [
        pd.DataFrame({
            "account": raw_src,
            "label": raw_labels
        }),
        pd.DataFrame({
            "account": raw_dst,
            "label": raw_labels
        })
    ],
    ignore_index=True
)

raw_status = (
    account_check
    .groupby("account")["label"]
    .agg(["min", "max"])
)

raw_mixed = (
    (raw_status["min"] == 0)
    &
    (raw_status["max"] == 1)
)

print()
print("=" * 80)
print("RAW DATA ACCOUNT STRUCTURE")
print("=" * 80)

print(
    f"Unique Bank+Account identities : "
    f"{len(raw_status):,}"
)

print(
    f"Mixed Bank+Account identities  : "
    f"{int(raw_mixed.sum()):,}"
)

# ------------------------------------------------------------
# 5. Run feature engine from CLEAN RAW DATA
# ------------------------------------------------------------

print()
print("=" * 80)
print("REBUILDING FEATURES FROM CLEAN RAW DATA")
print("=" * 80)

feature_start = time.time()

feature_engine = AMLFeatureEngine()

(
    df_features,
    X_node,
    E_features,
    edge_feature_names,
    node_feature_names
) = feature_engine.transform(
    df
)

feature_time = (
    time.time()
    - feature_start
)

X_node = np.asarray(
    X_node,
    dtype=np.float32
)

E_features = np.asarray(
    E_features,
    dtype=np.float32
)

print()
print(
    f"Feature generation time : "
    f"{feature_time:.2f} sec"
)

print(
    f"Transactions            : "
    f"{len(df_features):,}"
)

print(
    f"Nodes                   : "
    f"{X_node.shape[0]:,}"
)

print(
    f"Node features           : "
    f"{X_node.shape}"
)

print(
    f"Edge features           : "
    f"{E_features.shape}"
)

# ------------------------------------------------------------
# 6. Verify transaction sorting
# ------------------------------------------------------------

timestamps = df_features[
    "Timestamp"
].to_numpy()

print()
print("=" * 80)
print("POST-TRANSFORM VERIFICATION")
print("=" * 80)

print(
    f"Missing timestamps : "
    f"{pd.isna(timestamps).sum():,}"
)

print(
    f"Minimum timestamp  : "
    f"{df_features['Timestamp'].min()}"
)

print(
    f"Maximum timestamp  : "
    f"{df_features['Timestamp'].max()}"
)

print(
    "Chronologically sorted:",
    np.all(
        timestamps[:-1]
        <=
        timestamps[1:]
    )
)

# ------------------------------------------------------------
# 7. Verify labels after sorting
# ------------------------------------------------------------

feature_labels = df_features[
    "Is Laundering"
].to_numpy(dtype=np.int8)

feature_counts = np.bincount(
    feature_labels,
    minlength=2
)

print()
print("POST-TRANSFORM LABEL DISTRIBUTION")
print("-" * 80)

print(
    f"Legitimate : {feature_counts[0]:,}"
)

print(
    f"Laundering : {feature_counts[1]:,}"
)

assert feature_counts[0] == 5_000_000
assert feature_counts[1] == 5_000_000

# ------------------------------------------------------------
# 8. Check mixed Bank+Account identities AFTER transform
# ------------------------------------------------------------

feature_src = df_features[
    "src_key"
].astype(str)

feature_dst = df_features[
    "dst_key"
].astype(str)

feature_accounts = pd.concat(
    [
        pd.DataFrame({
            "account": feature_src,
            "label": feature_labels
        }),
        pd.DataFrame({
            "account": feature_dst,
            "label": feature_labels
        })
    ],
    ignore_index=True
)

feature_status = (
    feature_accounts
    .groupby("account")["label"]
    .agg(["min", "max"])
)

feature_mixed = (
    (feature_status["min"] == 0)
    &
    (feature_status["max"] == 1)
)

print()
print("=" * 80)
print("POST-TRANSFORM ACCOUNT STRUCTURE")
print("=" * 80)

print(
    f"Unique graph accounts : "
    f"{len(feature_status):,}"
)

print(
    f"Mixed graph accounts  : "
    f"{int(feature_mixed.sum()):,}"
)

# ------------------------------------------------------------
# 9. Feature validity
# ------------------------------------------------------------

print()
print("=" * 80)
print("FEATURE VALIDITY")
print("=" * 80)

print(
    f"Edge NaNs : {np.isnan(E_features).sum():,}"
)

print(
    f"Edge Infs : {np.isinf(E_features).sum():,}"
)

print(
    f"Node NaNs : {np.isnan(X_node).sum():,}"
)

print(
    f"Node Infs : {np.isinf(X_node).sum():,}"
)

# ------------------------------------------------------------
# 10. Final comparison
# ------------------------------------------------------------

print()
print("=" * 80)
print("FINAL RESET CHECK")
print("=" * 80)

print(
    f"Raw mixed accounts       : "
    f"{int(raw_mixed.sum()):,}"
)

print(
    f"Feature mixed accounts   : "
    f"{int(feature_mixed.sum()):,}"
)

if int(raw_mixed.sum()) == int(feature_mixed.sum()):

    print(
        "✅ Raw and feature-engine account structure now agree."
    )

else:

    print(
        "⚠️ Account structure still differs."
    )

print()
print(
    "✅ Working dataframe reset from original raw data."
)

print(
    "✅ No CSV modification performed."
)

RESETTING WORKING DATA FROM ORIGINAL RAW CSV
Raw rows copied : 10,000,000
Missing timestamps after parsing : 0

RAW LABEL DISTRIBUTION
--------------------------------------------------------------------------------
Legitimate : 5,000,000
Laundering : 5,000,000

RAW DATA ACCOUNT STRUCTURE
Unique Bank+Account identities : 843,523
Mixed Bank+Account identities  : 329,321

REBUILDING FEATURES FROM CLEAN RAW DATA
✅ Features generated in 91.99 sec
Transactions : 10,000,000
Nodes        : 843,523
Edge features: (10000000, 20)
Node features: (843523, 13)

Feature generation time : 92.48 sec
Transactions            : 10,000,000
Nodes                   : 843,523
Node features           : (843523, 13)
Edge features           : (10000000, 20)

POST-TRANSFORM VERIFICATION
Missing timestamps : 0
Minimum timestamp  : 2022-09-01 00:00:00
Maximum timestamp  : 2022-09-18 11:18:00
Chronologically sorted: True

POST-TRANSFORM LABEL DISTRIBUTION
------------------------------------------------------------

In [48]:
# ============================================================
# CELL 35 — REBUILD GRAPH + SPLITS FROM CLEAN DATA
# ============================================================

import numpy as np
import torch
import time

print("=" * 80)
print("REBUILDING GRAPH FROM CLEAN FEATURE STATE")
print("=" * 80)

# ------------------------------------------------------------
# 1. Basic dimensions
# ------------------------------------------------------------

num_transactions = len(df_features)

num_nodes = len(
    feature_engine.node_to_id
)

print(
    f"Transactions : {num_transactions:,}"
)

print(
    f"Nodes        : {num_nodes:,}"
)

assert num_transactions == 10_000_000
assert X_node.shape[0] == num_nodes
assert E_features.shape[0] == num_transactions


# ------------------------------------------------------------
# 2. Source / destination node IDs
# ------------------------------------------------------------

src_np = df_features[
    "src_id"
].to_numpy(
    dtype=np.int64,
    copy=False
)

dst_np = df_features[
    "dst_id"
].to_numpy(
    dtype=np.int64,
    copy=False
)

labels_np = df_features[
    "Is Laundering"
].to_numpy(
    dtype=np.int8,
    copy=False
)

timestamps = df_features[
    "Timestamp"
].to_numpy()

# ------------------------------------------------------------
# 3. Sanity checks
# ------------------------------------------------------------

assert src_np.min() >= 0
assert dst_np.min() >= 0

assert src_np.max() < num_nodes
assert dst_np.max() < num_nodes

assert not np.isnan(
    E_features
).any()

assert not np.isnan(
    X_node
).any()

assert not np.isinf(
    E_features
).any()

assert not np.isinf(
    X_node
).any()

print()
print("✅ Node IDs and feature arrays are valid.")


# ------------------------------------------------------------
# 4. Verify chronological ordering
# ------------------------------------------------------------

is_chronological = np.all(
    timestamps[:-1] <= timestamps[1:]
)

print(
    f"Chronologically sorted : {is_chronological}"
)

assert is_chronological


# ------------------------------------------------------------
# 5. Create 70 / 15 / 15 chronological split
# ------------------------------------------------------------

TRAIN_END = int(
    num_transactions * 0.70
)

VAL_END = int(
    num_transactions * 0.85
)

train_idx = torch.arange(
    0,
    TRAIN_END,
    dtype=torch.long
)

val_idx = torch.arange(
    TRAIN_END,
    VAL_END,
    dtype=torch.long
)

test_idx = torch.arange(
    VAL_END,
    num_transactions,
    dtype=torch.long
)

print()
print("=" * 80)
print("CHRONOLOGICAL SPLIT")
print("=" * 80)

print(
    f"Train      : {len(train_idx):,}"
)

print(
    f"Validation : {len(val_idx):,}"
)

print(
    f"Test       : {len(test_idx):,}"
)


# ------------------------------------------------------------
# 6. Class distribution
# ------------------------------------------------------------

def show_split_distribution(
    name,
    indices
):

    ids = indices.numpy()

    y = labels_np[
        ids
    ]

    legitimate = int(
        np.sum(y == 0)
    )

    laundering = int(
        np.sum(y == 1)
    )

    total = len(y)

    print()
    print(name)
    print("-" * 80)

    print(
        f"Total        : {total:,}"
    )

    print(
        f"Legitimate   : "
        f"{legitimate:,} "
        f"({legitimate / total:.2%})"
    )

    print(
        f"Laundering   : "
        f"{laundering:,} "
        f"({laundering / total:.2%})"
    )


show_split_distribution(
    "TRAIN",
    train_idx
)

show_split_distribution(
    "VALIDATION",
    val_idx
)

show_split_distribution(
    "TEST",
    test_idx
)


# ------------------------------------------------------------
# 7. Time range per split
# ------------------------------------------------------------

print()
print("=" * 80)
print("SPLIT TIME RANGES")
print("=" * 80)

for name, indices in [
    ("TRAIN", train_idx),
    ("VALIDATION", val_idx),
    ("TEST", test_idx)
]:

    ids = indices.numpy()

    print(
        f"{name:12s}: "
        f"{timestamps[ids[0]]} "
        f"--> "
        f"{timestamps[ids[-1]]}"
    )


# ------------------------------------------------------------
# 8. Original directed transaction graph
# ------------------------------------------------------------

print()
print("=" * 80)
print("BUILDING TRANSACTION GRAPH")
print("=" * 80)

edge_index = torch.from_numpy(
    np.vstack([
        src_np,
        dst_np
    ])
).long()

print(
    f"Original edge_index : "
    f"{tuple(edge_index.shape)}"
)


# ------------------------------------------------------------
# 9. Bidirectional message-passing graph
# ------------------------------------------------------------

message_edge_index = torch.cat(
    [
        edge_index,
        edge_index.flip(0)
    ],
    dim=1
)

print(
    f"Message edge_index  : "
    f"{tuple(message_edge_index.shape)}"
)

print(
    f"Transaction edges   : "
    f"{edge_index.shape[1]:,}"
)

print(
    f"Message edges       : "
    f"{message_edge_index.shape[1]:,}"
)


# ------------------------------------------------------------
# 10. Train-only adjacency
# ------------------------------------------------------------

print()
print("=" * 80)
print("BUILDING TRAIN-ONLY CSR ADJACENCY")
print("=" * 80)

TRAIN_EDGE_IDS = train_idx.numpy()

train_src = src_np[
    TRAIN_EDGE_IDS
]

train_dst = dst_np[
    TRAIN_EDGE_IDS
]

# Bidirectional references
adj_nodes = np.concatenate(
    [
        train_src,
        train_dst
    ]
)

adj_neighbors = np.concatenate(
    [
        train_dst,
        train_src
    ]
)

adj_edge_pos = np.concatenate(
    [
        TRAIN_EDGE_IDS,
        TRAIN_EDGE_IDS
    ]
)

# Sort by source node
sort_order = np.argsort(
    adj_nodes,
    kind="stable"
)

adj_nodes = adj_nodes[
    sort_order
]

TRAIN_ADJ_NEIGHBORS = (
    adj_neighbors[
        sort_order
    ]
)

TRAIN_ADJ_EDGE_POS = (
    adj_edge_pos[
        sort_order
    ]
)

# ------------------------------------------------------------
# CSR pointers
# ------------------------------------------------------------

TRAIN_ADJ_PTR = np.zeros(
    num_nodes + 1,
    dtype=np.int64
)

np.add.at(
    TRAIN_ADJ_PTR,
    adj_nodes + 1,
    1
)

TRAIN_ADJ_PTR = np.cumsum(
    TRAIN_ADJ_PTR
)

# ------------------------------------------------------------
# 11. Adjacency summary
# ------------------------------------------------------------

print(
    f"Training edges           : "
    f"{len(TRAIN_EDGE_IDS):,}"
)

print(
    f"Adjacency references     : "
    f"{len(TRAIN_ADJ_NEIGHBORS):,}"
)

print(
    f"Active source nodes     : "
    f"{np.count_nonzero(np.diff(TRAIN_ADJ_PTR)):,}"
)

print(
    f"CSR pointers            : "
    f"{TRAIN_ADJ_PTR.shape}"
)


# ------------------------------------------------------------
# 12. Verify mixed accounts in graph
# ------------------------------------------------------------

legit_nodes = set(
    src_np[labels_np == 0]
) | set(
    dst_np[labels_np == 0]
)

aml_nodes = set(
    src_np[labels_np == 1]
) | set(
    dst_np[labels_np == 1]
)

shared_nodes = (
    legit_nodes &
    aml_nodes
)

print()
print("=" * 80)
print("GRAPH ACCOUNT SHARING")
print("=" * 80)

print(
    f"Legitimate nodes : "
    f"{len(legit_nodes):,}"
)

print(
    f"AML nodes        : "
    f"{len(aml_nodes):,}"
)

print(
    f"Shared nodes     : "
    f"{len(shared_nodes):,}"
)


# ------------------------------------------------------------
# 13. Save clean numerical feature cache
# ------------------------------------------------------------

FEATURE_CACHE = os.path.join(
    DATA_DIR,
    "gat_aml_clean_features.npz"
)

np.savez(
    FEATURE_CACHE,
    X_node=X_node,
    E_features=E_features
)

print()
print(
    f"Feature cache saved : "
    f"{FEATURE_CACHE}"
)


# ------------------------------------------------------------
# 14. Final assertions
# ------------------------------------------------------------

assert (
    len(train_idx)
    + len(val_idx)
    + len(test_idx)
    == num_transactions
)

assert len(TRAIN_EDGE_IDS) == 7_000_000

assert (
    len(TRAIN_ADJ_NEIGHBORS)
    == 14_000_000
)

assert (
    len(TRAIN_ADJ_EDGE_POS)
    == 14_000_000
)

assert (
    len(TRAIN_ADJ_PTR)
    == num_nodes + 1
)

print()
print("=" * 80)
print("✅ CLEAN GRAPH SETUP COMPLETE")
print("=" * 80)

print(
    f"Nodes              : {num_nodes:,}"
)

print(
    f"Transactions       : {num_transactions:,}"
)

print(
    f"Train              : {len(train_idx):,}"
)

print(
    f"Validation         : {len(val_idx):,}"
)

print(
    f"Test               : {len(test_idx):,}"
)

print(
    f"Shared graph nodes : {len(shared_nodes):,}"
)

REBUILDING GRAPH FROM CLEAN FEATURE STATE
Transactions : 10,000,000
Nodes        : 843,523

✅ Node IDs and feature arrays are valid.
Chronologically sorted : True

CHRONOLOGICAL SPLIT
Train      : 7,000,000
Validation : 1,500,000
Test       : 1,500,000

TRAIN
--------------------------------------------------------------------------------
Total        : 7,000,000
Legitimate   : 3,904,126 (55.77%)
Laundering   : 3,095,874 (44.23%)

VALIDATION
--------------------------------------------------------------------------------
Total        : 1,500,000
Legitimate   : 818,893 (54.59%)
Laundering   : 681,107 (45.41%)

TEST
--------------------------------------------------------------------------------
Total        : 1,500,000
Legitimate   : 276,981 (18.47%)
Laundering   : 1,223,019 (81.53%)

SPLIT TIME RANGES
TRAIN       : 2022-09-01T00:00:00.000000000 --> 2022-09-08T11:11:00.000000000
VALIDATION  : 2022-09-08T11:11:00.000000000 --> 2022-09-09T21:15:00.000000000
TEST        : 2022-09-09T21:15:

In [49]:
# ============================================================
# CELL 36 — REBUILD FAST SAMPLER FOR CLEAN GRAPH
# ============================================================

import numpy as np
import torch
import time

print("=" * 80)
print("REBUILDING FAST NEIGHBOR SAMPLER")
print("=" * 80)

# ------------------------------------------------------------
# Make sure the sampler uses the CURRENT clean graph
# ------------------------------------------------------------

TRAIN_EDGE_IDS = train_idx.cpu().numpy()

print(
    f"Training transaction IDs : "
    f"{len(TRAIN_EDGE_IDS):,}"
)

print(
    f"Number of graph nodes    : "
    f"{num_nodes:,}"
)

# ------------------------------------------------------------
# Fast neighbor sampling
# ------------------------------------------------------------

def sample_train_neighbors_fast(
    node_ids,
    num_neighbors=8,
    rng=None
):

    if rng is None:
        rng = np.random.default_rng(SEED)

    node_ids = np.asarray(
        node_ids,
        dtype=np.int64
    )

    if len(node_ids) == 0:
        return (
            np.empty(0, dtype=np.int64),
            np.empty(0, dtype=np.int64),
            np.empty(0, dtype=np.int64)
        )

    starts = TRAIN_ADJ_PTR[
        node_ids
    ]

    ends = TRAIN_ADJ_PTR[
        node_ids + 1
    ]

    degrees = (
        ends - starts
    )

    take = np.minimum(
        degrees,
        num_neighbors
    )

    active = (
        take > 0
    )

    if not np.any(active):

        return (
            np.empty(0, dtype=np.int64),
            np.empty(0, dtype=np.int64),
            np.empty(0, dtype=np.int64)
        )

    active_nodes = node_ids[
        active
    ]

    active_starts = starts[
        active
    ]

    active_take = take[
        active
    ]

    # --------------------------------------------------------
    # Random offset into each node's adjacency list
    # --------------------------------------------------------

    random_offset = np.zeros(
        len(active_nodes),
        dtype=np.int64
    )

    multi = (
        active_take > 1
    )

    random_offset[multi] = (
        rng.random(
            np.sum(multi)
        )
        * (
            (
                TRAIN_ADJ_PTR[
                    active_nodes[multi] + 1
                ]
                -
                TRAIN_ADJ_PTR[
                    active_nodes[multi]
                ]
                -
                active_take[multi]
                + 1
            )
        )
    ).astype(np.int64)

    selected_starts = (
        active_starts
        + random_offset
    )

    # --------------------------------------------------------
    # Build positions for contiguous sampled windows
    # --------------------------------------------------------

    total_selected = int(
        active_take.sum()
    )

    repeated_starts = np.repeat(
        selected_starts,
        active_take
    )

    repeated_nodes = np.repeat(
        active_nodes,
        active_take
    )

    cumulative = np.cumsum(
        active_take
    )

    group_start = np.repeat(
        np.r_[0, cumulative[:-1]],
        active_take
    )

    local_offsets = (
        np.arange(total_selected)
        - group_start
    )

    positions = (
        repeated_starts
        + local_offsets
    )

    neighbors = (
        TRAIN_ADJ_NEIGHBORS[
            positions
        ]
    )

    edge_positions = (
        TRAIN_ADJ_EDGE_POS[
            positions
        ]
    )

    return (
        repeated_nodes,
        neighbors,
        edge_positions
    )


# ============================================================
# TWO-HOP SUBGRAPH SAMPLER
# ============================================================

def sample_train_subgraph_fast(
    target_edge_ids,
    num_neighbors_1=8,
    num_neighbors_2=8,
    seed=42
):

    target_edge_ids = np.asarray(
        target_edge_ids,
        dtype=np.int64
    )

    rng = np.random.default_rng(
        seed
    )

    # --------------------------------------------------------
    # Target endpoints
    # --------------------------------------------------------

    target_src = src_np[
        target_edge_ids
    ]

    target_dst = dst_np[
        target_edge_ids
    ]

    seed_nodes = np.unique(
        np.concatenate(
            [
                target_src,
                target_dst
            ]
        )
    )

    # --------------------------------------------------------
    # Hop 1
    # --------------------------------------------------------

    hop1_src, hop1_dst, hop1_edges = (
        sample_train_neighbors_fast(
            seed_nodes,
            num_neighbors=num_neighbors_1,
            rng=rng
        )
    )

    hop1_nodes = (
        np.unique(
            np.concatenate(
                [
                    seed_nodes,
                    hop1_dst
                ]
            )
        )
    )

    # --------------------------------------------------------
    # Hop 2
    # --------------------------------------------------------

    hop2_src, hop2_dst, hop2_edges = (
        sample_train_neighbors_fast(
            hop1_nodes,
            num_neighbors=num_neighbors_2,
            rng=rng
        )
    )

    # --------------------------------------------------------
    # All local nodes
    # --------------------------------------------------------

    local_nodes = np.unique(
        np.concatenate(
            [
                seed_nodes,
                hop1_dst,
                hop2_dst
            ]
        )
    )

    # --------------------------------------------------------
    # Convert sampled edge positions -> original
    # transaction IDs
    # --------------------------------------------------------

    sampled_train_positions = np.concatenate(
        [
            hop1_edges,
            hop2_edges,
            target_edge_ids
        ]
    )

    sampled_edge_ids = np.unique(
        sampled_train_positions
    )

    # --------------------------------------------------------
    # Local node mapping using sorted node IDs
    # --------------------------------------------------------

    def to_local(global_ids):

        return np.searchsorted(
            local_nodes,
            global_ids
        )

    # --------------------------------------------------------
    # Sampled message-passing edges
    # --------------------------------------------------------

    sampled_src = src_np[
        sampled_edge_ids
    ]

    sampled_dst = dst_np[
        sampled_edge_ids
    ]

    local_src = to_local(
        sampled_src
    )

    local_dst = to_local(
        sampled_dst
    )

    # Bidirectional message passing
    message_src = np.concatenate(
        [
            local_src,
            local_dst
        ]
    )

    message_dst = np.concatenate(
        [
            local_dst,
            local_src
        ]
    )

    message_edge_index = np.vstack(
        [
            message_src,
            message_dst
        ]
    ).astype(
        np.int64,
        copy=False
    )

    message_edge_ids = np.concatenate(
        [
            sampled_edge_ids,
            sampled_edge_ids
        ]
    )

    # --------------------------------------------------------
    # Target edge index
    # --------------------------------------------------------

    target_local_src = to_local(
        target_src
    )

    target_local_dst = to_local(
        target_dst
    )

    target_edge_index = np.vstack(
        [
            target_local_src,
            target_local_dst
        ]
    ).astype(
        np.int64,
        copy=False
    )

    return (
        local_nodes,
        message_edge_index,
        message_edge_ids,
        target_edge_index
    )


# ============================================================
# BATCH BUILDER
# ============================================================

def build_training_batch_fast(
    target_edge_ids,
    seed=42
):

    target_edge_ids = np.asarray(
        target_edge_ids,
        dtype=np.int64
    )

    (
        local_nodes,
        message_edge_index,
        message_edge_ids,
        target_edge_index
    ) = sample_train_subgraph_fast(
        target_edge_ids,
        num_neighbors_1=8,
        num_neighbors_2=8,
        seed=seed
    )

    # --------------------------------------------------------
    # Local tensors
    # --------------------------------------------------------

    x = torch.from_numpy(
        X_node[
            local_nodes
        ]
    ).float()

    edge_index = torch.from_numpy(
        message_edge_index
    ).long()

    edge_attr = torch.from_numpy(
        E_features[
            message_edge_ids
        ]
    ).float()

    target_edge_index = torch.from_numpy(
        target_edge_index
    ).long()

    target_edge_attr = torch.from_numpy(
        E_features[
            target_edge_ids
        ]
    ).float()

    y = torch.from_numpy(
        labels_np[
            target_edge_ids
        ]
    ).float()

    return {
        "x": x,
        "edge_index": edge_index,
        "edge_attr": edge_attr,
        "target_edge_index": target_edge_index,
        "target_edge_attr": target_edge_attr,
        "y": y,
        "local_nodes": local_nodes
    }


# ============================================================
# SANITY TEST
# ============================================================

print()
print("=" * 80)
print("SAMPLER SANITY TEST")
print("=" * 80)

test_batch_size = 8192

rng = np.random.default_rng(
    SEED
)

test_target_ids = rng.choice(
    TRAIN_EDGE_IDS,
    size=test_batch_size,
    replace=False
)

start_time = time.time()

test_batch = build_training_batch_fast(
    test_target_ids,
    seed=SEED
)

elapsed = (
    time.time()
    - start_time
)

print(
    f"Target transactions : "
    f"{test_batch_size:,}"
)

print(
    f"Local nodes         : "
    f"{test_batch['x'].shape[0]:,}"
)

print(
    f"Message edges       : "
    f"{test_batch['edge_index'].shape[1]:,}"
)

print(
    f"Message edge attrs  : "
    f"{tuple(test_batch['edge_attr'].shape)}"
)

print(
    f"Target edges        : "
    f"{test_batch['target_edge_index'].shape[1]:,}"
)

print(
    f"Target edge attrs   : "
    f"{tuple(test_batch['target_edge_attr'].shape)}"
)

print(
    f"Labels              : "
    f"{test_batch['y'].shape}"
)

print(
    f"Build time          : "
    f"{elapsed:.4f} sec"
)

# ------------------------------------------------------------
# Sanity assertions
# ------------------------------------------------------------

assert test_batch["x"].shape[1] == 13

assert (
    test_batch["edge_attr"].shape[1]
    == 20
)

assert (
    test_batch["target_edge_attr"].shape[1]
    == 20
)

assert (
    test_batch["target_edge_index"].shape[1]
    == test_batch_size
)

assert (
    len(test_batch["y"])
    == test_batch_size
)

assert (
    test_batch["edge_index"].shape[1]
    ==
    test_batch["edge_attr"].shape[0]
)

assert (
    test_batch["target_edge_index"].shape[1]
    ==
    test_batch["target_edge_attr"].shape[0]
)

# Verify target labels are actually mixed
unique, counts = np.unique(
    test_batch["y"].numpy(),
    return_counts=True
)

print()
print("Target batch labels:")

for cls, count in zip(
    unique,
    counts
):

    name = (
        "Legitimate"
        if int(cls) == 0
        else "Laundering"
    )

    print(
        f"  {name:12s}: {int(count):,}"
    )

print()
print("=" * 80)
print("✅ CLEAN-GRAPH SAMPLER READY")
print("=" * 80)

REBUILDING FAST NEIGHBOR SAMPLER
Training transaction IDs : 7,000,000
Number of graph nodes    : 843,523

SAMPLER SANITY TEST
Target transactions : 8,192
Local nodes         : 262,722
Message edges       : 1,081,758
Message edge attrs  : (1081758, 20)
Target edges        : 8,192
Target edge attrs   : (8192, 20)
Labels              : torch.Size([8192])
Build time          : 0.1937 sec

Target batch labels:
  Legitimate  : 4,482
  Laundering  : 3,710

✅ CLEAN-GRAPH SAMPLER READY


In [50]:
# ============================================================
# CELL 37 — FRESH GATv2 TRAINING ON CLEAN GRAPH
# ============================================================

import os
import gc
import time
import numpy as np
import torch
import torch.nn as nn
from tqdm.auto import tqdm

print("=" * 80)
print("FRESH GATv2 AML TRAINING — CLEAN GRAPH")
print("=" * 80)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

SEED = 42

EPOCHS = 3
TRAIN_BATCH_SIZE = 8192

NEIGHBORS_1 = 8
NEIGHBORS_2 = 8

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

DROPOUT = 0.20
HIDDEN_DIM = 64
HEADS = 4

CHECKPOINT_PATH = os.path.join(
    DATA_DIR,
    "gat_aml_stage1.pt"
)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.set_float32_matmul_precision("high")

# ------------------------------------------------------------
# Fresh model
# ------------------------------------------------------------

model = GATAMLModel(
    node_in_dim=13,
    edge_in_dim=20,
    hidden_dim=HIDDEN_DIM,
    heads=HEADS,
    dropout=DROPOUT
).to(DEVICE)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Device             : {DEVICE}")
print(f"Node input dim     : 13")
print(f"Edge input dim     : 20")
print(f"Hidden dimension   : {HIDDEN_DIM}")
print(f"Attention heads    : {HEADS}")
print(f"Dropout            : {DROPOUT}")
print(f"Trainable params   : {trainable_params:,}")
print()

# ------------------------------------------------------------
# Optimizer + loss
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

criterion = nn.BCEWithLogitsLoss()

use_amp = DEVICE.type == "cuda"

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp
)

# ------------------------------------------------------------
# Training transaction IDs
# ------------------------------------------------------------

TRAIN_IDS = train_idx.cpu().numpy()

num_train = len(TRAIN_IDS)

num_batches = (
    num_train + TRAIN_BATCH_SIZE - 1
) // TRAIN_BATCH_SIZE

print(
    f"Training transactions : {num_train:,}"
)

print(
    f"Batch size            : {TRAIN_BATCH_SIZE:,}"
)

print(
    f"Batches / epoch       : {num_batches:,}"
)

print(
    f"Neighbors             : "
    f"{NEIGHBORS_1} + {NEIGHBORS_2}"
)

# ------------------------------------------------------------
# Verify training distribution
# ------------------------------------------------------------

train_y = labels_np[
    TRAIN_IDS
]

print()
print("TRAINING LABEL DISTRIBUTION")
print("-" * 80)

print(
    f"Legitimate : "
    f"{np.sum(train_y == 0):,}"
)

print(
    f"Laundering : "
    f"{np.sum(train_y == 1):,}"
)

# ------------------------------------------------------------
# GPU batch preparation
# ------------------------------------------------------------

def prepare_gpu_batch_clean(
    target_ids,
    seed
):

    batch = build_training_batch_fast(
        target_ids,
        seed=seed
    )

    x = batch["x"].to(
        DEVICE,
        non_blocking=True
    )

    edge_index = batch["edge_index"].to(
        DEVICE,
        non_blocking=True
    )

    edge_attr = batch["edge_attr"].to(
        DEVICE,
        non_blocking=True
    )

    target_edge_index = batch[
        "target_edge_index"
    ].to(
        DEVICE,
        non_blocking=True
    )

    target_edge_attr = batch[
        "target_edge_attr"
    ].to(
        DEVICE,
        non_blocking=True
    )

    y = batch["y"].to(
        DEVICE,
        non_blocking=True
    ).float()

    return (
        x,
        edge_index,
        edge_attr,
        target_edge_index,
        target_edge_attr,
        y
    )


# ============================================================
# TRAIN
# ============================================================

history = []

total_start = time.time()

for epoch in range(
    1,
    EPOCHS + 1
):

    epoch_start = time.time()

    model.train()

    # --------------------------------------------------------
    # Shuffle target transactions
    # --------------------------------------------------------

    rng = np.random.default_rng(
        SEED + epoch
    )

    shuffled_ids = TRAIN_IDS.copy()

    rng.shuffle(
        shuffled_ids
    )

    running_loss = 0.0

    print()
    print("=" * 80)
    print(
        f"EPOCH {epoch}/{EPOCHS}"
    )
    print("=" * 80)

    progress = tqdm(
        range(
            0,
            num_train,
            TRAIN_BATCH_SIZE
        ),
        total=num_batches,
        desc=f"Epoch {epoch}"
    )

    for batch_no, start in enumerate(
        progress,
        start=1
    ):

        target_ids = shuffled_ids[
            start:start + TRAIN_BATCH_SIZE
        ]

        # ----------------------------------------------------
        # Build local graph
        # ----------------------------------------------------

        (
            x,
            edge_index,
            edge_attr,
            target_edge_index,
            target_edge_attr,
            y
        ) = prepare_gpu_batch_clean(
            target_ids,
            seed=(
                SEED
                + epoch * 100000
                + batch_no
            )
        )

        # ----------------------------------------------------
        # Gradient reset
        # ----------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )

        # ----------------------------------------------------
        # Forward pass
        # ----------------------------------------------------

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=use_amp
        ):

            logits = model(
                x,
                edge_index,
                edge_attr,
                target_edge_index,
                target_edge_attr
            )

            loss = criterion(
                logits.view(-1),
                y.view(-1)
            )

        # ----------------------------------------------------
        # Backward
        # ----------------------------------------------------

        scaler.scale(
            loss
        ).backward()

        scaler.step(
            optimizer
        )

        scaler.update()

        # ----------------------------------------------------
        # Running loss
        # ----------------------------------------------------

        batch_n = len(target_ids)

        running_loss += (
            loss.item()
            * batch_n
        )

        average_loss = (
            running_loss
            /
            (
                start + batch_n
            )
        )

        progress.set_postfix(
            loss=f"{average_loss:.8f}"
        )

        # ----------------------------------------------------
        # Cleanup
        # ----------------------------------------------------

        del (
            x,
            edge_index,
            edge_attr,
            target_edge_index,
            target_edge_attr,
            y,
            logits,
            loss
        )

    # --------------------------------------------------------
    # Epoch result
    # --------------------------------------------------------

    epoch_loss = (
        running_loss
        /
        num_train
    )

    epoch_time = (
        time.time()
        - epoch_start
    )

    history.append({
        "epoch": epoch,
        "loss": float(epoch_loss),
        "time_sec": float(epoch_time)
    })

    print()
    print(
        f"Epoch {epoch} complete"
    )

    print(
        f"Average loss : "
        f"{epoch_loss:.8f}"
    )

    print(
        f"Time         : "
        f"{epoch_time:.2f} sec"
    )

    # --------------------------------------------------------
    # Save checkpoint
    # --------------------------------------------------------

    checkpoint = {
        "model_state": model.state_dict(),

        "model_config": {
            "node_in_dim": 13,
            "edge_in_dim": 20,
            "hidden_dim": HIDDEN_DIM,
            "heads": HEADS,
            "dropout": DROPOUT
        },

        "edge_feature_names": edge_feature_names,
        "node_feature_names": node_feature_names,

        "num_nodes": int(num_nodes),
        "num_transactions": int(num_transactions),

        "train_transactions": int(num_train),

        "epoch": int(epoch),
        "batch_size": int(TRAIN_BATCH_SIZE),

        "neighbors": {
            "hop1": int(NEIGHBORS_1),
            "hop2": int(NEIGHBORS_2)
        },

        "learning_rate": float(
            LEARNING_RATE
        ),

        "weight_decay": float(
            WEIGHT_DECAY
        ),

        "seed": int(SEED),

        "dataset": {
            "nodes": int(num_nodes),
            "transactions": int(num_transactions),
            "train": int(len(train_idx)),
            "validation": int(len(val_idx)),
            "test": int(len(test_idx))
        },

        "history": history
    }

    torch.save(
        checkpoint,
        CHECKPOINT_PATH
    )

    print(
        f"Checkpoint saved: "
        f"{CHECKPOINT_PATH}"
    )

    # --------------------------------------------------------
    # GPU memory
    # --------------------------------------------------------

    if DEVICE.type == "cuda":

        allocated_gb = (
            torch.cuda.memory_allocated(
                DEVICE
            )
            / 1024**3
        )

        reserved_gb = (
            torch.cuda.memory_reserved(
                DEVICE
            )
            / 1024**3
        )

        print(
            f"GPU allocated : "
            f"{allocated_gb:.3f} GB"
        )

        print(
            f"GPU reserved  : "
            f"{reserved_gb:.3f} GB"
        )

    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


# ============================================================
# FINAL TRAINING SUMMARY
# ============================================================

total_time = (
    time.time()
    - total_start
)

print()
print("=" * 80)
print("TRAINING COMPLETE")
print("=" * 80)

for item in history:

    print(
        f"Epoch {item['epoch']} | "
        f"Loss={item['loss']:.8f} | "
        f"Time={item['time_sec']:.2f}s"
    )

print()
print(
    f"Total training time : "
    f"{total_time:.2f} sec"
)

print(
    f"Final checkpoint    : "
    f"{CHECKPOINT_PATH}"
)

print()
print("✅ Model trained on the corrected 843,523-node graph.")
print("✅ Mixed-behavior accounts are now represented in the graph.")

FRESH GATv2 AML TRAINING — CLEAN GRAPH
Device             : cuda
Node input dim     : 13
Edge input dim     : 20
Hidden dimension   : 64
Attention heads    : 4
Dropout            : 0.2
Trainable params   : 32,385

Training transactions : 7,000,000
Batch size            : 8,192
Batches / epoch       : 855
Neighbors             : 8 + 8

TRAINING LABEL DISTRIBUTION
--------------------------------------------------------------------------------
Legitimate : 3,904,126
Laundering : 3,095,874

EPOCH 1/3


Epoch 1:   0%|          | 0/855 [00:00<?, ?it/s]


Epoch 1 complete
Average loss : 0.04019928
Time         : 677.45 sec
Checkpoint saved: /home/llyods-aids5/AML/gat_aml_stage1.pt
GPU allocated : 0.024 GB
GPU reserved  : 5.289 GB

EPOCH 2/3


Epoch 2:   0%|          | 0/855 [00:00<?, ?it/s]


Epoch 2 complete
Average loss : 0.01639517
Time         : 673.39 sec
Checkpoint saved: /home/llyods-aids5/AML/gat_aml_stage1.pt
GPU allocated : 0.024 GB
GPU reserved  : 5.643 GB

EPOCH 3/3


Epoch 3:   0%|          | 0/855 [00:00<?, ?it/s]


Epoch 3 complete
Average loss : 0.01331772
Time         : 680.09 sec
Checkpoint saved: /home/llyods-aids5/AML/gat_aml_stage1.pt
GPU allocated : 0.024 GB
GPU reserved  : 6.254 GB

TRAINING COMPLETE
Epoch 1 | Loss=0.04019928 | Time=677.45s
Epoch 2 | Loss=0.01639517 | Time=673.39s
Epoch 3 | Loss=0.01331772 | Time=680.09s

Total training time : 2033.30 sec
Final checkpoint    : /home/llyods-aids5/AML/gat_aml_stage1.pt

✅ Model trained on the corrected 843,523-node graph.
✅ Mixed-behavior accounts are now represented in the graph.


In [51]:
# ============================================================
# CELL 38 — CLEAN VALIDATION + TEST EVALUATION
# ============================================================
#
# Important:
# The target transaction itself is NOT included in the
# message-passing graph.
#
# The GAT only receives:
#   - target source/destination nodes
#   - sampled TRAIN neighborhood
#   - target transaction's own 20 edge features
#
# This avoids using the target edge twice.
# ============================================================

import os
import json
import time
import numpy as np
import torch

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from tqdm.auto import tqdm


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

EVAL_BATCH_SIZE = 8192
EVAL_THRESHOLD = 0.50

VAL_IDS = val_idx.cpu().numpy()
TEST_IDS = test_idx.cpu().numpy()

print("=" * 80)
print("CLEAN GATv2 VALIDATION + TEST EVALUATION")
print("=" * 80)

print(
    f"Validation transactions : {len(VAL_IDS):,}"
)

print(
    f"Test transactions       : {len(TEST_IDS):,}"
)

print(
    f"Threshold               : {EVAL_THRESHOLD}"
)

print(
    f"Device                  : {DEVICE}"
)


# ============================================================
# CLEAN EVALUATION SUBGRAPH SAMPLER
# ============================================================

def sample_eval_subgraph_fast(
    target_edge_ids,
    num_neighbors_1=8,
    num_neighbors_2=8,
    seed=42
):

    target_edge_ids = np.asarray(
        target_edge_ids,
        dtype=np.int64
    )

    rng = np.random.default_rng(
        seed
    )

    # --------------------------------------------------------
    # Target endpoints
    # --------------------------------------------------------

    target_src = src_np[
        target_edge_ids
    ]

    target_dst = dst_np[
        target_edge_ids
    ]

    seed_nodes = np.unique(
        np.concatenate(
            [
                target_src,
                target_dst
            ]
        )
    )

    # --------------------------------------------------------
    # Hop 1
    # --------------------------------------------------------

    hop1_src, hop1_dst, hop1_edges = (
        sample_train_neighbors_fast(
            seed_nodes,
            num_neighbors=num_neighbors_1,
            rng=rng
        )
    )

    hop1_nodes = np.unique(
        np.concatenate(
            [
                seed_nodes,
                hop1_dst
            ]
        )
    )

    # --------------------------------------------------------
    # Hop 2
    # --------------------------------------------------------

    hop2_src, hop2_dst, hop2_edges = (
        sample_train_neighbors_fast(
            hop1_nodes,
            num_neighbors=num_neighbors_2,
            rng=rng
        )
    )

    # --------------------------------------------------------
    # IMPORTANT:
    # Only neighborhood edges are used for message passing.
    # Target transaction IDs are NOT added here.
    # --------------------------------------------------------

    sampled_edge_ids = np.unique(
        np.concatenate(
            [
                hop1_edges,
                hop2_edges
            ]
        )
    )

    # --------------------------------------------------------
    # Local node set
    # --------------------------------------------------------

    local_nodes = np.unique(
        np.concatenate(
            [
                seed_nodes,
                hop1_dst,
                hop2_dst
            ]
        )
    )

    # --------------------------------------------------------
    # Global → local mapping
    # --------------------------------------------------------

    def to_local(global_ids):

        return np.searchsorted(
            local_nodes,
            global_ids
        )

    # --------------------------------------------------------
    # Message-passing edges
    # --------------------------------------------------------

    sampled_src = src_np[
        sampled_edge_ids
    ]

    sampled_dst = dst_np[
        sampled_edge_ids
    ]

    local_src = to_local(
        sampled_src
    )

    local_dst = to_local(
        sampled_dst
    )

    message_src = np.concatenate(
        [
            local_src,
            local_dst
        ]
    )

    message_dst = np.concatenate(
        [
            local_dst,
            local_src
        ]
    )

    message_edge_index = np.vstack(
        [
            message_src,
            message_dst
        ]
    ).astype(
        np.int64,
        copy=False
    )

    message_edge_ids = np.concatenate(
        [
            sampled_edge_ids,
            sampled_edge_ids
        ]
    )

    # --------------------------------------------------------
    # Target edge
    # --------------------------------------------------------

    target_local_src = to_local(
        target_src
    )

    target_local_dst = to_local(
        target_dst
    )

    target_edge_index = np.vstack(
        [
            target_local_src,
            target_local_dst
        ]
    ).astype(
        np.int64,
        copy=False
    )

    return (
        local_nodes,
        message_edge_index,
        message_edge_ids,
        target_edge_index
    )


# ============================================================
# CLEAN EVALUATION BATCH
# ============================================================

def build_eval_batch_clean(
    target_edge_ids,
    seed=42
):

    target_edge_ids = np.asarray(
        target_edge_ids,
        dtype=np.int64
    )

    (
        local_nodes,
        message_edge_index,
        message_edge_ids,
        target_edge_index
    ) = sample_eval_subgraph_fast(
        target_edge_ids,
        num_neighbors_1=8,
        num_neighbors_2=8,
        seed=seed
    )

    x = torch.from_numpy(
        X_node[
            local_nodes
        ]
    ).float()

    edge_index = torch.from_numpy(
        message_edge_index
    ).long()

    edge_attr = torch.from_numpy(
        E_features[
            message_edge_ids
        ]
    ).float()

    target_edge_index = torch.from_numpy(
        target_edge_index
    ).long()

    target_edge_attr = torch.from_numpy(
        E_features[
            target_edge_ids
        ]
    ).float()

    y = torch.from_numpy(
        labels_np[
            target_edge_ids
        ]
    ).float()

    return {
        "x": x,
        "edge_index": edge_index,
        "edge_attr": edge_attr,
        "target_edge_index": target_edge_index,
        "target_edge_attr": target_edge_attr,
        "y": y
    }


# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate_clean(
    edge_ids,
    split_name
):

    model.eval()

    all_probs = []

    start_time = time.time()

    num_batches = (
        len(edge_ids)
        + EVAL_BATCH_SIZE
        - 1
    ) // EVAL_BATCH_SIZE

    print()
    print("=" * 80)
    print(f"EVALUATING {split_name}")
    print("=" * 80)

    with torch.inference_mode():

        for batch_no, start in enumerate(
            tqdm(
                range(
                    0,
                    len(edge_ids),
                    EVAL_BATCH_SIZE
                ),
                total=num_batches,
                desc=split_name
            ),
            start=1
        ):

            target_ids = edge_ids[
                start:start + EVAL_BATCH_SIZE
            ]

            batch = build_eval_batch_clean(
                target_ids,
                seed=SEED + batch_no
            )

            x = batch["x"].to(
                DEVICE,
                non_blocking=True
            )

            edge_index = batch[
                "edge_index"
            ].to(
                DEVICE,
                non_blocking=True
            )

            edge_attr = batch[
                "edge_attr"
            ].to(
                DEVICE,
                non_blocking=True
            )

            target_edge_index = batch[
                "target_edge_index"
            ].to(
                DEVICE,
                non_blocking=True
            )

            target_edge_attr = batch[
                "target_edge_attr"
            ].to(
                DEVICE,
                non_blocking=True
            )

            # ------------------------------------------------
            # Forward pass
            # ------------------------------------------------

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(DEVICE.type == "cuda")
            ):

                logits = model(
                    x,
                    edge_index,
                    edge_attr,
                    target_edge_index,
                    target_edge_attr
                )

            probs = torch.sigmoid(
                logits
            ).float().cpu().numpy().reshape(-1)

            all_probs.append(
                probs
            )

            del (
                x,
                edge_index,
                edge_attr,
                target_edge_index,
                target_edge_attr,
                logits,
                probs
            )

    # --------------------------------------------------------
    # Predictions + labels
    # --------------------------------------------------------

    y_prob = np.concatenate(
        all_probs
    )

    y_true = labels_np[
        edge_ids
    ].astype(np.int64)

    assert len(y_true) == len(y_prob)

    y_pred = (
        y_prob >= EVAL_THRESHOLD
    ).astype(np.int64)

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_true,
        y_prob
    )

    pr_auc = average_precision_score(
        y_true,
        y_prob
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    elapsed = (
        time.time()
        - start_time
    )

    # --------------------------------------------------------
    # Results
    # --------------------------------------------------------

    print()
    print("=" * 80)
    print(f"{split_name} RESULTS")
    print("=" * 80)

    print(
        f"Precision : {precision:.6f}"
    )

    print(
        f"Recall    : {recall:.6f}"
    )

    print(
        f"F1 Score  : {f1:.6f}"
    )

    print(
        f"ROC-AUC   : {roc_auc:.6f}"
    )

    print(
        f"PR-AUC    : {pr_auc:.6f}"
    )

    print()
    print("CONFUSION MATRIX")
    print("-" * 80)

    print(
        f"True Negatives  : {tn:,}"
    )

    print(
        f"False Positives : {fp:,}"
    )

    print(
        f"False Negatives : {fn:,}"
    )

    print(
        f"True Positives   : {tp:,}"
    )

    print()
    print("CLASS-WISE METRICS")
    print("-" * 80)

    print(
        classification_report(
            y_true,
            y_pred,
            labels=[0, 1],
            target_names=[
                "Legitimate",
                "Laundering"
            ],
            digits=6,
            zero_division=0
        )
    )

    # --------------------------------------------------------
    # AML workload
    # --------------------------------------------------------

    alerts = tp + fp
    total = len(y_true)

    print("AML ALERT STATISTICS")
    print("-" * 80)

    print(
        f"Total transactions : {total:,}"
    )

    print(
        f"Alerts generated   : {alerts:,}"
    )

    print(
        f"Alert rate         : "
        f"{alerts / total:.4%}"
    )

    if tp > 0:

        print(
            f"Transactions reviewed / TP : "
            f"{alerts / tp:.2f}"
        )

    print(
        f"Evaluation time    : "
        f"{elapsed:.2f} sec"
    )

    return {
        "y_true": y_true,
        "y_prob": y_prob,
        "y_pred": y_pred,
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "confusion_matrix": cm
    }


# ============================================================
# VALIDATION
# ============================================================

val_results = evaluate_clean(
    VAL_IDS,
    "VALIDATION"
)


# ============================================================
# TEST
# ============================================================

test_results = evaluate_clean(
    TEST_IDS,
    "TEST"
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 80)
print("FINAL CLEAN-GRAPH RESULTS")
print("=" * 80)

print()
print("VALIDATION")
print(
    f"Precision : {val_results['precision']:.6f}"
)
print(
    f"Recall    : {val_results['recall']:.6f}"
)
print(
    f"F1        : {val_results['f1']:.6f}"
)
print(
    f"ROC-AUC   : {val_results['roc_auc']:.6f}"
)
print(
    f"PR-AUC    : {val_results['pr_auc']:.6f}"
)

print()
print("TEST")
print(
    f"Precision : {test_results['precision']:.6f}"
)
print(
    f"Recall    : {test_results['recall']:.6f}"
)
print(
    f"F1        : {test_results['f1']:.6f}"
)
print(
    f"ROC-AUC   : {test_results['roc_auc']:.6f}"
)
print(
    f"PR-AUC    : {test_results['pr_auc']:.6f}"
)


# ============================================================
# SAVE RESULTS
# ============================================================

evaluation_file = os.path.join(
    DATA_DIR,
    "gat_aml_clean_evaluation.json"
)

summary = {
    "threshold": EVAL_THRESHOLD,

    "validation": {
        "precision": val_results["precision"],
        "recall": val_results["recall"],
        "f1": val_results["f1"],
        "roc_auc": val_results["roc_auc"],
        "pr_auc": val_results["pr_auc"],
        "confusion_matrix":
            val_results[
                "confusion_matrix"
            ].tolist()
    },

    "test": {
        "precision": test_results["precision"],
        "recall": test_results["recall"],
        "f1": test_results["f1"],
        "roc_auc": test_results["roc_auc"],
        "pr_auc": test_results["pr_auc"],
        "confusion_matrix":
            test_results[
                "confusion_matrix"
            ].tolist()
    }
}

with open(
    evaluation_file,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

print()
print(
    f"✅ Saved evaluation to: "
    f"{evaluation_file}"
)

CLEAN GATv2 VALIDATION + TEST EVALUATION
Validation transactions : 1,500,000
Test transactions       : 1,500,000
Threshold               : 0.5
Device                  : cuda

EVALUATING VALIDATION


VALIDATION:   0%|          | 0/184 [00:00<?, ?it/s]


VALIDATION RESULTS
Precision : 0.998214
Recall    : 0.993841
F1 Score  : 0.996023
ROC-AUC   : 0.999955
PR-AUC    : 0.999943

CONFUSION MATRIX
--------------------------------------------------------------------------------
True Negatives  : 817,682
False Positives : 1,211
False Negatives : 4,195
True Positives   : 676,912

CLASS-WISE METRICS
--------------------------------------------------------------------------------
              precision    recall  f1-score   support

  Legitimate   0.994896  0.998521  0.996705    818893
  Laundering   0.998214  0.993841  0.996023    681107

    accuracy                       0.996396   1500000
   macro avg   0.996555  0.996181  0.996364   1500000
weighted avg   0.996403  0.996396  0.996395   1500000

AML ALERT STATISTICS
--------------------------------------------------------------------------------
Total transactions : 1,500,000
Alerts generated   : 678,123
Alert rate         : 45.2082%
Transactions reviewed / TP : 1.00
Evaluation time    : 

TEST:   0%|          | 0/184 [00:00<?, ?it/s]


TEST RESULTS
Precision : 0.998879
Recall    : 0.998455
F1 Score  : 0.998667
ROC-AUC   : 0.999892
PR-AUC    : 0.999970

CONFUSION MATRIX
--------------------------------------------------------------------------------
True Negatives  : 275,610
False Positives : 1,371
False Negatives : 1,890
True Positives   : 1,221,129

CLASS-WISE METRICS
--------------------------------------------------------------------------------
              precision    recall  f1-score   support

  Legitimate   0.993189  0.995050  0.994119    276981
  Laundering   0.998879  0.998455  0.998667   1223019

    accuracy                       0.997826   1500000
   macro avg   0.996034  0.996752  0.996393   1500000
weighted avg   0.997828  0.997826  0.997827   1500000

AML ALERT STATISTICS
--------------------------------------------------------------------------------
Total transactions : 1,500,000
Alerts generated   : 1,222,500
Alert rate         : 81.5000%
Transactions reviewed / TP : 1.00
Evaluation time    : 55

In [52]:
# ============================================================
# CELL 39 — VALIDATION THRESHOLD OPTIMIZATION
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("=" * 80)
print("VALIDATION THRESHOLD ANALYSIS")
print("=" * 80)

# ------------------------------------------------------------
# Validation predictions
# ------------------------------------------------------------

y_val = val_results["y_true"]
p_val = val_results["y_prob"]

# ------------------------------------------------------------
# Precision / Recall curve
# ------------------------------------------------------------

precision_curve, recall_curve, thresholds = (
    precision_recall_curve(
        y_val,
        p_val
    )
)

# precision_curve and recall_curve are one element longer
# than thresholds, so use[:-1] for threshold-level metrics.

threshold_table = pd.DataFrame({
    "threshold": thresholds,
    "precision": precision_curve[:-1],
    "recall": recall_curve[:-1]
})

threshold_table["f1"] = (
    2
    * threshold_table["precision"]
    * threshold_table["recall"]
    /
    (
        threshold_table["precision"]
        + threshold_table["recall"]
        + 1e-12
    )
)

# ------------------------------------------------------------
# Alert count / workload for every threshold
# ------------------------------------------------------------

threshold_table["alerts"] = (
    threshold_table["threshold"]
    .apply(
        lambda t:
        int(np.sum(p_val >= t))
    )
)

threshold_table["alert_rate"] = (
    threshold_table["alerts"]
    / len(y_val)
)

# ------------------------------------------------------------
# Operating points
# ------------------------------------------------------------

print()
print("=" * 80)
print("SELECTED VALIDATION OPERATING POINTS")
print("=" * 80)

# ------------------------------------------------------------
# 1. Max F1
# ------------------------------------------------------------

max_f1_idx = (
    threshold_table["f1"]
    .idxmax()
)

max_f1_row = (
    threshold_table
    .loc[max_f1_idx]
)

print()
print("MAXIMUM VALIDATION F1")
print("-" * 80)

print(
    f"Threshold  : "
    f"{max_f1_row['threshold']:.10f}"
)

print(
    f"Precision  : "
    f"{max_f1_row['precision']:.6f}"
)

print(
    f"Recall     : "
    f"{max_f1_row['recall']:.6f}"
)

print(
    f"F1         : "
    f"{max_f1_row['f1']:.6f}"
)

print(
    f"Alerts     : "
    f"{int(max_f1_row['alerts']):,}"
)

print(
    f"Alert rate : "
    f"{max_f1_row['alert_rate']:.4%}"
)


# ------------------------------------------------------------
# 2. Highest recall with precision >= 99%
# ------------------------------------------------------------

precision_constraint = (
    threshold_table[
        threshold_table["precision"] >= 0.99
    ]
)

if len(precision_constraint) > 0:

    high_precision_idx = (
        precision_constraint["recall"]
        .idxmax()
    )

    high_precision_row = (
        threshold_table
        .loc[high_precision_idx]
    )

    print()
    print(
        "MAXIMUM RECALL WITH PRECISION >= 99%"
    )
    print("-" * 80)

    print(
        f"Threshold  : "
        f"{high_precision_row['threshold']:.10f}"
    )

    print(
        f"Precision  : "
        f"{high_precision_row['precision']:.6f}"
    )

    print(
        f"Recall     : "
        f"{high_precision_row['recall']:.6f}"
    )

    print(
        f"F1         : "
        f"{high_precision_row['f1']:.6f}"
    )

    print(
        f"Alerts     : "
        f"{int(high_precision_row['alerts']):,}"
    )

    print(
        f"Alert rate : "
        f"{high_precision_row['alert_rate']:.4%}"
    )

else:

    high_precision_row = None

    print(
        "\nNo validation threshold achieved "
        "precision >= 99%."
    )


# ------------------------------------------------------------
# 3. Common alert-budget thresholds
# ------------------------------------------------------------

print()
print("=" * 80)
print("ALERT-BUDGET ANALYSIS — VALIDATION")
print("=" * 80)

for percentage in [
    0.1,
    0.5,
    1.0,
    2.0,
    5.0,
    10.0,
    20.0
]:

    k = max(
        1,
        int(
            len(y_val)
            * percentage
            / 100.0
        )
    )

    sorted_probs = np.argsort(
        p_val
    )[::-1]

    selected = sorted_probs[
        :k
    ]

    y_selected = y_val[
        selected
    ]

    tp = int(
        np.sum(y_selected == 1)
    )

    fp = int(
        np.sum(y_selected == 0)
    )

    actual_positive = int(
        np.sum(y_val == 1)
    )

    precision = (
        tp / (tp + fp)
        if tp + fp > 0
        else 0.0
    )

    recall = (
        tp / actual_positive
        if actual_positive > 0
        else 0.0
    )

    threshold = (
        p_val[
            sorted_probs[k - 1]
        ]
    )

    print(
        f"Top {percentage:>5.1f}% | "
        f"Threshold={threshold:.8f} | "
        f"Alerts={k:,} | "
        f"Precision={precision:.6f} | "
        f"Recall={recall:.6f}"
    )


# ------------------------------------------------------------
# 4. Apply validation-selected threshold to TEST
# ------------------------------------------------------------

if high_precision_row is not None:

    selected_threshold = float(
        high_precision_row["threshold"]
    )

    y_test = test_results["y_true"]
    p_test = test_results["y_prob"]

    y_test_pred = (
        p_test >= selected_threshold
    ).astype(np.int8)

    test_precision = precision_score(
        y_test,
        y_test_pred,
        zero_division=0
    )

    test_recall = recall_score(
        y_test,
        y_test_pred,
        zero_division=0
    )

    test_f1 = f1_score(
        y_test,
        y_test_pred,
        zero_division=0
    )

    test_cm = confusion_matrix(
        y_test,
        y_test_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = (
        test_cm.ravel()
    )

    test_alerts = tp + fp

    print()
    print("=" * 80)
    print("TEST PERFORMANCE USING VALIDATION-SELECTED THRESHOLD")
    print("=" * 80)

    print(
        f"Selected threshold : "
        f"{selected_threshold:.10f}"
    )

    print(
        f"Precision          : "
        f"{test_precision:.6f}"
    )

    print(
        f"Recall             : "
        f"{test_recall:.6f}"
    )

    print(
        f"F1                 : "
        f"{test_f1:.6f}"
    )

    print()
    print("CONFUSION MATRIX")
    print("-" * 80)

    print(
        f"TN : {tn:,}"
    )

    print(
        f"FP : {fp:,}"
    )

    print(
        f"FN : {fn:,}"
    )

    print(
        f"TP : {tp:,}"
    )

    print()
    print("ALERT WORKLOAD")
    print("-" * 80)

    print(
        f"Alerts generated : "
        f"{test_alerts:,}"
    )

    print(
        f"Alert rate       : "
        f"{test_alerts / len(y_test):.4%}"
    )

else:

    selected_threshold = 0.50


# ============================================================
# SAVE THRESHOLD INFORMATION
# ============================================================

threshold_file = os.path.join(
    DATA_DIR,
    "gat_aml_threshold_analysis.json"
)

threshold_summary = {
    "selection_rule":
        "Maximum validation recall subject to precision >= 99%",

    "selected_threshold":
        float(selected_threshold),

    "validation_default_0.5": {
        "precision":
            float(
                val_results["precision"]
            ),
        "recall":
            float(
                val_results["recall"]
            ),
        "f1":
            float(
                val_results["f1"]
            )
    },

    "test_at_selected_threshold": {
        "precision":
            float(test_precision)
            if high_precision_row is not None
            else None,

        "recall":
            float(test_recall)
            if high_precision_row is not None
            else None,

        "f1":
            float(test_f1)
            if high_precision_row is not None
            else None,

        "confusion_matrix":
            test_cm.tolist()
            if high_precision_row is not None
            else None
    }
}

import json

with open(
    threshold_file,
    "w"
) as f:

    json.dump(
        threshold_summary,
        f,
        indent=2
    )

print()
print(
    f"✅ Threshold analysis saved to: "
    f"{threshold_file}"
)

VALIDATION THRESHOLD ANALYSIS

SELECTED VALIDATION OPERATING POINTS

MAXIMUM VALIDATION F1
--------------------------------------------------------------------------------
Threshold  : 0.0520324707
Precision  : 0.996437
Recall     : 0.999505
F1         : 0.997969
Alerts     : 683,204
Alert rate : 45.5469%

MAXIMUM RECALL WITH PRECISION >= 99%
--------------------------------------------------------------------------------
Threshold  : 0.0018100739
Precision  : 0.990001
Recall     : 0.999950
F1         : 0.994951
Alerts     : 687,952
Alert rate : 45.8635%

ALERT-BUDGET ANALYSIS — VALIDATION
Top   0.1% | Threshold=1.00000000 | Alerts=1,500 | Precision=1.000000 | Recall=0.002202
Top   0.5% | Threshold=1.00000000 | Alerts=7,500 | Precision=1.000000 | Recall=0.011011
Top   1.0% | Threshold=1.00000000 | Alerts=15,000 | Precision=1.000000 | Recall=0.022023
Top   2.0% | Threshold=1.00000000 | Alerts=30,000 | Precision=1.000000 | Recall=0.044046
Top   5.0% | Threshold=1.00000000 | Alerts=75,000

In [53]:
# ============================================================
# CELL 41 — FINAL TRAINING-SET METRICS
# ============================================================

import time
import numpy as np
import torch

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from tqdm.auto import tqdm


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

TRAIN_EVAL_BATCH_SIZE = 8192
TRAIN_EVAL_THRESHOLD = 0.50

TRAIN_EVAL_IDS = train_idx.cpu().numpy()

print("=" * 80)
print("FINAL TRAINING-SET EVALUATION")
print("=" * 80)

print(
    f"Training transactions : "
    f"{len(TRAIN_EVAL_IDS):,}"
)

print(
    f"Batch size            : "
    f"{TRAIN_EVAL_BATCH_SIZE:,}"
)

print(
    f"Threshold             : "
    f"{TRAIN_EVAL_THRESHOLD}"
)

print(
    f"Device                : "
    f"{DEVICE}"
)


# ============================================================
# EVALUATE FINAL TRAINING MODEL
# ============================================================

model.eval()

all_train_probs = []

start_time = time.time()

num_batches = (
    len(TRAIN_EVAL_IDS)
    + TRAIN_EVAL_BATCH_SIZE
    - 1
) // TRAIN_EVAL_BATCH_SIZE

with torch.inference_mode():

    for batch_no, start in enumerate(
        tqdm(
            range(
                0,
                len(TRAIN_EVAL_IDS),
                TRAIN_EVAL_BATCH_SIZE
            ),
            total=num_batches,
            desc="TRAIN EVAL"
        ),
        start=1
    ):

        target_ids = TRAIN_EVAL_IDS[
            start:start + TRAIN_EVAL_BATCH_SIZE
        ]

        # ----------------------------------------------------
        # Build evaluation graph
        # Target transaction is excluded from message passing
        # ----------------------------------------------------

        batch = build_eval_batch_clean(
            target_ids,
            seed=SEED + batch_no
        )

        x = batch["x"].to(
            DEVICE,
            non_blocking=True
        )

        edge_index = batch[
            "edge_index"
        ].to(
            DEVICE,
            non_blocking=True
        )

        edge_attr = batch[
            "edge_attr"
        ].to(
            DEVICE,
            non_blocking=True
        )

        target_edge_index = batch[
            "target_edge_index"
        ].to(
            DEVICE,
            non_blocking=True
        )

        target_edge_attr = batch[
            "target_edge_attr"
        ].to(
            DEVICE,
            non_blocking=True
        )

        # ----------------------------------------------------
        # Full FP32 inference
        # ----------------------------------------------------

        logits = model(
            x,
            edge_index,
            edge_attr,
            target_edge_index,
            target_edge_attr
        )

        probs = torch.sigmoid(
            logits.float()
        )

        all_train_probs.append(
            probs.cpu().numpy().reshape(-1)
        )

        del (
            x,
            edge_index,
            edge_attr,
            target_edge_index,
            target_edge_attr,
            logits,
            probs
        )


# ============================================================
# COLLECT RESULTS
# ============================================================

train_y_prob = np.concatenate(
    all_train_probs
)

train_y_true = labels_np[
    TRAIN_EVAL_IDS
].astype(
    np.int64
)

assert len(train_y_prob) == len(
    train_y_true
)

train_y_pred = (
    train_y_prob
    >= TRAIN_EVAL_THRESHOLD
).astype(
    np.int8
)


# ============================================================
# METRICS
# ============================================================

train_precision = precision_score(
    train_y_true,
    train_y_pred,
    zero_division=0
)

train_recall = recall_score(
    train_y_true,
    train_y_pred,
    zero_division=0
)

train_f1 = f1_score(
    train_y_true,
    train_y_pred,
    zero_division=0
)

train_accuracy = accuracy_score(
    train_y_true,
    train_y_pred
)

train_roc_auc = roc_auc_score(
    train_y_true,
    train_y_prob
)

train_pr_auc = average_precision_score(
    train_y_true,
    train_y_prob
)

train_cm = confusion_matrix(
    train_y_true,
    train_y_pred,
    labels=[0, 1]
)

tn, fp, fn, tp = train_cm.ravel()


# ============================================================
# PRINT RESULTS
# ============================================================

elapsed = (
    time.time()
    - start_time
)

print()
print("=" * 80)
print("TRAINING RESULTS")
print("=" * 80)

print(
    f"Precision : {train_precision:.6f}"
)

print(
    f"Recall    : {train_recall:.6f}"
)

print(
    f"F1 Score  : {train_f1:.6f}"
)

print(
    f"Accuracy  : {train_accuracy:.6f}"
)

print(
    f"ROC-AUC   : {train_roc_auc:.6f}"
)

print(
    f"PR-AUC    : {train_pr_auc:.6f}"
)

print()
print("CONFUSION MATRIX")
print("-" * 80)

print(
    f"True Negatives  : {tn:,}"
)

print(
    f"False Positives : {fp:,}"
)

print(
    f"False Negatives : {fn:,}"
)

print(
    f"True Positives   : {tp:,}"
)


# ============================================================
# CLASS-WISE METRICS
# ============================================================

print()
print("CLASS-WISE METRICS")
print("-" * 80)

print(
    classification_report(
        train_y_true,
        train_y_pred,
        labels=[0, 1],
        target_names=[
            "Legitimate",
            "Laundering"
        ],
        digits=6,
        zero_division=0
    )
)


# ============================================================
# ALERT WORKLOAD
# ============================================================

alerts = tp + fp

print()
print("AML ALERT STATISTICS")
print("-" * 80)

print(
    f"Total transactions : "
    f"{len(train_y_true):,}"
)

print(
    f"Alerts generated   : "
    f"{alerts:,}"
)

print(
    f"Alert rate         : "
    f"{alerts / len(train_y_true):.4%}"
)

if tp > 0:

    print(
        f"Transactions reviewed / TP : "
        f"{alerts / tp:.2f}"
    )


# ============================================================
# PROBABILITY DISTRIBUTION
# ============================================================

print()
print("TRAINING PROBABILITY DISTRIBUTION")
print("-" * 80)

print(
    f"Minimum : "
    f"{train_y_prob.min():.12f}"
)

print(
    f"1%      : "
    f"{np.percentile(train_y_prob, 1):.12f}"
)

print(
    f"10%     : "
    f"{np.percentile(train_y_prob, 10):.12f}"
)

print(
    f"50%     : "
    f"{np.percentile(train_y_prob, 50):.12f}"
)

print(
    f"90%     : "
    f"{np.percentile(train_y_prob, 90):.12f}"
)

print(
    f"99%     : "
    f"{np.percentile(train_y_prob, 99):.12f}"
)

print(
    f"Maximum : "
    f"{train_y_prob.max():.12f}"
)

print()
print(
    f"Evaluation time : {elapsed:.2f} sec"
)

print()
print("=" * 80)
print("✅ FINAL TRAINING METRICS COMPLETE")
print("=" * 80)

FINAL TRAINING-SET EVALUATION
Training transactions : 7,000,000
Batch size            : 8,192
Threshold             : 0.5
Device                : cuda


TRAIN EVAL:   0%|          | 0/855 [00:00<?, ?it/s]


TRAINING RESULTS
Precision : 0.994872
Recall    : 0.997565
F1 Score  : 0.996217
Accuracy  : 0.996649
ROC-AUC   : 0.999917
PR-AUC    : 0.999893

CONFUSION MATRIX
--------------------------------------------------------------------------------
True Negatives  : 3,888,208
False Positives : 15,918
False Negatives : 7,539
True Positives   : 3,088,335

CLASS-WISE METRICS
--------------------------------------------------------------------------------
              precision    recall  f1-score   support

  Legitimate   0.998065  0.995923  0.996993   3904126
  Laundering   0.994872  0.997565  0.996217   3095874

    accuracy                       0.996649   7000000
   macro avg   0.996469  0.996744  0.996605   7000000
weighted avg   0.996653  0.996649  0.996649   7000000


AML ALERT STATISTICS
--------------------------------------------------------------------------------
Total transactions : 7,000,000
Alerts generated   : 3,104,253
Alert rate         : 44.3465%
Transactions reviewed / TP :

In [54]:
# ============================================================
# CELL 42 — TRAIN vs VALIDATION vs TEST SUMMARY
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 80)

comparison = pd.DataFrame({
    "Metric": [
        "Precision",
        "Recall",
        "F1 Score",
        "Accuracy",
        "ROC-AUC",
        "PR-AUC",
        "Alert Rate"
    ],

    "Train": [
        train_precision,
        train_recall,
        train_f1,
        train_accuracy,
        train_roc_auc,
        train_pr_auc,
        np.mean(train_y_pred)
    ],

    "Validation": [
        val_results["precision"],
        val_results["recall"],
        val_results["f1"],
        np.mean(
            val_results["y_true"]
            ==
            val_results["y_pred"]
        ),
        val_results["roc_auc"],
        val_results["pr_auc"],
        np.mean(val_results["y_pred"])
    ],

    "Test": [
        test_results["precision"],
        test_results["recall"],
        test_results["f1"],
        np.mean(
            test_results["y_true"]
            ==
            test_results["y_pred"]
        ),
        test_results["roc_auc"],
        test_results["pr_auc"],
        np.mean(test_results["y_pred"])
    ]
})

# Convert to percentages for human-readable output
percentage_metrics = {
    "Precision",
    "Recall",
    "F1 Score",
    "Accuracy",
    "Alert Rate"
}

display_table = comparison.copy()

for metric in percentage_metrics:

    mask = (
        display_table["Metric"]
        == metric
    )

    display_table.loc[
        mask,
        ["Train", "Validation", "Test"]
    ] *= 100

print()

print(
    display_table.to_string(
        index=False,
        formatters={
            "Train": lambda x: f"{x:.4f}",
            "Validation": lambda x: f"{x:.4f}",
            "Test": lambda x: f"{x:.4f}"
        }
    )
)

# ------------------------------------------------------------
# Generalization gaps
# ------------------------------------------------------------

print()
print("=" * 80)
print("GENERALIZATION GAP")
print("=" * 80)

train_val_f1_gap = (
    train_f1
    -
    val_results["f1"]
)

train_test_f1_gap = (
    train_f1
    -
    test_results["f1"]
)

print(
    f"Train → Validation F1 gap : "
    f"{train_val_f1_gap:.6f}"
)

print(
    f"Train → Test F1 gap       : "
    f"{train_test_f1_gap:.6f}"
)

print()
print("=" * 80)
print("INTERPRETATION")
print("=" * 80)

if abs(train_val_f1_gap) < 0.02:

    print(
        "✅ No large train/validation F1 gap is visible."
    )

else:

    print(
        "⚠️ There is a noticeable train/validation F1 gap."
    )

print()
print(
    "Important: these metrics are strong benchmark results, "
    "but the current node aggregates are computed globally."
)

print(
    "The next methodological step is causal temporal features "
    "before treating the result as production-style AML performance."
)

MODEL PERFORMANCE COMPARISON

    Metric   Train Validation    Test
 Precision 99.4872    99.8214 99.8879
    Recall 99.7565    99.3841 99.8455
  F1 Score 99.6217    99.6023 99.8667
  Accuracy 99.6649    99.6396 99.7826
   ROC-AUC  0.9999     1.0000  0.9999
    PR-AUC  0.9999     0.9999  1.0000
Alert Rate 44.3465    45.2082 81.5000

GENERALIZATION GAP
Train → Validation F1 gap : 0.000194
Train → Test F1 gap       : -0.002450

INTERPRETATION
✅ No large train/validation F1 gap is visible.

Important: these metrics are strong benchmark results, but the current node aggregates are computed globally.
The next methodological step is causal temporal features before treating the result as production-style AML performance.


In [55]:
# ============================================================
# CELL 43 — FULL-FP32 SCORE EVALUATION
# ============================================================

import os
import time
import numpy as np
import torch

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

from tqdm.auto import tqdm

EVAL_BATCH_SIZE = 8192

def evaluate_fp32_scores(edge_ids, split_name):

    model.eval()

    probabilities = []

    start_time = time.time()

    for batch_no, start in enumerate(
        tqdm(
            range(
                0,
                len(edge_ids),
                EVAL_BATCH_SIZE
            ),
            desc=split_name
        ),
        start=1
    ):

        target_ids = edge_ids[
            start:start + EVAL_BATCH_SIZE
        ]

        batch = build_eval_batch_clean(
            target_ids,
            seed=SEED + batch_no
        )

        x = batch["x"].to(
            DEVICE,
            non_blocking=True
        )

        edge_index = batch[
            "edge_index"
        ].to(
            DEVICE,
            non_blocking=True
        )

        edge_attr = batch[
            "edge_attr"
        ].to(
            DEVICE,
            non_blocking=True
        )

        target_edge_index = batch[
            "target_edge_index"
        ].to(
            DEVICE,
            non_blocking=True
        )

        target_edge_attr = batch[
            "target_edge_attr"
        ].to(
            DEVICE,
            non_blocking=True
        )

        # ----------------------------------------------------
        # FULL FP32
        # ----------------------------------------------------

        with torch.inference_mode():

            logits = model(
                x,
                edge_index,
                edge_attr,
                target_edge_index,
                target_edge_attr
            )

            probs = torch.sigmoid(
                logits.float()
            )

        probabilities.append(
            probs.cpu().numpy().reshape(-1)
        )

        del (
            x,
            edge_index,
            edge_attr,
            target_edge_index,
            target_edge_attr,
            logits,
            probs
        )

    y_prob = np.concatenate(
        probabilities
    )

    y_true = labels_np[
        edge_ids
    ].astype(np.int8)

    auc = roc_auc_score(
        y_true,
        y_prob
    )

    pr_auc = average_precision_score(
        y_true,
        y_prob
    )

    unique_probs = np.unique(
        y_prob
    )

    exact_one = np.sum(
        y_prob == 1.0
    )

    exact_zero = np.sum(
        y_prob == 0.0
    )

    print()
    print("=" * 80)
    print(f"{split_name} — FP32 RESULTS")
    print("=" * 80)

    print(
        f"ROC-AUC : {auc:.8f}"
    )

    print(
        f"PR-AUC  : {pr_auc:.8f}"
    )

    print()
    print("PROBABILITY RESOLUTION")
    print("-" * 80)

    print(
        f"Total predictions      : {len(y_prob):,}"
    )

    print(
        f"Unique probabilities   : {len(unique_probs):,}"
    )

    print(
        f"Exactly 0.0            : {exact_zero:,}"
    )

    print(
        f"Exactly 1.0            : {exact_one:,}"
    )

    print()
    print("PROBABILITY PERCENTILES")
    print("-" * 80)

    for q in [
        0,
        1,
        5,
        10,
        25,
        50,
        75,
        90,
        95,
        99,
        99.9,
        100
    ]:

        print(
            f"{q:>5.1f}% : "
            f"{np.percentile(y_prob, q):.12f}"
        )

    print()
    print(
        f"Evaluation time : "
        f"{time.time() - start_time:.2f} sec"
    )

    return {
        "y_true": y_true,
        "y_prob": y_prob,
        "roc_auc": float(auc),
        "pr_auc": float(pr_auc)
    }


# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

val_results_fp32 = evaluate_fp32_scores(
    VAL_IDS,
    "VALIDATION"
)

# ------------------------------------------------------------
# TEST
# ------------------------------------------------------------

test_results_fp32 = evaluate_fp32_scores(
    TEST_IDS,
    "TEST"
)

# ------------------------------------------------------------
# SAVE RAW SCORE ARRAYS
# ------------------------------------------------------------

score_file = os.path.join(
    DATA_DIR,
    "gat_aml_fp32_scores.npz"
)

np.savez(
    score_file,
    val_y_true=val_results_fp32["y_true"],
    val_y_prob=val_results_fp32["y_prob"],
    test_y_true=test_results_fp32["y_true"],
    test_y_prob=test_results_fp32["y_prob"]
)

print()
print("=" * 80)
print("✅ FP32 SCORE EVALUATION COMPLETE")
print("=" * 80)

print(
    f"Saved: {score_file}"
)

VALIDATION:   0%|          | 0/184 [00:00<?, ?it/s]


VALIDATION — FP32 RESULTS
ROC-AUC : 0.99995566
PR-AUC  : 0.99994496

PROBABILITY RESOLUTION
--------------------------------------------------------------------------------
Total predictions      : 1,500,000
Unique probabilities   : 868,668
Exactly 0.0            : 0
Exactly 1.0            : 16,971

PROBABILITY PERCENTILES
--------------------------------------------------------------------------------
  0.0% : 0.000000000715
  1.0% : 0.000000006239
  5.0% : 0.000000016292
 10.0% : 0.000000024941
 25.0% : 0.000000077699
 50.0% : 0.000002255946
 75.0% : 0.999917149544
 90.0% : 0.999997735023
 95.0% : 0.999999642372
 99.0% : 1.000000000000
 99.9% : 1.000000000000
100.0% : 1.000000000000

Evaluation time : 51.82 sec


TEST:   0%|          | 0/184 [00:00<?, ?it/s]


TEST — FP32 RESULTS
ROC-AUC : 0.99990419
PR-AUC  : 0.99997756

PROBABILITY RESOLUTION
--------------------------------------------------------------------------------
Total predictions      : 1,500,000
Unique probabilities   : 359,725
Exactly 0.0            : 0
Exactly 1.0            : 138,487

PROBABILITY PERCENTILES
--------------------------------------------------------------------------------
  0.0% : 0.000000000874
  1.0% : 0.000000018369
  5.0% : 0.000000081637
 10.0% : 0.000000258063
 25.0% : 0.997937560081
 50.0% : 0.999987721443
 75.0% : 0.999999165535
 90.0% : 0.999999880791
 95.0% : 1.000000000000
 99.0% : 1.000000000000
 99.9% : 1.000000000000
100.0% : 1.000000000000

Evaluation time : 60.26 sec

✅ FP32 SCORE EVALUATION COMPLETE
Saved: /home/llyods-aids5/AML/gat_aml_fp32_scores.npz


In [56]:
# ============================================================
# CELL 44 — SAVE RAW LOGITS FOR PROPER ALERT RANKING
# ============================================================

import os
import time
import numpy as np
import torch

from tqdm.auto import tqdm
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

LOGIT_BATCH_SIZE = 8192


# ============================================================
# LOGIT EVALUATION FUNCTION
# ============================================================

def evaluate_raw_logits(
    edge_ids,
    split_name
):

    model.eval()

    all_logits = []

    start_time = time.time()

    num_batches = (
        len(edge_ids)
        + LOGIT_BATCH_SIZE
        - 1
    ) // LOGIT_BATCH_SIZE

    print()
    print("=" * 80)
    print(f"RAW LOGIT EVALUATION — {split_name}")
    print("=" * 80)

    with torch.inference_mode():

        for batch_no, start in enumerate(
            tqdm(
                range(
                    0,
                    len(edge_ids),
                    LOGIT_BATCH_SIZE
                ),
                total=num_batches,
                desc=split_name
            ),
            start=1
        ):

            target_ids = edge_ids[
                start:start + LOGIT_BATCH_SIZE
            ]

            batch = build_eval_batch_clean(
                target_ids,
                seed=SEED + batch_no
            )

            x = batch["x"].to(
                DEVICE,
                non_blocking=True
            )

            edge_index = batch[
                "edge_index"
            ].to(
                DEVICE,
                non_blocking=True
            )

            edge_attr = batch[
                "edge_attr"
            ].to(
                DEVICE,
                non_blocking=True
            )

            target_edge_index = batch[
                "target_edge_index"
            ].to(
                DEVICE,
                non_blocking=True
            )

            target_edge_attr = batch[
                "target_edge_attr"
            ].to(
                DEVICE,
                non_blocking=True
            )

            # ------------------------------------------------
            # FULL FP32 RAW LOGITS
            # ------------------------------------------------

            logits = model(
                x,
                edge_index,
                edge_attr,
                target_edge_index,
                target_edge_attr
            )

            logits = (
                logits
                .float()
                .cpu()
                .numpy()
                .reshape(-1)
            )

            all_logits.append(
                logits
            )

            del (
                x,
                edge_index,
                edge_attr,
                target_edge_index,
                target_edge_attr
            )

    raw_logits = np.concatenate(
        all_logits
    )

    y_true = labels_np[
        edge_ids
    ].astype(np.int8)

    # --------------------------------------------------------
    # Metrics using logits
    # --------------------------------------------------------

    roc_auc = roc_auc_score(
        y_true,
        raw_logits
    )

    pr_auc = average_precision_score(
        y_true,
        raw_logits
    )

    # --------------------------------------------------------
    # Logit uniqueness
    # --------------------------------------------------------

    unique_logits = len(
        np.unique(raw_logits)
    )

    print()
    print("=" * 80)
    print(f"{split_name} RAW LOGIT RESULTS")
    print("=" * 80)

    print(
        f"ROC-AUC              : "
        f"{roc_auc:.8f}"
    )

    print(
        f"PR-AUC               : "
        f"{pr_auc:.8f}"
    )

    print()
    print("LOGIT RESOLUTION")
    print("-" * 80)

    print(
        f"Total predictions     : "
        f"{len(raw_logits):,}"
    )

    print(
        f"Unique logits         : "
        f"{unique_logits:,}"
    )

    print()
    print("LOGIT DISTRIBUTION")
    print("-" * 80)

    for q in [
        0,
        1,
        5,
        10,
        25,
        50,
        75,
        90,
        95,
        99,
        99.9,
        100
    ]:

        print(
            f"{q:>5.1f}% : "
            f"{np.percentile(raw_logits, q):.8f}"
        )

    print()
    print(
        f"Evaluation time : "
        f"{time.time() - start_time:.2f} sec"
    )

    return {
        "y_true": y_true,
        "logits": raw_logits,
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc)
    }


# ============================================================
# VALIDATION
# ============================================================

val_results_logits = evaluate_raw_logits(
    VAL_IDS,
    "VALIDATION"
)


# ============================================================
# TEST
# ============================================================

test_results_logits = evaluate_raw_logits(
    TEST_IDS,
    "TEST"
)


# ============================================================
# SAVE LOGITS
# ============================================================

logit_file = os.path.join(
    DATA_DIR,
    "gat_aml_raw_logits.npz"
)

np.savez(
    logit_file,
    val_y_true=val_results_logits["y_true"],
    val_logits=val_results_logits["logits"],
    test_y_true=test_results_logits["y_true"],
    test_logits=test_results_logits["logits"]
)

print()
print("=" * 80)
print("✅ RAW LOGIT EVALUATION COMPLETE")
print("=" * 80)

print(
    f"Saved logits to: {logit_file}"
)


RAW LOGIT EVALUATION — VALIDATION


VALIDATION:   0%|          | 0/184 [00:00<?, ?it/s]


VALIDATION RAW LOGIT RESULTS
ROC-AUC              : 0.99995566
PR-AUC               : 0.99994496

LOGIT RESOLUTION
--------------------------------------------------------------------------------
Total predictions     : 1,500,000
Unique logits         : 1,396,521

LOGIT DISTRIBUTION
--------------------------------------------------------------------------------
  0.0% : -21.05862236
  1.0% : -18.89238384
  5.0% : -17.93258152
 10.0% : -17.50673981
 25.0% : -16.37042618
 50.0% : -13.00193882
 75.0% : 9.39809442
 90.0% : 13.02074280
 95.0% : 14.85332952
 99.0% : 16.73937819
 99.9% : 28.25819602
100.0% : 29.84141731

Evaluation time : 51.04 sec

RAW LOGIT EVALUATION — TEST


TEST:   0%|          | 0/184 [00:00<?, ?it/s]


TEST RAW LOGIT RESULTS
ROC-AUC              : 0.99990419
PR-AUC               : 0.99997756

LOGIT RESOLUTION
--------------------------------------------------------------------------------
Total predictions     : 1,500,000
Unique logits         : 1,435,612

LOGIT DISTRIBUTION
--------------------------------------------------------------------------------
  0.0% : -20.85789490
  1.0% : -17.81262285
  5.0% : -16.32098637
 10.0% : -15.17006102
 25.0% : 6.18181193
 50.0% : 11.31006527
 75.0% : 14.04918694
 90.0% : 16.48471069
 95.0% : 17.51906614
 99.0% : 18.92575289
 99.9% : 28.74231357
100.0% : 29.96191406

Evaluation time : 60.32 sec

✅ RAW LOGIT EVALUATION COMPLETE
Saved logits to: /home/llyods-aids5/AML/gat_aml_raw_logits.npz


In [58]:
# ============================================================
# CELL 45 — PROPER TOP-K AML ALERT-BUDGET ANALYSIS
# ============================================================

import numpy as np
import pandas as pd

print("=" * 80)
print("AML TOP-K ALERT-BUDGET ANALYSIS")
print("=" * 80)

# ------------------------------------------------------------
# Use raw logits
# ------------------------------------------------------------

val_logits = val_results_logits["logits"]
val_y = val_results_logits["y_true"]

test_logits = test_results_logits["logits"]
test_y = test_results_logits["y_true"]

# ------------------------------------------------------------
# Evaluation helper
# ------------------------------------------------------------

def top_k_analysis(
    logits,
    y_true,
    split_name
):

    print()
    print("=" * 80)
    print(f"{split_name} TOP-K ANALYSIS")
    print("=" * 80)

    n = len(y_true)

    actual_aml = int(
        np.sum(y_true == 1)
    )

    # Highest risk first
    ranking = np.argsort(
        logits
    )[::-1]

    rows = []

    for pct in [
        0.01,
        0.05,
        0.10,
        0.25,
        0.50,
        1.0,
        2.0,
        5.0,
        10.0,
        20.0
    ]:

        k = max(
            1,
            int(
                n * pct / 100.0
            )
        )

        top_ids = ranking[:k]

        selected_labels = y_true[
            top_ids
        ]

        tp = int(
            np.sum(
                selected_labels == 1
            )
        )

        fp = int(
            np.sum(
                selected_labels == 0
            )
        )

        precision = (
            tp / k
            if k > 0
            else 0.0
        )

        recall = (
            tp / actual_aml
            if actual_aml > 0
            else 0.0
        )

        threshold_logit = float(
            logits[
                ranking[k - 1]
            ]
        )

        # Convert only this individual threshold score
        # to a probability for readability.
        threshold_prob = float(
            1.0 / (
                1.0
                + np.exp(
                    -np.clip(
                        threshold_logit,
                        -50,
                        50
                    )
                )
            )
        )

        workload_ratio = (
            k / tp
            if tp > 0
            else np.inf
        )

        rows.append({
            "Alert Budget (%)": pct,
            "Alerts": k,
            "Threshold Logit": threshold_logit,
            "Threshold Probability": threshold_prob,
            "TP": tp,
            "FP": fp,
            "Precision": precision,
            "Recall": recall,
            "Transactions / TP": workload_ratio
        })

    result = pd.DataFrame(
        rows
    )

    # --------------------------------------------------------
    # Print
    # --------------------------------------------------------

    print(
        result.to_string(
            index=False,
            formatters={
                "Alert Budget (%)":
                    lambda x: f"{x:.2f}",

                "Threshold Logit":
                    lambda x: f"{x:.6f}",

                "Threshold Probability":
                    lambda x: f"{x:.9f}",

                "Precision":
                    lambda x: f"{x:.6f}",

                "Recall":
                    lambda x: f"{x:.6f}",

                "Transactions / TP":
                    lambda x: f"{x:.2f}"
            }
        )
    )

    return result


# ============================================================
# VALIDATION
# ============================================================

val_topk = top_k_analysis(
    val_logits,
    val_y,
    "VALIDATION"
)


# ============================================================
# TEST
# ============================================================

test_topk = top_k_analysis(
    test_logits,
    test_y,
    "TEST"
)


# ============================================================
# SAVE
# ============================================================

topk_file = os.path.join(
    DATA_DIR,
    "gat_aml_topk_alert_budget.csv"
)

combined = pd.concat(
    [
        val_topk.assign(
            Split="Validation"
        ),
        test_topk.assign(
            Split="Test"
        )
    ],
    ignore_index=True
)

combined.to_csv(
    topk_file,
    index=False
)

print()
print("=" * 80)
print("✅ TOP-K ANALYSIS SAVED")
print("=" * 80)

print(
    f"File: {topk_file}"
)

AML TOP-K ALERT-BUDGET ANALYSIS

VALIDATION TOP-K ANALYSIS
Alert Budget (%)  Alerts Threshold Logit Threshold Probability     TP  FP Precision   Recall Transactions / TP
            0.01     150       29.557787           1.000000000    150   0  1.000000 0.000220              1.00
            0.05     750       28.865101           1.000000000    750   0  1.000000 0.001101              1.00
            0.10    1500       28.258339           1.000000000   1500   0  1.000000 0.002202              1.00
            0.25    3750       18.330536           0.999999989   3750   0  1.000000 0.005506              1.00
            0.50    7500       17.307684           0.999999970   7500   0  1.000000 0.011011              1.00
            1.00   15000       16.739399           0.999999946  15000   0  1.000000 0.022023              1.00
            2.00   30000       16.119431           0.999999900  30000   0  1.000000 0.044046              1.00
            5.00   75000       14.853381           0.